In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display as _raw_display

_COLUMN_LABELS = {
    'column': 'Column', 'type': 'Type', 'missing': 'Missing Count',
    'missing_pct': 'Missing Share', 'unique_values': 'Unique Values',
    'review_share_pct': 'Review Share', 'records': 'Records',
    'selected_records': 'Selected Records', 'found_positives': 'Found Dropouts',
    'false_alarms': 'False Alarms', 'missed_positives': 'Missed Dropouts',
    'precision_pct': 'Precision', 'recall_pct': 'Recall',
    'unique_entities_to_review': 'Unique Students Selected',
    'group_column': 'Group Feature', 'group_value': 'Group',
    'total_positives': 'Dropouts', 'total_negatives': 'Non-Dropouts',
    'positive_rate_pct': 'Dropout Rate', 'selection_rate_pct': 'Selection Rate',
    'false_positive_rate_pct': 'False-Positive Rate',
    'lowest_group': 'Lowest Group', 'lowest_pct': 'Lowest Rate',
    'highest_group': 'Highest Group', 'highest_pct': 'Highest Rate',
    'gap_pp': 'Gap', 'risk_band': 'Risk Band',
    'mean_predicted_pct': 'Mean Predicted Risk',
    'observed_rate_pct': 'Observed Dropout Rate',
    'actual_positive_pct': 'Observed Dropout Rate',
    'metric': 'Metric', 'value': 'Value',
    'median': 'Median', 'min': 'Minimum', 'max': 'Maximum',
    'with_mode_median': 'With Study Mode',
    'without_mode_median': 'Without Study Mode',
    'paired_change_median': 'Median Change',
    'paired_change_range': 'Change Range',
}

_ROW_LABELS = {
    'code_module': 'Course Module', 'code_presentation': 'Presentation',
    'id_student': 'Student ID', 'gender': 'Gender', 'region': 'Region',
    'highest_education': 'Highest Education', 'imd_band': 'IMD Band',
    'age_band': 'Age Band', 'num_of_prev_attempts': 'Previous Attempts',
    'studied_credits': 'Studied Credits', 'disability': 'Disability',
    'final_result': 'Final Result', 'dedicacion': 'Study Mode',
    'desplazado_hash': 'Displacement Status',
    'estudios_m_hash': "Mother's Education",
    'recall_pct': 'Review Recall', 'false_positive_rate_pct': 'False-Positive Rate',
    'roc_auc': 'ROC AUC', 'precision_pct': 'Review Precision',
    'TC_recall_pct': 'TC Recall', 'TP_recall_pct': 'TP Recall',
    'TP_minus_TC_recall_pp': 'TP − TC Recall Gap',
    'TC_false_positive_rate_pct': 'TC False-Positive Rate',
    'TP_false_positive_rate_pct': 'TP False-Positive Rate',
    'TP_minus_TC_false_positive_rate_pp': 'TP − TC False-Positive Rate Gap',
    'TP_students': 'TP Students', 'TP_dropouts': 'TP Dropouts',
    'Validation records': 'Validation Records',
    'Validation ROC AUC': 'Validation ROC AUC',
    'Mean predicted risk, %': 'Mean Predicted Risk',
    'Observed positive rate, %': 'Observed Dropout Rate',
    'Records selected for review': 'Selected Records',
    'Found positives': 'Found Dropouts', 'False alarms': 'False Alarms',
    'Missed positives': 'Missed Dropouts',
    'Review precision, %': 'Review Precision',
    'Review recall, %': 'Review Recall',
    'Precision in top 20%, %': 'Review Precision',
    'Recall in top 20%, %': 'Review Recall',
    'TC recall, %': 'TC Recall', 'TP recall, %': 'TP Recall',
    'TP - TC recall gap, pp': 'TP − TC Recall Gap',
    'TC false-positive rate, %': 'TC False-Positive Rate',
    'TP false-positive rate, %': 'TP False-Positive Rate',
    'TP - TC false-positive gap, pp': 'TP − TC False-Positive Rate Gap',
}

def _table_value(value, column, metric=''):
    if column == 'risk_band':
        band = str(value)
        if len(band) > 3 and band[0] in '[(' and band[-1] in '])' and ',' in band:
            try:
                low, high = band[1:-1].split(',')
                return f'{band[0]}{float(low)*100:.1f}%, {float(high)*100:.1f}%{band[-1]}'
            except ValueError:
                pass
    if pd.isna(value): return '—'
    key = (str(column) + ' ' + str(metric)).lower()
    if isinstance(value, (bool, np.bool_)): return 'Yes' if value else 'No'
    if isinstance(value, (int, float, np.integer, np.floating)):
        if 'auc' in key: return f'{value:.3f}'
        if 'pp' in key: return f'{value:.1f} pp'
        if 'pct' in key or '%' in key: return f'{value:.1f}%'
        if float(value).is_integer(): return f'{value:,.0f}'.replace(',', ' ')
        return f'{value:,.1f}'.replace(',', ' ')
    if isinstance(value, str) and column != 'metric':
        if column in ('with_mode_median', 'without_mode_median', 'paired_change_median', 'paired_change_range'):
            if 'auc' in key: return value
            unit = ' pp' if 'pp' in key or column.startswith('paired_change') else '%' if '%' in key else ''
            return value + unit if unit and not value.endswith(unit) else value
    return str(value)

def _show_table(frame):
    if frame.index.name is not None or frame.index.dtype == 'object':
        view = frame.reset_index().rename(columns={'index': 'metric'})
    else:
        view = frame.reset_index(drop=True)
    metric_col = 'metric' if 'metric' in view.columns else None
    for column in view.columns:
        view[column] = [
            _table_value(value, column, view.at[i, metric_col] if metric_col else '')
            for i, value in enumerate(view[column])
        ]
    if 'group_column' in view.columns:
        imd = view['group_column'].eq('imd_band')
        for column in ('group_value', 'lowest_group', 'highest_group'):
            if column in view.columns:
                view.loc[imd, column] = view.loc[imd, column].map(
                    lambda text: text + '%' if isinstance(text, str) and '-' in text and '%' not in text and text.replace('-', '').isdigit() else text
                )
    if 'type' in view.columns:
        view['type'] = view['type'].replace({'object': 'Text', 'int64': 'Integer', 'float64': 'Decimal'})
    for column in ('column', 'group_column', 'metric'):
        if column in view.columns:
            view[column] = view[column].replace(_ROW_LABELS)
    view = view.rename(columns=_COLUMN_LABELS)
    _raw_display(view.style.hide(axis='index').format(escape='html'))

def display(*objects, **kwargs):
    for obj in objects:
        if isinstance(obj, pd.DataFrame):
            _show_table(obj)
        else:
            _raw_display(obj, **kwargs)


In [ ]:
import pandas as pd
from google.colab import drive

# Change these values when testing another dataset.
data_path = "/content/drive/MyDrive/eduai-audit/data/studentInfo.csv"
target_column = "final_result"
positive_value = "Withdrawn"
entity_id_column = "id_student"  # Use None if there is no ID column.
group_columns = ["imd_band"]
review_share = 0.20

drive.mount("/content/drive")
data = pd.read_csv(data_path, na_values=["?"])

required_columns = [target_column, *group_columns]
if entity_id_column is not None:
    required_columns.append(entity_id_column)

missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")
if positive_value not in data[target_column].dropna().unique():
    raise ValueError(f"Positive value not found: {positive_value}")

print("Records:", len(data))
print("Columns:", len(data.columns))
print("Target values:")
print(data[target_column].value_counts(dropna=False))
print("Group columns:", group_columns)
print("Review share:", review_share)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Records: 32593
Columns: 12
Target values:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64
Group columns: ['imd_band']
Review share: 0.2


In [ ]:
column_summary = pd.DataFrame({
    "column": data.columns,
    "type": data.dtypes.astype(str).to_numpy(),
    "missing": data.isna().sum().to_numpy(),
    "missing_pct": (100 * data.isna().mean()).round(1).to_numpy(),
    "unique_values": data.nunique(dropna=True).to_numpy(),
})

display(column_summary)

Column,Type,Missing Count,Missing Share,Unique Values
Course Module,Text,0,0.0%,7
Presentation,Text,0,0.0%,4
Student ID,Integer,0,0.0%,28 785
Gender,Text,0,0.0%,2
Region,Text,0,0.0%,13
Highest Education,Text,0,0.0%,5
IMD Band,Text,1 111,3.4%,10
Age Band,Text,0,0.0%,3
Previous Attempts,Integer,0,0.0%,7
Studied Credits,Integer,0,0.0%,61


In [ ]:
if data[target_column].isna().any():
    raise ValueError("Target column contains missing values")

y = data[target_column].eq(positive_value).astype(int)

excluded_columns = {target_column}
if entity_id_column is not None:
    excluded_columns.add(entity_id_column)

# Check this list before training on a new dataset.
feature_columns = [column for column in data.columns if column not in excluded_columns]
X = data[feature_columns].copy()

print("Feature columns:", feature_columns)
print("X shape:", X.shape)
print("Positive cases:", int(y.sum()))
print("Positive rate:", round(y.mean(), 3))

Feature columns: ['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability']
X shape: (32593, 10)
Positive cases: 10156
Positive rate: 0.312


In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

if entity_id_column is None:
    split_groups = np.arange(len(data))
else:
    if data[entity_id_column].isna().any():
        raise ValueError("Entity ID column contains missing values")
    split_groups = data[entity_id_column].to_numpy()

# Keep the same students together in every split.
first_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
development_positions, reserved_positions = next(
    first_split.split(X, y, groups=split_groups)
)

second_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43)
train_local, val_local = next(
    second_split.split(
        X.iloc[development_positions],
        y.iloc[development_positions],
        groups=split_groups[development_positions],
    )
)

train_positions = development_positions[train_local]
val_positions = development_positions[val_local]

X_train = X.iloc[train_positions]
y_train = y.iloc[train_positions]
X_val = X.iloc[val_positions]
y_val = y.iloc[val_positions]

train_groups = set(split_groups[train_positions])
val_groups = set(split_groups[val_positions])
reserved_groups = set(split_groups[reserved_positions])

print("Train records:", len(X_train), "Positive rate:", round(y_train.mean(), 3))
print("Validation records:", len(X_val), "Positive rate:", round(y_val.mean(), 3))
print("Reserved records:", len(reserved_positions))
print("Train/validation group overlap:", len(train_groups & val_groups))
print("Train/reserved group overlap:", len(train_groups & reserved_groups))
print("Validation/reserved group overlap:", len(val_groups & reserved_groups))

Train records: 19584 Positive rate: 0.316
Validation records: 6538 Positive rate: 0.308
Reserved records: 6471
Train/validation group overlap: 0
Train/reserved group overlap: 0
Validation/reserved group overlap: 0


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(
    include=["object", "category", "bool", "string"]
).columns.tolist()

unhandled_columns = [
    column for column in X_train.columns
    if column not in numeric_columns + categorical_columns
]
if unhandled_columns:
    raise ValueError(f"Columns need separate processing: {unhandled_columns}")

numeric_processing = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_processing = Pipeline([
    ("fill_missing", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessing = ColumnTransformer([
    ("numeric", numeric_processing, numeric_columns),
    ("categorical", categorical_processing, categorical_columns),
])

print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

Numeric columns: ['num_of_prev_attempts', 'studied_credits']
Categorical columns: ['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']


In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

print("Model trained on", len(X_train), "records")

Model trained on 19584 records


In [ ]:
from sklearn.metrics import roc_auc_score

if y_val.nunique() != 2:
    raise ValueError("Validation must contain both classes for ROC AUC")

val_probabilities = model.predict_proba(X_val)[:, 1]

print("Validation records:", len(X_val))
print("First five predicted probabilities:", val_probabilities[:5].round(3))
print("Mean predicted probability:", round(val_probabilities.mean(), 3))
print("Observed positive rate:", round(y_val.mean(), 3))
print("ROC AUC:", round(roc_auc_score(y_val, val_probabilities), 3))

Validation records: 6538
First five predicted probabilities: [0.111 0.062 0.15  0.112 0.159]
Mean predicted probability: 0.314
Observed positive rate: 0.308
ROC AUC: 0.66


In [ ]:
def select_top_risk(y_true, probabilities, share):
    actual = np.asarray(y_true)
    scores = np.asarray(probabilities)

    if actual.ndim != 1 or scores.ndim != 1 or len(actual) == 0 or len(actual) != len(scores):
        raise ValueError("Actual values and probabilities must have the same non-zero length")
    if not 0 < share <= 1:
        raise ValueError("Review share must be greater than 0 and at most 1")
    if not np.isin(actual, [0, 1]).all():
        raise ValueError("Actual values must contain only 0 and 1")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be between 0 and 1")

    records_to_review = int(np.ceil(share * len(scores)))
    ranked_positions = np.argsort(-scores, kind="stable")
    selected = np.zeros(len(scores), dtype=bool)
    selected[ranked_positions[:records_to_review]] = True

    found_positives = int(((actual == 1) & selected).sum())
    false_alarms = int(((actual == 0) & selected).sum())
    total_positives = int((actual == 1).sum())

    summary = {
        "review_share_pct": 100 * share,
        "records": len(actual),
        "selected_records": records_to_review,
        "found_positives": found_positives,
        "false_alarms": false_alarms,
        "missed_positives": total_positives - found_positives,
        "precision_pct": 100 * found_positives / records_to_review,
        "recall_pct": 100 * found_positives / total_positives if total_positives else np.nan,
    }
    return summary, selected


review_result, selected_for_review = select_top_risk(
    y_val, val_probabilities, review_share
)

if entity_id_column is not None:
    validation_ids = data[entity_id_column].iloc[val_positions].to_numpy()
    review_result["unique_entities_to_review"] = len(
        np.unique(validation_ids[selected_for_review])
    )

display(pd.DataFrame([review_result]).round(1))

Review Share,Records,Selected Records,Found Dropouts,False Alarms,Missed Dropouts,Precision,Recall,Unique Students Selected
20.0%,6 538,1 308,642,666,1 370,49.1%,31.9%,1 165


In [ ]:
def audit_review_groups(features, y_true, selected, columns):
    actual = np.asarray(y_true)
    selected = np.asarray(selected, dtype=bool)

    if len(features) != len(actual) or len(selected) != len(actual):
        raise ValueError("Features, actual values and selection must have the same length")

    rows = []
    for column in columns:
        if column not in features.columns:
            raise ValueError(f"Group column not found: {column}")

        values = features[column].astype("object").fillna("Unknown").astype(str).to_numpy()
        for group_value in np.unique(values):
            in_group = values == group_value
            positives = in_group & (actual == 1)
            negatives = in_group & (actual == 0)

            group_size = int(in_group.sum())
            selected_count = int((in_group & selected).sum())
            total_positives = int(positives.sum())
            total_negatives = int(negatives.sum())
            found_positives = int((positives & selected).sum())
            false_alarms = int((negatives & selected).sum())

            rows.append({
                "group_column": column,
                "group_value": group_value,
                "records": group_size,
                "total_positives": total_positives,
                "total_negatives": total_negatives,
                "positive_rate_pct": 100 * total_positives / group_size,
                "selected_records": selected_count,
                "selection_rate_pct": 100 * selected_count / group_size,
                "found_positives": found_positives,
                "false_alarms": false_alarms,
                "recall_pct": 100 * found_positives / total_positives if total_positives else np.nan,
                "false_positive_rate_pct": 100 * false_alarms / total_negatives if total_negatives else np.nan,
            })

    return pd.DataFrame(rows)


group_audit = audit_review_groups(
    X_val, y_val, selected_for_review, group_columns
)

if group_audit.empty:
    print("No group columns configured")
else:
    display(group_audit.round(1))

Group Feature,Group,Records,Dropouts,Non-Dropouts,Dropout Rate,Selected Records,Selection Rate,Found Dropouts,False Alarms,Recall,False-Positive Rate
IMD Band,0-10%,653,236,417,36.1%,218,33.4%,109,109,46.2%,26.1%
IMD Band,10-20%,684,221,463,32.3%,228,33.3%,100,128,45.2%,27.6%
IMD Band,20-30%,702,244,458,34.8%,219,31.2%,102,117,41.8%,25.5%
IMD Band,30-40%,715,220,495,30.8%,133,18.6%,62,71,28.2%,14.3%
IMD Band,40-50%,674,229,445,34.0%,126,18.7%,79,47,34.5%,10.6%
IMD Band,50-60%,640,189,451,29.5%,88,13.8%,46,42,24.3%,9.3%
IMD Band,60-70%,572,165,407,28.8%,78,13.6%,41,37,24.8%,9.1%
IMD Band,70-80%,578,171,407,29.6%,67,11.6%,28,39,16.4%,9.6%
IMD Band,80-90%,584,170,414,29.1%,76,13.0%,41,35,24.1%,8.5%
IMD Band,90-100%,527,124,403,23.5%,65,12.3%,31,34,25.0%,8.4%


In [ ]:
# These are provisional screening limits, not a statistical test.
minimum_group_records = max(100, int(np.ceil(0.05 * len(X_val))))
minimum_outcomes_per_class = 30

group_gaps = pd.DataFrame()
small_groups = pd.DataFrame()

if group_audit.empty:
    print("No group columns to compare")
else:
    group_size_check = group_audit.copy()
    group_size_check["passes_size_check"] = (
        (group_size_check["records"] >= minimum_group_records)
        & (group_size_check["total_positives"] >= minimum_outcomes_per_class)
        & (group_size_check["total_negatives"] >= minimum_outcomes_per_class)
    )

    small_groups = group_size_check.loc[
        ~group_size_check["passes_size_check"],
        ["group_column", "group_value", "records", "total_positives", "total_negatives"],
    ]

    gap_rows = []
    for column, groups in group_size_check.groupby("group_column"):
        comparable = groups[groups["passes_size_check"]]
        if len(comparable) < 2:
            continue

        for metric in ["recall_pct", "false_positive_rate_pct"]:
            lowest = comparable.loc[comparable[metric].idxmin()]
            highest = comparable.loc[comparable[metric].idxmax()]
            gap_rows.append({
                "group_column": column,
                "metric": metric,
                "lowest_group": lowest["group_value"],
                "lowest_pct": lowest[metric],
                "highest_group": highest["group_value"],
                "highest_pct": highest[metric],
                "gap_pp": highest[metric] - lowest[metric],
            })

    group_gaps = pd.DataFrame(gap_rows)
    print("Minimum group records:", minimum_group_records)
    print("Minimum records per outcome:", minimum_outcomes_per_class)

    if group_gaps.empty:
        print("Not enough groups pass the size check for comparison")
    else:
        display(group_gaps.round(1))

    if small_groups.empty:
        print("No groups flagged as small")
    else:
        display(small_groups)

Minimum group records: 327
Minimum records per outcome: 30


Group Feature,Metric,Lowest Group,Lowest Rate,Highest Group,Highest Rate,Gap
IMD Band,Review Recall,70-80%,16.4%,0-10%,46.2%,29.8 pp
IMD Band,False-Positive Rate,90-100%,8.4%,10-20%,27.6%,19.2 pp


Group Feature,Group,Records,Dropouts,Non-Dropouts
IMD Band,Unknown,209,43,166


In [ ]:
def calibration_by_risk(y_true, probabilities, bands=5):
    actual = np.asarray(y_true)
    scores = np.asarray(probabilities)

    if actual.ndim != 1 or scores.ndim != 1 or len(actual) == 0 or len(actual) != len(scores):
        raise ValueError("Actual values and probabilities must have the same non-zero length")
    if not np.isin(actual, [0, 1]).all():
        raise ValueError("Actual values must contain only 0 and 1")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be between 0 and 1")
    if bands < 2:
        raise ValueError("Number of risk bands must be at least 2")

    calibration_data = pd.DataFrame({"actual": actual, "probability": scores})
    if np.unique(scores).size == 1:
        calibration_data["risk_band"] = "All probabilities equal"
    else:
        calibration_data["risk_band"] = pd.qcut(
            calibration_data["probability"], q=bands, duplicates="drop"
        )

    summary = (
        calibration_data.groupby("risk_band", observed=True)
        .agg(
            records=("actual", "size"),
            mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
            actual_positive_pct=("actual", lambda values: 100 * values.mean()),
        )
        .reset_index()
    )
    summary["gap_pp"] = summary["actual_positive_pct"] - summary["mean_predicted_pct"]
    return summary


calibration_summary = calibration_by_risk(y_val, val_probabilities)
display(calibration_summary.round(1))

Risk Band,Records,Mean Predicted Risk,Observed Dropout Rate,Gap
"(4.9%, 19.7%]",1 309,14.3%,14.9%,0.6 pp
"(19.7%, 26.6%]",1 306,23.2%,23.4%,0.1 pp
"(26.6%, 33.3%]",1 309,29.8%,30.5%,0.7 pp
"(33.3%, 42.1%]",1 306,37.4%,36.1%,-1.3 pp
"(42.1%, 89.1%]",1 308,52.1%,49.1%,-3.0 pp


In [ ]:
def report_rows(frame):
    """Convert a result table to plain Python records."""
    return frame.astype(object).where(pd.notna(frame), None).to_dict(orient="records")


assert len(val_probabilities) == len(X_val) == len(y_val)
assert int(selected_for_review.sum()) == review_result["selected_records"]

calibration_for_report = calibration_summary.copy()
calibration_for_report["risk_band"] = calibration_for_report["risk_band"].astype(str)

audit_report = {
    "configuration": {
        "target_column": target_column,
        "positive_value": str(positive_value),
        "entity_id_column": entity_id_column,
        "group_columns": list(group_columns),
        "review_share": float(review_share),
        "selection_unit": "record",
    },
    "data": {
        "total_records": int(len(data)),
        "train_records": int(len(X_train)),
        "validation_records": int(len(X_val)),
        "reserved_records": int(len(reserved_positions)),
        "validation_positive_rate_pct": float(100 * y_val.mean()),
    },
    "model": {
        "validation_roc_auc": float(roc_auc_score(y_val, val_probabilities)),
        "mean_predicted_pct": float(100 * val_probabilities.mean()),
        "observed_positive_pct": float(100 * y_val.mean()),
    },
    "review": dict(review_result),
    "group_audit": report_rows(group_audit),
    "group_gaps": report_rows(group_gaps),
    "small_groups": report_rows(small_groups),
    "calibration": report_rows(calibration_for_report),
    "screening_rule": {
        "minimum_group_records": int(minimum_group_records),
        "minimum_outcomes_per_class": int(minimum_outcomes_per_class),
        "is_statistical_test": False,
    },
    "limitations": [
        "The review share is an illustrative capacity assumption, not an optimized threshold.",
        "Selection is per record; one entity can have multiple records.",
        "Group gaps are descriptive and do not establish unfairness or causality.",
        "The reserved OULAD split was examined in earlier work and is not an independent final test.",
    ],
}

print("Structured report ready; sections:", ", ".join(audit_report))

summary = pd.DataFrame([
    {"metric": "Validation records", "value": audit_report["data"]["validation_records"]},
    {"metric": "Validation ROC AUC", "value": round(audit_report["model"]["validation_roc_auc"], 3)},
    {"metric": "Mean predicted risk, %", "value": round(audit_report["model"]["mean_predicted_pct"], 1)},
    {"metric": "Observed positive rate, %", "value": round(audit_report["model"]["observed_positive_pct"], 1)},
    {"metric": "Records selected for review", "value": audit_report["review"]["selected_records"]},
    {"metric": "Found positives", "value": audit_report["review"]["found_positives"]},
    {"metric": "False alarms", "value": audit_report["review"]["false_alarms"]},
    {"metric": "Missed positives", "value": audit_report["review"]["missed_positives"]},
    {"metric": "Review precision, %", "value": round(audit_report["review"]["precision_pct"], 1)},
    {"metric": "Review recall, %", "value": round(audit_report["review"]["recall_pct"], 1)},
])
display(summary)


Structured report ready; sections: configuration, data, model, review, group_audit, group_gaps, small_groups, calibration, screening_rule, limitations


Metric,Value
Validation Records,6 538
Validation ROC AUC,0.660
Mean Predicted Risk,31.4%
Observed Dropout Rate,30.8%
Selected Records,1 308
Found Dropouts,642
False Alarms,666
Missed Dropouts,1 370
Review Precision,49.1%
Review Recall,31.9%


In [ ]:
"""Reusable validation audit for a binary outcome in a tabular dataset."""

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def run_audit(
    dataset,
    target_column,
    positive_value,
    entity_id_column=None,
    group_columns=None,
    review_share=0.20,
    feature_columns=None,
):
    """Train a baseline model and audit its validation predictions.

    The input is an already loaded DataFrame. The reserved split is kept out
    of all model fitting and validation calculations.
    """
    if not isinstance(dataset, pd.DataFrame) or dataset.empty:
        raise ValueError("dataset must be a non-empty pandas DataFrame")
    if not 0 < review_share <= 1:
        raise ValueError("review_share must be greater than 0 and at most 1")
    if isinstance(group_columns, str):
        group_columns = [group_columns]
    group_columns = list(group_columns or [])
    if len(group_columns) != len(set(group_columns)):
        raise ValueError("group_columns contains duplicates")

    required = [target_column, *group_columns]
    if entity_id_column is not None:
        required.append(entity_id_column)
    missing = [column for column in required if column not in dataset.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if dataset[target_column].isna().any():
        raise ValueError("Target column contains missing values")
    if positive_value not in dataset[target_column].unique():
        raise ValueError(f"Positive value not found: {positive_value}")
    if entity_id_column is not None and dataset[entity_id_column].isna().any():
        raise ValueError("Entity ID column contains missing values")

    excluded = {target_column}
    if entity_id_column is not None:
        excluded.add(entity_id_column)
    if feature_columns is None:
        feature_columns = [column for column in dataset.columns if column not in excluded]
    else:
        feature_columns = list(feature_columns)
    if not feature_columns or len(feature_columns) != len(set(feature_columns)):
        raise ValueError("feature_columns must be non-empty and unique")
    if excluded.intersection(feature_columns):
        raise ValueError("feature_columns cannot contain target or entity ID")
    missing = [column for column in feature_columns if column not in dataset.columns]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")

    X = dataset[feature_columns].copy()
    y = dataset[target_column].eq(positive_value).astype(int)
    if y.nunique() != 2:
        raise ValueError("Dataset must contain both outcome classes")
    split_groups = (
        np.arange(len(dataset))
        if entity_id_column is None
        else dataset[entity_id_column].to_numpy()
    )
    first_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    development_positions, reserved_positions = next(
        first_split.split(X, y, groups=split_groups)
    )
    second_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43)
    train_local, val_local = next(
        second_split.split(
            X.iloc[development_positions],
            y.iloc[development_positions],
            groups=split_groups[development_positions],
        )
    )
    train_positions = development_positions[train_local]
    val_positions = development_positions[val_local]
    X_train, y_train = X.iloc[train_positions], y.iloc[train_positions]
    X_val, y_val = X.iloc[val_positions], y.iloc[val_positions]
    if y_train.nunique() != 2 or y_val.nunique() != 2:
        raise ValueError("Train and validation must each contain both classes")

    numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
    categorical_columns = X_train.select_dtypes(
        include=["object", "category", "bool", "string"]
    ).columns.tolist()
    unhandled = [
        column for column in feature_columns
        if column not in numeric_columns + categorical_columns
    ]
    if unhandled:
        raise ValueError(f"Columns need separate processing: {unhandled}")
    transformers = []
    if numeric_columns:
        transformers.append((
            "numeric",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            numeric_columns,
        ))
    if categorical_columns:
        transformers.append((
            "categorical",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("encode", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_columns,
        ))
    model = Pipeline([
        ("preprocessing", ColumnTransformer(transformers)),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_val)[:, 1]
    actual = y_val.to_numpy()

    records_to_review = int(np.ceil(review_share * len(actual)))
    ranked_positions = np.argsort(-probabilities, kind="stable")
    selected = np.zeros(len(actual), dtype=bool)
    selected[ranked_positions[:records_to_review]] = True
    found = int(((actual == 1) & selected).sum())
    false_alarms = int(((actual == 0) & selected).sum())
    total_positives = int((actual == 1).sum())
    review = {
        "review_share_pct": 100 * review_share,
        "records": len(actual),
        "selected_records": records_to_review,
        "found_positives": found,
        "false_alarms": false_alarms,
        "missed_positives": total_positives - found,
        "precision_pct": 100 * found / records_to_review,
        "recall_pct": 100 * found / total_positives,
    }
    if entity_id_column is not None:
        validation_ids = dataset[entity_id_column].iloc[val_positions].to_numpy()
        review["unique_entities_to_review"] = len(np.unique(validation_ids[selected]))

    group_rows = []
    for column in group_columns:
        values = (
            dataset[column].iloc[val_positions]
            .astype("object").fillna("Unknown").astype(str).to_numpy()
        )
        for group_value in np.unique(values):
            in_group = values == group_value
            positives = in_group & (actual == 1)
            negatives = in_group & (actual == 0)
            group_size = int(in_group.sum())
            total_group_positives = int(positives.sum())
            total_group_negatives = int(negatives.sum())
            found_group = int((positives & selected).sum())
            false_group = int((negatives & selected).sum())
            group_rows.append({
                "group_column": column,
                "group_value": group_value,
                "records": group_size,
                "total_positives": total_group_positives,
                "total_negatives": total_group_negatives,
                "positive_rate_pct": 100 * total_group_positives / group_size,
                "selected_records": int((in_group & selected).sum()),
                "selection_rate_pct": 100 * (in_group & selected).sum() / group_size,
                "found_positives": found_group,
                "false_alarms": false_group,
                "recall_pct": (
                    100 * found_group / total_group_positives
                    if total_group_positives else np.nan
                ),
                "false_positive_rate_pct": (
                    100 * false_group / total_group_negatives
                    if total_group_negatives else np.nan
                ),
            })
    group_audit = pd.DataFrame(group_rows)
    minimum_group_records = max(100, int(np.ceil(0.05 * len(X_val))))
    minimum_outcomes_per_class = 30
    gap_rows = []
    small_groups = pd.DataFrame(columns=[
        "group_column", "group_value", "records", "total_positives", "total_negatives"
    ])
    if not group_audit.empty:
        size_check = group_audit.copy()
        size_check["passes_size_check"] = (
            (size_check["records"] >= minimum_group_records)
            & (size_check["total_positives"] >= minimum_outcomes_per_class)
            & (size_check["total_negatives"] >= minimum_outcomes_per_class)
        )
        small_groups = size_check.loc[
            ~size_check["passes_size_check"], small_groups.columns
        ]
        for column, groups in size_check.groupby("group_column"):
            comparable = groups[groups["passes_size_check"]]
            if len(comparable) < 2:
                continue
            for metric in ["recall_pct", "false_positive_rate_pct"]:
                lowest = comparable.loc[comparable[metric].idxmin()]
                highest = comparable.loc[comparable[metric].idxmax()]
                gap_rows.append({
                    "group_column": column,
                    "metric": metric,
                    "lowest_group": lowest["group_value"],
                    "lowest_pct": lowest[metric],
                    "highest_group": highest["group_value"],
                    "highest_pct": highest[metric],
                    "gap_pp": highest[metric] - lowest[metric],
                })
    group_gaps = pd.DataFrame(gap_rows)

    calibration_data = pd.DataFrame({"actual": actual, "probability": probabilities})
    if np.unique(probabilities).size == 1:
        calibration_data["risk_band"] = "All probabilities equal"
    else:
        calibration_data["risk_band"] = pd.qcut(
            calibration_data["probability"], q=5, duplicates="drop"
        )
    calibration = (
        calibration_data.groupby("risk_band", observed=True)
        .agg(
            records=("actual", "size"),
            mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
            actual_positive_pct=("actual", lambda values: 100 * values.mean()),
        )
        .reset_index()
    )
    calibration["gap_pp"] = (
        calibration["actual_positive_pct"] - calibration["mean_predicted_pct"]
    )

    report = {
        "configuration": {
            "target_column": target_column,
            "positive_value": positive_value,
            "entity_id_column": entity_id_column,
            "group_columns": group_columns,
            "feature_columns": feature_columns,
            "review_share": review_share,
            "selection_unit": "record",
        },
        "data": {
            "total_records": len(dataset),
            "train_records": len(X_train),
            "validation_records": len(X_val),
            "reserved_records": len(reserved_positions),
            "validation_positive_rate_pct": 100 * y_val.mean(),
        },
        "model": {
            "validation_roc_auc": roc_auc_score(y_val, probabilities),
            "mean_predicted_pct": 100 * probabilities.mean(),
            "observed_positive_pct": 100 * y_val.mean(),
        },
        "review": review,
        "group_audit": group_audit,
        "group_gaps": group_gaps,
        "small_groups": small_groups,
        "calibration": calibration,
        "screening_rule": {
            "minimum_group_records": minimum_group_records,
            "minimum_outcomes_per_class": minimum_outcomes_per_class,
            "is_statistical_test": False,
        },
    }
    return report, model


# Check the reusable function against the earlier OULAD cells.
reusable_report, reusable_model = run_audit(
    data,
    target_column=target_column,
    positive_value=positive_value,
    entity_id_column=entity_id_column,
    group_columns=group_columns,
    review_share=review_share,
)
assert reusable_report["data"]["train_records"] == len(X_train)
assert reusable_report["data"]["validation_records"] == len(X_val)
assert reusable_report["data"]["reserved_records"] == len(reserved_positions)
assert np.isclose(
    reusable_report["model"]["validation_roc_auc"],
    roc_auc_score(y_val, val_probabilities),
)
for metric in ["selected_records", "found_positives", "false_alarms", "missed_positives", "unique_entities_to_review"]:
    assert reusable_report["review"][metric] == review_result[metric], metric
for key, old_table in [
    ("group_audit", group_audit),
    ("group_gaps", group_gaps),
    ("small_groups", small_groups),
    ("calibration", calibration_summary),
]:
    pd.testing.assert_frame_equal(
        reusable_report[key].reset_index(drop=True),
        old_table.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=1e-10,
        atol=1e-10,
    )
print("Reusable audit matches the current OULAD prototype")
display(pd.DataFrame([reusable_report["review"]]).round(1))


Reusable audit matches the current OULAD prototype


Review Share,Records,Selected Records,Found Dropouts,False Alarms,Missed Dropouts,Precision,Recall,Unique Students Selected
20.0%,6 538,1 308,642,666,1 370,49.1%,31.9%,1 165


## Второй набор данных: университет UPV, 2022 год

Источник: [архив UPV на Zenodo](https://zenodo.org/records/17239943), [описание набора](https://www.mdpi.com/2306-5729/10/10/162). В архиве одна строка соответствует записи студента на предмет. Для нашего аудита ниже оставляем одну строку на студента и проверяем, что исход и выбранные признаки одинаковы во всех его строках.

**Цель:** код `A` означает уход, `B` — отсутствие ухода ([описание кодировки](https://www.mdpi.com/2227-7390/14/17/3147)). **Момент прогноза:** начало учебного года. Берём только данные о поступлении, фоне и результатах предыдущего года; оценки, активность LMS и Wi-Fi за текущий год не используем.

Сравнение групп здесь описательное. Пол и возраст не опубликованы, а часть других групп закодирована буквами без понятной расшифровки. Результат на UPV за один год не является проверкой модели на другом университете: модель обучается заново на данных UPV.

In [ ]:
# UPV, academic year 2022. Source: https://zenodo.org/records/17239943
import io
import zipfile
import requests
import pandas as pd

upv_url = 'https://zenodo.org/records/17239943/files/dataset_2022_hash.zip?download=1'
response = requests.get(upv_url, timeout=120)
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    upv_raw = pd.read_csv(archive.open('dataset_2022_hash.csv'), sep=';', low_memory=False)

print(f'UPV 2022: {len(upv_raw):,} course records, {upv_raw["dni_hash"].nunique():,} students')
print('Outcome codes:', upv_raw['abandono_hash'].value_counts().to_dict())

UPV 2022: 159,173 course records, 20,427 students
Outcome codes: {'B': 148384, 'A': 10789}


In [ ]:
# Prediction point: start of the 2022 academic year.
# Admission, background, and previous-year performance only.
upv_features = [
    'anyo_ingreso', 'tipo_ingreso', 'nota10_hash', 'nota14_hash',
    'estudios_p_hash', 'estudios_m_hash', 'dedicacion',
    'desplazado_hash', 'preferencia_seleccion',
    'anyo_inicio_estudios', 'rend_total_ultimo',
]
upv_columns = ['dni_hash', 'abandono_hash', *upv_features]
if upv_raw['dni_hash'].isna().any():
    raise ValueError('UPV student IDs contain missing values')
if not upv_raw.groupby('dni_hash')[upv_columns[1:]].nunique(dropna=False).le(1).all().all():
    raise ValueError('A student has conflicting outcome or feature values across course rows')
upv_students = upv_raw[upv_columns].drop_duplicates('dni_hash').copy()
assert upv_students['abandono_hash'].isin(['A', 'B']).all()
print('One row per student:', len(upv_students))
print('Outcome codes (A = dropout):', upv_students['abandono_hash'].value_counts().to_dict())
print('Features available at prediction time:', upv_features)

One row per student: 20427
Outcome codes (A = dropout): {'B': 18933, 'A': 1494}
Features available at prediction time: ['anyo_ingreso', 'tipo_ingreso', 'nota10_hash', 'nota14_hash', 'estudios_p_hash', 'estudios_m_hash', 'dedicacion', 'desplazado_hash', 'preferencia_seleccion', 'anyo_inicio_estudios', 'rend_total_ultimo']


In [ ]:
# Here a record is one student, so review_share is the share of students.
upv_report, upv_model = run_audit(
    upv_students,
    target_column='abandono_hash',
    positive_value='A',
    entity_id_column='dni_hash',
    group_columns=['dedicacion', 'desplazado_hash', 'estudios_m_hash'],
    review_share=0.20,
    feature_columns=upv_features,
)
print('UPV validation ROC AUC:', round(upv_report['model']['validation_roc_auc'], 3))
print('Student review summary:')
display(pd.DataFrame([upv_report['review']]).round(1))
print('Group audit:')
display(upv_report['group_audit'].round(1))
print('Group gaps with enough records:')
display(upv_report['group_gaps'].round(1))
print('Groups too small for gap comparison:')
display(upv_report['small_groups'])
print('Calibration by risk band:')
display(upv_report['calibration'].round(1))

UPV validation ROC AUC: 0.731
Student review summary:


Review Share,Records,Selected Records,Found Dropouts,False Alarms,Missed Dropouts,Precision,Recall,Unique Students Selected
20.0%,4 086,818,133,685,159,16.3%,45.5%,818


Group audit:


Group Feature,Group,Records,Dropouts,Non-Dropouts,Dropout Rate,Selected Records,Selection Rate,Found Dropouts,False Alarms,Recall,False-Positive Rate
Study Mode,TC,3 977,271,3 706,6.8%,736,18.5%,116,620,42.8%,16.7%
Study Mode,TP,109,21,88,19.3%,82,75.2%,17,65,81.0%,73.9%
Displacement Status,A,2 508,181,2 327,7.2%,547,21.8%,96,451,53.0%,19.4%
Displacement Status,B,1 578,111,1 467,7.0%,271,17.2%,37,234,33.3%,16.0%
Mother's Education,F,1 347,75,1 272,5.6%,230,17.1%,30,200,40.0%,15.7%
Mother's Education,L,1 905,135,1 770,7.1%,300,15.7%,51,249,37.8%,14.1%
Mother's Education,M,33,5,28,15.2%,16,48.5%,3,13,60.0%,46.4%
Mother's Education,P,8,0,8,0.0%,2,25.0%,0,2,—,25.0%
Mother's Education,R,45,5,40,11.1%,12,26.7%,3,9,60.0%,22.5%
Mother's Education,T,708,58,650,8.2%,220,31.1%,32,188,55.2%,28.9%


Group gaps with enough records:


Group Feature,Metric,Lowest Group,Lowest Rate,Highest Group,Highest Rate,Gap
Displacement Status,Review Recall,B,33.3%,A,53.0%,19.7 pp
Displacement Status,False-Positive Rate,B,16.0%,A,19.4%,3.4 pp
Mother's Education,Review Recall,L,37.8%,T,55.2%,17.4 pp
Mother's Education,False-Positive Rate,L,14.1%,T,28.9%,14.9 pp


Groups too small for gap comparison:


Group Feature,Group,Records,Dropouts,Non-Dropouts
Study Mode,TP,109,21,88
Mother's Education,M,33,5,28
Mother's Education,P,8,0,8
Mother's Education,R,45,5,40
Mother's Education,Unknown,40,14,26


Calibration by risk band:


Risk Band,Records,Mean Predicted Risk,Observed Dropout Rate,Gap
"(0.6%, 1.9%]",818,1.4%,1.3%,-0.1 pp
"(1.9%, 3.9%]",817,2.8%,3.1%,0.2 pp
"(3.9%, 6.5%]",817,5.2%,5.8%,0.6 pp
"(6.5%, 10.0%]",824,8.4%,9.2%,0.8 pp
"(10.0%, 76.8%]",810,18.5%,16.4%,-2.1 pp


In [ ]:
# Stability across 20 train/validation splits. Keep the same 20% reserved set untouched.
# Each row is one student; the model and the 20% review capacity match the audit above.
from sklearn.base import clone

upv_X = upv_students[upv_features]
upv_y = upv_students['abandono_hash'].eq('A').astype(int)
upv_ids = upv_students['dni_hash'].to_numpy()
development_idx, reserved_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(
        upv_X, upv_y, groups=upv_ids
    )
)

stability_rows = []
for seed in range(43, 63):
    train_local, val_local = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
            upv_X.iloc[development_idx], upv_y.iloc[development_idx],
            groups=upv_ids[development_idx]
        )
    )
    train_idx = development_idx[train_local]
    val_idx = development_idx[val_local]
    assert not (set(upv_ids[train_idx]) & set(upv_ids[val_idx]))
    assert not (set(upv_ids[val_idx]) & set(upv_ids[reserved_idx]))

    fitted = clone(upv_model)
    fitted.fit(upv_X.iloc[train_idx], upv_y.iloc[train_idx])
    actual = upv_y.iloc[val_idx].to_numpy()
    scores = fitted.predict_proba(upv_X.iloc[val_idx])[:, 1]
    selected = np.zeros(len(actual), dtype=bool)
    selected[np.argsort(-scores, kind='stable')[:int(np.ceil(0.20 * len(actual)))]] = True
    found = int(((actual == 1) & selected).sum())
    positives = int((actual == 1).sum())
    row = {
        'seed': seed, 'roc_auc': roc_auc_score(actual, scores),
        'precision_pct': 100 * found / selected.sum(),
        'recall_pct': 100 * found / positives,
        'validation_students': len(actual), 'validation_dropouts': positives,
    }
    study_mode = upv_students['dedicacion'].iloc[val_idx].to_numpy()
    for group in ('TC', 'TP'):
        in_group = study_mode == group
        group_positive = in_group & (actual == 1)
        group_negative = in_group & (actual == 0)
        n_positive = int(group_positive.sum())
        n_negative = int(group_negative.sum())
        row[f'{group}_students'] = int(in_group.sum())
        row[f'{group}_dropouts'] = n_positive
        row[f'{group}_recall_pct'] = (
            100 * (group_positive & selected).sum() / n_positive if n_positive else np.nan
        )
        row[f'{group}_false_positive_rate_pct'] = (
            100 * (group_negative & selected).sum() / n_negative if n_negative else np.nan
        )
    row['TP_minus_TC_recall_pp'] = row['TP_recall_pct'] - row['TC_recall_pct']
    row['TP_minus_TC_false_positive_rate_pp'] = (
        row['TP_false_positive_rate_pct'] - row['TC_false_positive_rate_pct']
    )
    stability_rows.append(row)

upv_stability = pd.DataFrame(stability_rows)
original = upv_stability.loc[upv_stability['seed'].eq(43)].iloc[0]
assert np.isclose(original['roc_auc'], upv_report['model']['validation_roc_auc'])
assert np.isclose(original['recall_pct'], upv_report['review']['recall_pct'])

metrics = [
    'roc_auc', 'precision_pct', 'recall_pct',
    'TC_recall_pct', 'TP_recall_pct', 'TP_minus_TC_recall_pp',
    'TC_false_positive_rate_pct', 'TP_false_positive_rate_pct',
    'TP_minus_TC_false_positive_rate_pp',
    'TP_students', 'TP_dropouts',
]
summary = upv_stability[metrics].agg(['median', 'min', 'max']).T
print('20 repeated splits; range is descriptive, not a confidence interval.')
display(summary)
print('TP recall exceeds TC in', int((upv_stability['TP_minus_TC_recall_pp'] > 0).sum()), 'of 20 splits.')
print('TP false-positive rate exceeds TC in', int((upv_stability['TP_minus_TC_false_positive_rate_pp'] > 0).sum()), 'of 20 splits.')
print('ROC AUC (median/min/max):', upv_stability['roc_auc'].agg(['median', 'min', 'max']).round(3).to_dict())

20 repeated splits; range is descriptive, not a confidence interval.


Metric,Median,Minimum,Maximum
ROC AUC,0.715,0.696,0.739
Review Precision,16.5%,13.9%,19.3%
Review Recall,43.9%,39.8%,49.5%
TC Recall,40.5%,36.4%,47.0%
TP Recall,81.8%,73.9%,89.3%
TP − TC Recall Gap,41.3 pp,32.2 pp,50.0 pp
TC False-Positive Rate,16.6%,16.2%,17.1%
TP False-Positive Rate,74.1%,62.2%,82.8%
TP − TC False-Positive Rate Gap,57.3 pp,45.6 pp,66.4 pp
TP Students,116,106,137


TP recall exceeds TC in 20 of 20 splits.
TP false-positive rate exceeds TC in 20 of 20 splits.
ROC AUC (median/min/max): {'median': 0.715, 'min': 0.696, 'max': 0.739}


In [ ]:
# Paired ablation: same students, same 20 splits, same 20% review capacity.
# Remove study mode from model inputs, but keep it for evaluating TC and TP.
no_mode_features = [column for column in upv_features if column != 'dedicacion']
assert len(no_mode_features) == len(upv_features) - 1

original_preprocessing = upv_model.named_steps['preprocessing']
no_mode_transformers = []
for name, transformer, columns in original_preprocessing.transformers:
    kept_columns = [column for column in columns if column != 'dedicacion']
    if kept_columns:
        no_mode_transformers.append((name, clone(transformer), kept_columns))
assert set(sum((columns for _, _, columns in no_mode_transformers), [])) == set(no_mode_features)
no_mode_template = clone(upv_model)
no_mode_template.set_params(preprocessing=ColumnTransformer(no_mode_transformers))
X_no_mode = upv_students[no_mode_features]

ablation_rows = []
for seed in upv_stability['seed']:
    train_local, val_local = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=int(seed)).split(
            upv_X.iloc[development_idx], upv_y.iloc[development_idx],
            groups=upv_ids[development_idx]
        )
    )
    train_idx = development_idx[train_local]
    val_idx = development_idx[val_local]
    fitted = clone(no_mode_template)
    fitted.fit(X_no_mode.iloc[train_idx], upv_y.iloc[train_idx])
    actual = upv_y.iloc[val_idx].to_numpy()
    scores = fitted.predict_proba(X_no_mode.iloc[val_idx])[:, 1]
    selected = np.zeros(len(actual), dtype=bool)
    selected[np.argsort(-scores, kind='stable')[:int(np.ceil(0.20 * len(actual)))]] = True
    found = int(((actual == 1) & selected).sum())
    row = {
        'seed': int(seed), 'roc_auc': roc_auc_score(actual, scores),
        'precision_pct': 100 * found / selected.sum(),
        'recall_pct': 100 * found / (actual == 1).sum(),
    }
    study_mode = upv_students['dedicacion'].iloc[val_idx].to_numpy()
    for group in ('TC', 'TP'):
        in_group = study_mode == group
        group_positive = in_group & (actual == 1)
        group_negative = in_group & (actual == 0)
        row[f'{group}_recall_pct'] = 100 * (group_positive & selected).sum() / group_positive.sum()
        row[f'{group}_false_positive_rate_pct'] = (
            100 * (group_negative & selected).sum() / group_negative.sum()
        )
        row[f'{group}_selection_pct'] = 100 * (in_group & selected).sum() / in_group.sum()
    row['TP_minus_TC_recall_pp'] = row['TP_recall_pct'] - row['TC_recall_pct']
    row['TP_minus_TC_false_positive_rate_pp'] = (
        row['TP_false_positive_rate_pct'] - row['TC_false_positive_rate_pct']
    )
    ablation_rows.append(row)

upv_without_mode = pd.DataFrame(ablation_rows).set_index('seed').sort_index()
upv_with_mode = upv_stability.set_index('seed').sort_index()
assert upv_without_mode.index.equals(upv_with_mode.index)
metric_labels = {
    'roc_auc': 'ROC AUC',
    'precision_pct': 'Precision in top 20%, %',
    'recall_pct': 'Recall in top 20%, %',
    'TC_recall_pct': 'TC recall, %',
    'TP_recall_pct': 'TP recall, %',
    'TP_minus_TC_recall_pp': 'TP - TC recall gap, pp',
    'TC_false_positive_rate_pct': 'TC false-positive rate, %',
    'TP_false_positive_rate_pct': 'TP false-positive rate, %',
    'TP_minus_TC_false_positive_rate_pp': 'TP - TC false-positive gap, pp',
}
comparison_rows = []
for metric, label in metric_labels.items():
    before = upv_with_mode[metric]
    after = upv_without_mode[metric]
    difference = after - before
    digits = 3 if metric == 'roc_auc' else 1
    comparison_rows.append({
        'metric': label,
        'with_mode_median': f'{before.median():.{digits}f}',
        'without_mode_median': f'{after.median():.{digits}f}',
        'paired_change_median': f'{difference.median():+.{digits}f}',
        'paired_change_range': f'{difference.min():+.{digits}f} to {difference.max():+.{digits}f}',
    })
print('Same 20 splits; change = without dedicacion minus with dedicacion.')
display(pd.DataFrame(comparison_rows).set_index('metric'))
print('TP false-positive rate fell in', int((
    upv_without_mode['TP_false_positive_rate_pct'] < upv_with_mode['TP_false_positive_rate_pct']
).sum()), 'of 20 splits.')
print('TP recall fell in', int((
    upv_without_mode['TP_recall_pct'] < upv_with_mode['TP_recall_pct']
).sum()), 'of 20 splits.')
print('Without mode, TP selection rate median/range:',
      round(upv_without_mode['TP_selection_pct'].median(), 1),
      round(upv_without_mode['TP_selection_pct'].min(), 1),
      round(upv_without_mode['TP_selection_pct'].max(), 1))

Same 20 splits; change = without dedicacion minus with dedicacion.


Metric,With Study Mode,Without Study Mode,Median Change,Change Range
ROC AUC,0.715,0.709,-0.010,-0.014 to -0.002
Review Precision,16.5%,16.1%,-0.4 pp,-1.3 to +0.4 pp
Review Recall,43.9%,43.2%,-1.1 pp,-3.2 to +1.1 pp
TC Recall,40.5%,43.4%,+2.3 pp,+1.0 to +4.4 pp
TP Recall,81.8%,41.9%,-40.9 pp,-53.8 to -29.0 pp
TP − TC Recall Gap,41.3 pp,-2.0 pp,-42.8 pp,-56.1 to -31.2 pp
TC False-Positive Rate,16.6%,18.0%,+1.4 pp,+1.1 to +1.7 pp
TP False-Positive Rate,74.1%,22.9%,-52.2 pp,-61.6 to -39.0 pp
TP − TC False-Positive Rate Gap,57.3 pp,5.0 pp,-53.8 pp,-63.1 to -40.2 pp


TP false-positive rate fell in 20 of 20 splits.
TP recall fell in 20 of 20 splits.
Without mode, TP selection rate median/range: 27.1 17.6 33.6


In [ ]:
review_shares = (0.10, 0.15, 0.20, 0.25, 0.30)
model_variants = (
    ('With Study Mode', upv_model, upv_X, upv_with_mode),
    ('Without Study Mode', no_mode_template, X_no_mode, upv_without_mode),
)

capacity_overall_rows = []
capacity_group_rows = []
for seed in upv_stability['seed'].astype(int):
    train_local, val_local = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
            upv_X.iloc[development_idx], upv_y.iloc[development_idx],
            groups=upv_ids[development_idx]
        )
    )
    train_idx = development_idx[train_local]
    val_idx = development_idx[val_local]
    actual = upv_y.iloc[val_idx].to_numpy()
    positive = actual == 1
    study_mode = upv_students['dedicacion'].iloc[val_idx].to_numpy()

    for model_name, template, features, reference in model_variants:
        fitted = clone(template)
        fitted.fit(features.iloc[train_idx], upv_y.iloc[train_idx])
        scores = fitted.predict_proba(features.iloc[val_idx])[:, 1]
        ranking = np.argsort(-scores, kind='stable')

        for share in review_shares:
            selected = np.zeros(len(actual), dtype=bool)
            selected[ranking[:int(np.ceil(share * len(actual)))]] = True
            selected_count = int(selected.sum())
            found = int((positive & selected).sum())
            false_alarms = int((~positive & selected).sum())
            overall_recall = 100 * found / positive.sum()
            overall_precision = 100 * found / selected_count
            capacity_overall_rows.append({
                'seed': seed, 'model': model_name, 'review_share_pct': 100 * share,
                'students': len(actual), 'selected': selected_count,
                'found': found, 'missed': int(positive.sum()) - found,
                'false_alarms': false_alarms,
                'precision_pct': overall_precision, 'recall_pct': overall_recall,
            })
            if share == 0.20:
                assert np.isclose(overall_recall, reference.loc[seed, 'recall_pct'])
                assert np.isclose(overall_precision, reference.loc[seed, 'precision_pct'])

            for group in ('TC', 'TP'):
                in_group = study_mode == group
                group_positive = in_group & positive
                group_negative = in_group & ~positive
                group_dropouts = int(group_positive.sum())
                group_non_dropouts = int(group_negative.sum())
                group_selected = int((in_group & selected).sum())
                group_found = int((group_positive & selected).sum())
                group_false_alarms = int((group_negative & selected).sum())
                group_recall = 100 * group_found / group_dropouts if group_dropouts else np.nan
                group_fpr = 100 * group_false_alarms / group_non_dropouts if group_non_dropouts else np.nan
                capacity_group_rows.append({
                    'seed': seed, 'model': model_name, 'review_share_pct': 100 * share,
                    'group': group, 'students': int(in_group.sum()),
                    'dropouts': group_dropouts, 'selected': group_selected,
                    'found': group_found, 'missed': group_dropouts - group_found,
                    'false_alarms': group_false_alarms,
                    'selection_pct': 100 * group_selected / in_group.sum(),
                    'recall_pct': group_recall, 'fpr_pct': group_fpr,
                })
                if share == 0.20:
                    assert np.isclose(group_recall, reference.loc[seed, f'{group}_recall_pct'])
                    assert np.isclose(group_fpr, reference.loc[seed, f'{group}_false_positive_rate_pct'])

capacity_overall = pd.DataFrame(capacity_overall_rows)
capacity_groups = pd.DataFrame(capacity_group_rows)
assert len(capacity_overall) == 20 * 2 * len(review_shares)
assert len(capacity_groups) == 20 * 2 * len(review_shares) * 2

model_order = {'With Study Mode': 0, 'Without Study Mode': 1}
capacity_overall_summary = (
    capacity_overall.groupby(['review_share_pct', 'model'], as_index=False)
    .agg(students=('students', 'median'), selected=('selected', 'median'),
         found=('found', 'median'), missed=('missed', 'median'),
         false_alarms=('false_alarms', 'median'),
         precision_pct=('precision_pct', 'median'), recall_pct=('recall_pct', 'median'))
)
capacity_overall_summary['model_order'] = capacity_overall_summary['model'].map(model_order)
capacity_overall_summary = capacity_overall_summary.sort_values(
    ['review_share_pct', 'model_order']
).drop(columns='model_order')
capacity_overall_summary = capacity_overall_summary.rename(columns={
    'review_share_pct': 'Review Share (%)', 'model': 'Model',
    'students': 'Students', 'selected': 'Selected Students',
    'found': 'Found Dropouts', 'missed': 'Missed Dropouts',
    'false_alarms': 'False Alarms', 'precision_pct': 'Precision (%)',
    'recall_pct': 'Recall (%)',
})

capacity_group_summary = (
    capacity_groups.groupby(['review_share_pct', 'model', 'group'], as_index=False)
    .agg(students=('students', 'median'), dropouts=('dropouts', 'median'),
         selected=('selected', 'median'), found=('found', 'median'),
         missed=('missed', 'median'), false_alarms=('false_alarms', 'median'),
         selection_pct=('selection_pct', 'median'),
         recall_pct=('recall_pct', 'median'), fpr_pct=('fpr_pct', 'median'))
)
capacity_group_summary['model_order'] = capacity_group_summary['model'].map(model_order)
capacity_group_summary = capacity_group_summary.sort_values(
    ['review_share_pct', 'model_order', 'group']
).drop(columns='model_order')
capacity_group_summary = capacity_group_summary.rename(columns={
    'review_share_pct': 'Review Share (%)', 'model': 'Model', 'group': 'Group',
    'students': 'Students', 'dropouts': 'Dropouts',
    'selected': 'Selected Students', 'found': 'Found Dropouts',
    'missed': 'Missed Dropouts', 'false_alarms': 'False Alarms',
    'selection_pct': 'Selection Rate (%)', 'recall_pct': 'Recall (%)',
    'fpr_pct': 'False-Positive Rate (%)',
})

print('Medians across the same 20 validation splits; the reserved set is untouched.')
print('Overall results:')
display(capacity_overall_summary)
print('TC and TP results:')
display(capacity_group_summary)

print('Count columns are separate medians across splits and may not add exactly.')
tp_variation = (
    capacity_groups[capacity_groups['group'].eq('TP')]
    .groupby(['review_share_pct', 'model'], as_index=False)
    .agg(dropouts_min=('dropouts', 'min'), dropouts_max=('dropouts', 'max'),
         recall_min=('recall_pct', 'min'), recall_max=('recall_pct', 'max'),
         fpr_min=('fpr_pct', 'min'), fpr_max=('fpr_pct', 'max'))
)
tp_variation['model_order'] = tp_variation['model'].map(model_order)
tp_variation = tp_variation.sort_values(['review_share_pct', 'model_order'])
tp_variation_view = pd.DataFrame({
    'Review Share (%)': tp_variation['review_share_pct'].to_numpy(),
    'Model': tp_variation['model'].to_numpy(),
    'TP Dropouts Per Split': [
        f'{int(low)} to {int(high)}'
        for low, high in zip(tp_variation['dropouts_min'], tp_variation['dropouts_max'])
    ],
    'TP Recall Range': [
        f'{low:.1f}% to {high:.1f}%'
        for low, high in zip(tp_variation['recall_min'], tp_variation['recall_max'])
    ],
    'TP False-Positive Rate Range': [
        f'{low:.1f}% to {high:.1f}%'
        for low, high in zip(tp_variation['fpr_min'], tp_variation['fpr_max'])
    ],
})
print('TP ranges across the 20 splits (descriptive, not confidence intervals):')
display(tp_variation_view)


Medians across the same 20 validation splits; the reserved set is untouched.
Overall results:


Review Share (%),Model,Students,Selected Students,Found Dropouts,Missed Dropouts,False Alarms,Precision (%),Recall (%)
10.0%,With Study Mode,4 086,409,81.5,221.5,327.5,19.9%,26.8%
10.0%,Without Study Mode,4 086,409,78,226.5,331,19.1%,25.5%
15.0%,With Study Mode,4 086,613,111,194.5,502,18.1%,36.1%
15.0%,Without Study Mode,4 086,613,107.5,197,505.5,17.5%,34.9%
20.0%,With Study Mode,4 086,818,135,169,683,16.5%,43.9%
20.0%,Without Study Mode,4 086,818,132,171.5,686,16.1%,43.2%
25.0%,With Study Mode,4 086,1 022,159,147,863,15.6%,52.0%
25.0%,Without Study Mode,4 086,1 022,156.5,147.5,865.5,15.3%,51.3%
30.0%,With Study Mode,4 086,1 226,181.5,126,1 044.5,14.8%,58.5%
30.0%,Without Study Mode,4 086,1 226,176,131,1 050,14.4%,57.3%


TC and TP results:


Review Share (%),Model,Group,Students,Dropouts,Selected Students,Found Dropouts,Missed Dropouts,False Alarms,Selection Rate (%),Recall (%),False-Positive Rate (%)
10.0%,With Study Mode,TC,3 970,277.5,331.5,63.5,214,269,8.4%,22.5%,7.3%
10.0%,With Study Mode,TP,116,24.5,77.5,18,8,60.5,63.1%,68.7%,61.7%
10.0%,Without Study Mode,TC,3 970,277.5,392,71,208,319,9.9%,25.3%,8.6%
10.0%,Without Study Mode,TP,116,24.5,17,7,19,11.5,14.7%,27.3%,12.5%
15.0%,With Study Mode,TC,3 970,277.5,527,90,187.5,436,13.3%,32.1%,11.9%
15.0%,With Study Mode,TP,116,24.5,86,20,5,67,71.4%,79.4%,69.7%
15.0%,Without Study Mode,TC,3 970,277.5,589.5,98.5,179,489.5,14.8%,35.4%,13.3%
15.0%,Without Study Mode,TP,116,24.5,23.5,8,17,15.5,20.2%,33.3%,16.2%
20.0%,With Study Mode,TC,3 970,277.5,729,114,163,611,18.4%,40.5%,16.6%
20.0%,With Study Mode,TP,116,24.5,89,20,4,70,76.0%,81.8%,74.1%


Count columns are separate medians across splits and may not add exactly.
TP ranges across the 20 splits (descriptive, not confidence intervals):


Review Share (%),Model,TP Dropouts Per Split,TP Recall Range,TP False-Positive Rate Range
10.0%,With Study Mode,18 to 33,57.1% to 85.7%,51.6% to 73.1%
10.0%,Without Study Mode,18 to 33,15.4% to 38.7%,7.0% to 20.5%
15.0%,With Study Mode,18 to 33,71.0% to 89.3%,57.3% to 80.0%
15.0%,Without Study Mode,18 to 33,19.2% to 45.2%,10.5% to 25.3%
20.0%,With Study Mode,18 to 33,73.9% to 89.3%,62.2% to 82.8%
20.0%,Without Study Mode,18 to 33,26.9% to 54.8%,12.8% to 28.4%
25.0%,With Study Mode,18 to 33,73.9% to 92.9%,63.4% to 84.9%
25.0%,Without Study Mode,18 to 33,30.8% to 61.3%,19.2% to 34.3%
30.0%,With Study Mode,18 to 33,73.9% to 92.9%,65.9% to 85.3%
30.0%,Without Study Mode,18 to 33,38.1% to 61.3%,23.2% to 37.3%


In [ ]:
# Diagnose score distributions and group calibration on the same 20 validation splits.
# The reserved set remains untouched; study mode is still used to label audit groups.
risk_distribution_rows = []
group_calibration_rows = []
for seed in upv_stability['seed'].astype(int):
    train_local, val_local = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
            upv_X.iloc[development_idx], upv_y.iloc[development_idx],
            groups=upv_ids[development_idx]
        )
    )
    train_idx = development_idx[train_local]
    val_idx = development_idx[val_local]
    actual = upv_y.iloc[val_idx].to_numpy()
    study_mode = upv_students['dedicacion'].iloc[val_idx].to_numpy()
    assert set(study_mode) == {'TC', 'TP'}

    for model_name, template, features, _ in model_variants:
        fitted = clone(template)
        fitted.fit(features.iloc[train_idx], upv_y.iloc[train_idx])
        scores = fitted.predict_proba(features.iloc[val_idx])[:, 1]
        assert len(scores) == len(actual) and np.isfinite(scores).all()

        for group in ('TC', 'TP'):
            in_group = study_mode == group
            group_scores = scores[in_group]
            group_actual = actual[in_group]
            mean_predicted = 100 * group_scores.mean()
            observed = 100 * group_actual.mean()
            group_calibration_rows.append({
                'seed': seed, 'model': model_name, 'group': group,
                'students': len(group_actual), 'dropouts': int(group_actual.sum()),
                'mean_predicted_pct': mean_predicted,
                'observed_pct': observed, 'gap_pp': observed - mean_predicted,
            })
            for outcome_value, outcome_name in ((0, 'Non-Dropout'), (1, 'Dropout')):
                outcome_scores = group_scores[group_actual == outcome_value]
                assert len(outcome_scores) > 0
                p10, p50, p90 = np.percentile(outcome_scores, (10, 50, 90))
                risk_distribution_rows.append({
                    'seed': seed, 'model': model_name, 'group': group,
                    'outcome': outcome_name, 'students': len(outcome_scores),
                    'p10_pct': 100 * p10, 'median_pct': 100 * p50,
                    'p90_pct': 100 * p90,
                })

risk_distribution = pd.DataFrame(risk_distribution_rows)
group_calibration = pd.DataFrame(group_calibration_rows)
assert len(risk_distribution) == 20 * 2 * 2 * 2
assert len(group_calibration) == 20 * 2 * 2

risk_distribution_summary = (
    risk_distribution.groupby(['model', 'group', 'outcome'], as_index=False)
    .agg(students=('students', 'median'), p10_pct=('p10_pct', 'median'),
         median_pct=('median_pct', 'median'), p90_pct=('p90_pct', 'median'))
)
risk_distribution_summary['model_order'] = risk_distribution_summary['model'].map(model_order)
risk_distribution_summary['outcome_order'] = risk_distribution_summary['outcome'].map(
    {'Non-Dropout': 0, 'Dropout': 1}
)
risk_distribution_summary = risk_distribution_summary.sort_values(
    ['model_order', 'group', 'outcome_order']
).drop(columns=['model_order', 'outcome_order']).rename(columns={
    'model': 'Model', 'group': 'Group', 'outcome': 'Actual Outcome',
    'students': 'Students', 'p10_pct': 'P10 Predicted Risk (%)',
    'median_pct': 'Median Predicted Risk (%)', 'p90_pct': 'P90 Predicted Risk (%)',
})

calibration_summary = (
    group_calibration.groupby(['model', 'group'], as_index=False)
    .agg(students=('students', 'median'), dropouts=('dropouts', 'median'),
         mean_predicted_pct=('mean_predicted_pct', 'median'),
         observed_pct=('observed_pct', 'median'), gap_pp=('gap_pp', 'median'),
         gap_min=('gap_pp', 'min'), gap_max=('gap_pp', 'max'))
)
calibration_summary['model_order'] = calibration_summary['model'].map(model_order)
calibration_summary = calibration_summary.sort_values(['model_order', 'group'])
calibration_summary['Gap Range (pp)'] = [
    f'{low:+.1f} to {high:+.1f} pp'
    for low, high in zip(calibration_summary['gap_min'], calibration_summary['gap_max'])
]
calibration_summary = calibration_summary.drop(columns=['model_order', 'gap_min', 'gap_max']).rename(columns={
    'model': 'Model', 'group': 'Group', 'students': 'Students', 'dropouts': 'Dropouts',
    'mean_predicted_pct': 'Mean Predicted Risk (%)',
    'observed_pct': 'Observed Dropout Rate (%)',
    'gap_pp': 'Observed − Predicted (pp)',
})

print('Medians across the same 20 validation splits; the reserved set is untouched.')
print('Risk score distribution by study mode and actual outcome:')
display(risk_distribution_summary)
print('Group-level calibration (positive gap means observed dropout exceeds predicted risk):')
display(calibration_summary)
print('Each quantile and count is a separate median across splits; gap ranges are descriptive, not confidence intervals.')

Medians across the same 20 validation splits; the reserved set is untouched.
Risk score distribution by study mode and actual outcome:


Model,Group,Actual Outcome,Students,P10 Predicted Risk (%),Median Predicted Risk (%),P90 Predicted Risk (%)
With Study Mode,TC,Non-Dropout,3 682.5,1.3%,4.8%,12.8%
With Study Mode,TC,Dropout,277.5,2.8%,8.9%,23.0%
With Study Mode,TP,Non-Dropout,95,5.3%,17.2%,34.4%
With Study Mode,TP,Dropout,24.5,7.5%,21.7%,54.5%
Without Study Mode,TC,Non-Dropout,3 682.5,1.4%,5.0%,13.4%
Without Study Mode,TC,Dropout,277.5,3.0%,9.3%,24.0%
Without Study Mode,TP,Non-Dropout,95,1.7%,6.7%,15.4%
Without Study Mode,TP,Dropout,24.5,2.6%,8.1%,29.7%


Group-level calibration (positive gap means observed dropout exceeds predicted risk):


Model,Group,Students,Dropouts,Mean Predicted Risk (%),Observed Dropout Rate (%),Observed − Predicted (pp),Gap Range (pp)
With Study Mode,TC,3 970,277.5,6.8%,7.0%,0.2 pp,-0.4 to +1.2 pp
With Study Mode,TP,116,24.5,21.1%,20.9%,-0.1 pp,-7.4 to +9.5 pp
Without Study Mode,TC,3 970,277.5,7.2%,7.0%,-0.2 pp,-0.8 to +0.9 pp
Without Study Mode,TP,116,24.5,9.6%,20.9%,11.4 pp,+5.8 to +19.0 pp


Each quantile and count is a separate median across splits; gap ranges are descriptive, not confidence intervals.


In [ ]:
# Discrimination within TC and TP; selection is the top 20% of all validation students.
within_group_rows = []
for seed in upv_stability['seed'].astype(int):
    train_local, val_local = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
            upv_X.iloc[development_idx], upv_y.iloc[development_idx],
            groups=upv_ids[development_idx]
        )
    )
    train_idx = development_idx[train_local]
    val_idx = development_idx[val_local]
    actual = upv_y.iloc[val_idx].to_numpy()
    study_mode = upv_students['dedicacion'].iloc[val_idx].to_numpy()

    for model_name, template, features, _ in model_variants:
        fitted = clone(template)
        fitted.fit(features.iloc[train_idx], upv_y.iloc[train_idx])
        scores = fitted.predict_proba(features.iloc[val_idx])[:, 1]
        selected = np.zeros(len(actual), dtype=bool)
        selected[np.argsort(-scores, kind='stable')[:int(np.ceil(0.20 * len(actual)))]] = True

        for group in ('TC', 'TP'):
            in_group = study_mode == group
            group_actual = actual[in_group]
            group_scores = scores[in_group]
            group_selected = selected[in_group]
            assert set(group_actual) == {0, 1} and group_selected.any()
            baseline = 100 * group_actual.mean()
            precision = 100 * group_actual[group_selected].mean()
            within_group_rows.append({
                'seed': seed, 'model': model_name, 'group': group,
                'students': len(group_actual), 'dropouts': int(group_actual.sum()),
                'auc': roc_auc_score(group_actual, group_scores),
                'baseline_pct': baseline, 'selected': int(group_selected.sum()),
                'precision_pct': precision, 'lift': precision / baseline,
            })

within_group = pd.DataFrame(within_group_rows)
assert len(within_group) == 20 * 2 * 2
within_group_summary = (
    within_group.groupby(['model', 'group'], as_index=False)
    .agg(students=('students', 'median'), dropouts=('dropouts', 'median'),
         auc=('auc', 'median'), auc_min=('auc', 'min'), auc_max=('auc', 'max'),
         baseline_pct=('baseline_pct', 'median'), selected=('selected', 'median'),
         precision_pct=('precision_pct', 'median'), lift=('lift', 'median'))
)
within_group_summary['model_order'] = within_group_summary['model'].map(model_order)
within_group_summary = within_group_summary.sort_values(['model_order', 'group'])
within_group_summary['ROC AUC Range'] = [
    f'{low:.3f} to {high:.3f}'
    for low, high in zip(within_group_summary['auc_min'], within_group_summary['auc_max'])
]
within_group_summary = within_group_summary.drop(
    columns=['model_order', 'auc_min', 'auc_max']
).rename(columns={
    'model': 'Model', 'group': 'Group', 'students': 'Students',
    'dropouts': 'Dropouts', 'auc': 'Within-Group ROC AUC',
    'baseline_pct': 'Dropout Rate (%)', 'selected': 'Selected Students',
    'precision_pct': 'Selected Precision (%)', 'lift': 'Precision Lift',
})
print('Medians across the same 20 validation splits; the reserved set is untouched.')
print('Top 20% is selected across all students, not separately within each group.')
display(within_group_summary)
print('ROC AUC ranges are descriptive, not confidence intervals.')

Medians across the same 20 validation splits; the reserved set is untouched.
Top 20% is selected across all students, not separately within each group.


Model,Group,Students,Dropouts,Within-Group ROC AUC,Dropout Rate (%),Selected Students,Selected Precision (%),Precision Lift,ROC AUC Range
With Study Mode,TC,3 970,277.5,0.711,7.0%,729,15.7%,2.2,0.691 to 0.738
With Study Mode,TP,116,24.5,0.619,20.9%,89,22.9%,1.1,0.517 to 0.681
Without Study Mode,TC,3 970,277.5,0.712,7.0%,786,15.2%,2.2,0.693 to 0.742
Without Study Mode,TP,116,24.5,0.611,20.9%,32,32.9%,1.6,0.514 to 0.685


ROC AUC ranges are descriptive, not confidence intervals.


In [ ]:
# One-time final check on the 20% reserved students after development analyses.
# Fit each pre-specified model on the full 80% development set and select the global top 20%.
assert not (set(upv_ids[development_idx]) & set(upv_ids[reserved_idx]))
assert len(np.unique(upv_ids[reserved_idx])) == len(reserved_idx)
reserved_actual = upv_y.iloc[reserved_idx].to_numpy()
reserved_mode = upv_students['dedicacion'].iloc[reserved_idx].to_numpy()
assert set(reserved_actual) == {0, 1} and set(reserved_mode) == {'TC', 'TP'}

holdout_overall_rows = []
holdout_group_rows = []
holdout_diagnostic_rows = []
for model_name, template, features, _ in model_variants:
    fitted = clone(template)
    fitted.fit(features.iloc[development_idx], upv_y.iloc[development_idx])
    scores = fitted.predict_proba(features.iloc[reserved_idx])[:, 1]
    selected = np.zeros(len(reserved_actual), dtype=bool)
    selected[np.argsort(-scores, kind='stable')[:int(np.ceil(0.20 * len(scores)))]] = True
    found = int(((reserved_actual == 1) & selected).sum())
    false_alarms = int(((reserved_actual == 0) & selected).sum())
    dropouts = int(reserved_actual.sum())
    holdout_overall_rows.append({
        'Model': model_name, 'Students': len(reserved_actual),
        'Dropouts': dropouts, 'Selected Students': int(selected.sum()),
        'Found Dropouts': found, 'Missed Dropouts': dropouts - found,
        'False Alarms': false_alarms, 'ROC AUC': roc_auc_score(reserved_actual, scores),
        'Precision (%)': 100 * found / selected.sum(),
        'Recall (%)': 100 * found / dropouts,
    })

    for group in ('TC', 'TP'):
        in_group = reserved_mode == group
        group_actual = reserved_actual[in_group]
        group_scores = scores[in_group]
        group_selected = selected[in_group]
        n_positive = int(group_actual.sum())
        n_negative = len(group_actual) - n_positive
        n_selected = int(group_selected.sum())
        n_found = int(((group_actual == 1) & group_selected).sum())
        n_false = int(((group_actual == 0) & group_selected).sum())
        baseline = 100 * n_positive / len(group_actual)
        selected_precision = 100 * n_found / n_selected if n_selected else np.nan
        group_auc = roc_auc_score(group_actual, group_scores) if n_positive and n_negative else np.nan
        holdout_group_rows.append({
            'Model': model_name, 'Group': group, 'Students': len(group_actual),
            'Dropouts': n_positive, 'Selected Students': n_selected,
            'Found Dropouts': n_found, 'Missed Dropouts': n_positive - n_found,
            'False Alarms': n_false, 'Selection Rate (%)': 100 * n_selected / len(group_actual),
            'Recall (%)': 100 * n_found / n_positive if n_positive else np.nan,
            'False-Positive Rate (%)': 100 * n_false / n_negative if n_negative else np.nan,
        })
        holdout_diagnostic_rows.append({
            'Model': model_name, 'Group': group, 'Within-Group ROC AUC': group_auc,
            'Dropout Rate (%)': baseline, 'Selected Precision (%)': selected_precision,
            'Precision Lift': selected_precision / baseline if baseline else np.nan,
            'Mean Predicted Risk (%)': 100 * group_scores.mean(),
            'Observed − Predicted (pp)': baseline - 100 * group_scores.mean(),
        })

holdout_overall = pd.DataFrame(holdout_overall_rows)
holdout_groups = pd.DataFrame(holdout_group_rows)
holdout_diagnostics = pd.DataFrame(holdout_diagnostic_rows)
assert len(holdout_overall) == 2 and len(holdout_groups) == 4
assert holdout_overall['Selected Students'].eq(int(np.ceil(0.20 * len(reserved_actual)))).all()
print('Final one-time check on reserved UPV 2022 students; top 20% selected across all students.')
print('Overall results:')
display(holdout_overall)
print('TC and TP results:')
display(holdout_groups)
print('Within-group discrimination and mean risk:')
display(holdout_diagnostics)
print('TP is a small group; these are descriptive results from one reserved split.')

Final one-time check on reserved UPV 2022 students; top 20% selected across all students.
Overall results:


Model,Students,Dropouts,Selected Students,Found Dropouts,Missed Dropouts,False Alarms,ROC AUC,Precision (%),Recall (%)
With Study Mode,4 086,273,818,114,159,704,0.708,13.9%,41.8%
Without Study Mode,4 086,273,818,115,158,703,0.692,14.1%,42.1%


TC and TP results:


Model,Group,Students,Dropouts,Selected Students,Found Dropouts,Missed Dropouts,False Alarms,Selection Rate (%),Recall (%),False-Positive Rate (%)
With Study Mode,TC,3 971,251,730,98,153,632,18.4%,39.0%,17.0%
With Study Mode,TP,115,22,88,16,6,72,76.5%,72.7%,77.4%
Without Study Mode,TC,3 971,251,799,110,141,689,20.1%,43.8%,18.5%
Without Study Mode,TP,115,22,19,5,17,14,16.5%,22.7%,15.1%


Within-group discrimination and mean risk:


Model,Group,Within-Group ROC AUC,Dropout Rate (%),Selected Precision (%),Precision Lift,Mean Predicted Risk (%),Observed − Predicted (pp)
With Study Mode,TC,0.707,6.3%,13.4%,2.1,7.0%,-0.7 pp
With Study Mode,TP,0.472,19.1%,18.2%,1.0,19.8%,-0.7 pp
Without Study Mode,TC,0.706,6.3%,13.8%,2.2,7.4%,-1.0 pp
Without Study Mode,TP,0.459,19.1%,26.3%,1.4,9.2%,9.9 pp


TP is a small group; these are descriptive results from one reserved split.


In [ ]:
# Dataset-specific inputs; the audit logic below is the same for both datasets.
audit_cases = [
    {
        'name': 'OULAD', 'dataset': data, 'record_unit': 'Course Enrollment',
        'target_column': 'final_result', 'positive_value': 'Withdrawn',
        'entity_id_column': 'id_student', 'feature_columns': feature_columns,
        'group_columns': ['imd_band'], 'review_share': 0.20,
    },
    {
        'name': 'UPV 2022', 'dataset': upv_students, 'record_unit': 'Student',
        'target_column': 'abandono_hash', 'positive_value': 'A',
        'entity_id_column': 'dni_hash', 'feature_columns': upv_features,
        'group_columns': ['dedicacion', 'desplazado_hash', 'estudios_m_hash'],
        'review_share': 0.20,
    },
]

case_reports = {}
case_models = {}
case_rows = []
for case in audit_cases:
    report, fitted_model = run_audit(
        case['dataset'], target_column=case['target_column'],
        positive_value=case['positive_value'],
        entity_id_column=case['entity_id_column'],
        feature_columns=case['feature_columns'],
        group_columns=case['group_columns'],
        review_share=case['review_share'],
    )
    case_reports[case['name']] = report
    case_models[case['name']] = fitted_model
    review = report['review']
    case_rows.append({
        'Dataset': case['name'], 'Record Unit': case['record_unit'],
        'Positive Outcome': case['positive_value'],
        'Group Features': ', '.join(_ROW_LABELS.get(column, column.replace('_', ' ').title()) for column in case['group_columns']) if case['group_columns'] else 'None',
        'Feature Count': len(case['feature_columns']),
        'Validation Records': report['data']['validation_records'],
        'Selected Records': review['selected_records'],
        'Found Cases': review['found_positives'],
        'Missed Cases': review['missed_positives'],
        'False Alarms': review['false_alarms'],
        'ROC AUC': report['model']['validation_roc_auc'],
        'Precision (%)': review['precision_pct'],
        'Recall (%)': review['recall_pct'],
        'Group Audit Rows': len(report['group_audit']),
    })

# The configured runs reproduce the earlier 20% validation audits.
assert case_reports['OULAD']['review']['selected_records'] == review_result['selected_records']
assert case_reports['OULAD']['review']['found_positives'] == review_result['found_positives']
assert case_reports['UPV 2022']['review']['selected_records'] == upv_report['review']['selected_records']
assert case_reports['UPV 2022']['review']['found_positives'] == upv_report['review']['found_positives']

# Group features are optional: the overall audit still works with an empty list.
upv_no_groups, _ = run_audit(
    upv_students, target_column='abandono_hash', positive_value='A',
    entity_id_column='dni_hash', feature_columns=upv_features,
    group_columns=[], review_share=0.20,
)
assert upv_no_groups['group_audit'].empty
assert upv_no_groups['group_gaps'].empty
assert upv_no_groups['review'] == case_reports['UPV 2022']['review']

print('The same audit function, with explicit inputs for each dataset.')
print('These are validation results; the final UPV reserved-set check is shown above.')
display(pd.DataFrame(case_rows))
print('No-group check passed: the overall audit is unchanged and group results are empty.')

The same audit function, with explicit inputs for each dataset.
These are validation results; the final UPV reserved-set check is shown above.


Dataset,Record Unit,Positive Outcome,Group Features,Feature Count,Validation Records,Selected Records,Found Cases,Missed Cases,False Alarms,ROC AUC,Precision (%),Recall (%),Group Audit Rows
OULAD,Course Enrollment,Withdrawn,IMD Band,10,6 538,1 308,642,1 370,666,0.660,49.1%,31.9%,11
UPV 2022,Student,A,"Study Mode, Displacement Status, Mother's Education",11,4 086,818,133,159,685,0.731,16.3%,45.5%,11


No-group check passed: the overall audit is unchanged and group results are empty.


## Third dataset: student dropout in Portugal

We run the same audit on [Predict Students' Dropout and Academic Success](https://archive.ics.uci.edu/dataset/697/predict%2Bstudents%2Bdropout%2Band%2Bacademic%2Bsuccess). Each row represents one student; the results below also compare daytime and evening groups.

In [ ]:
# UCI student dropout: a reproducible exploratory run of the same audit.
import io
import urllib.request
import zipfile

uci_source = 'https://archive.ics.uci.edu/static/public/697/predict%2Bstudents%2Bdropout%2Band%2Bacademic%2Bsuccess.zip'
with urllib.request.urlopen(uci_source) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
        uci_data = pd.read_csv(archive.open('data.csv'), sep=';')
uci_data.columns = uci_data.columns.str.strip()

# Semester outcomes are excluded from the predictors.
uci_features = [
    column for column in uci_data.columns
    if column != 'Target'
    and not column.startswith('Curricular units ')
    and column not in ['Unemployment rate', 'Inflation rate', 'GDP']
]
uci_categorical = [
    'Marital status', 'Application mode', 'Course',
    'Daytime/evening attendance', 'Previous qualification', 'Nacionality',
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation", 'Displaced',
    'Educational special needs', 'Debtor', 'Tuition fees up to date',
    'Gender', 'Scholarship holder', 'International',
]
for column in uci_categorical:
    uci_data[column] = uci_data[column].astype(str)
uci_data['Daytime/evening attendance'] = (
    uci_data['Daytime/evening attendance']
    .map({'1': 'Daytime', '0': 'Evening'})
)
assert len(uci_data) == 4424 and len(uci_features) == 21
assert not uci_data['Daytime/evening attendance'].isna().any()

uci_report, uci_model = run_audit(
    dataset=uci_data,
    target_column='Target',
    positive_value='Dropout',
    entity_id_column=None,
    group_columns=['Daytime/evening attendance'],
    review_share=0.20,
    feature_columns=uci_features,
)
uci_overall = pd.DataFrame([{
    'Dataset': 'UCI Student Dropout',
    'Record Unit': 'Student',
    'Feature Count': len(uci_features),
    'Validation Records': uci_report['data']['validation_records'],
    'Dropouts': round(uci_report['data']['validation_positive_rate_pct']
                      * uci_report['data']['validation_records'] / 100),
    'Selected Records': uci_report['review']['selected_records'],
    'Found Dropouts': uci_report['review']['found_positives'],
    'False Alarms': uci_report['review']['false_alarms'],
    'ROC AUC': uci_report['model']['validation_roc_auc'],
    'Precision (%)': uci_report['review']['precision_pct'],
    'Recall (%)': uci_report['review']['recall_pct'],
}])
print('Exploratory validation run; the reserved split is untouched.')
print('Enrolled and Graduate are both counted as non-dropout here.')
display(uci_overall)

uci_groups = uci_report['group_audit'].copy()
uci_groups['Size Check'] = uci_groups['group_value'].apply(
    lambda value: 'Small Group'
    if value in set(uci_report['small_groups']['group_value'])
    else 'Passed'
)
uci_groups = uci_groups.rename(columns={
    'group_value': 'Group', 'records': 'Students',
    'total_positives': 'Dropouts', 'selected_records': 'Selected Students',
    'found_positives': 'Found Dropouts', 'false_alarms': 'False Alarms',
    'recall_pct': 'Recall (%)',
    'false_positive_rate_pct': 'False-Positive Rate (%)',
})[
    ['Group', 'Students', 'Dropouts', 'Selected Students',
     'Found Dropouts', 'False Alarms', 'Recall (%)',
     'False-Positive Rate (%)', 'Size Check']
]
print('Daytime and evening groups:')
display(uci_groups)


Exploratory validation run; the reserved split is untouched.
Enrolled and Graduate are both counted as non-dropout here.


Dataset,Record Unit,Feature Count,Validation Records,Dropouts,Selected Records,Found Dropouts,False Alarms,ROC AUC,Precision (%),Recall (%)
UCI Student Dropout,Student,21,885,274,177,126,51,0.803,71.2%,46.0%


Daytime and evening groups:


Group,Students,Dropouts,Selected Students,Found Dropouts,False Alarms,Recall (%),False-Positive Rate (%),Size Check
Daytime,799,232,152,107,45,46.1%,7.9%,Passed
Evening,86,42,25,19,6,45.2%,13.6%,Small Group


## Separating the audit from model training (UCI example)

The audit below takes observed outcomes, group labels, and predicted dropout probabilities as input. It does not fit a model. The UCI example uses the predictions from the previous section and checks that the audit results match.

In [ ]:
"""Audit observed outcomes and existing dropout probabilities without fitting a model."""

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def audit_predictions(
    dataset,
    target_column,
    positive_value,
    score_column,
    group_columns=None,
    entity_id_column=None,
    review_share=0.20,
):
    """Audit one scored record per decision unit; scores must be probabilities.

    This function does not split data or fit a model. The caller is responsible
    for supplying predictions made without training on these outcomes.
    """
    if not isinstance(dataset, pd.DataFrame) or dataset.empty:
        raise ValueError("dataset must be a non-empty pandas DataFrame")
    if not 0 < review_share <= 1:
        raise ValueError("review_share must be greater than 0 and at most 1")
    if isinstance(group_columns, str):
        group_columns = [group_columns]
    group_columns = list(group_columns or [])
    if len(group_columns) != len(set(group_columns)):
        raise ValueError("group_columns contains duplicates")
    if target_column == score_column or score_column in group_columns:
        raise ValueError("The score column must be separate from outcome and groups")

    required = [target_column, score_column, *group_columns]
    if entity_id_column is not None:
        required.append(entity_id_column)
    missing = [column for column in required if column not in dataset.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if dataset[target_column].isna().any():
        raise ValueError("Target column contains missing values")
    if positive_value not in dataset[target_column].unique():
        raise ValueError(f"Positive value not found: {positive_value}")
    if entity_id_column is not None and dataset[entity_id_column].isna().any():
        raise ValueError("Entity ID column contains missing values")

    try:
        probabilities = pd.to_numeric(dataset[score_column], errors="raise").to_numpy(dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError("Score column must contain numeric probabilities") from exc
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Score column must contain finite probabilities between 0 and 1")
    actual = dataset[target_column].eq(positive_value).to_numpy(dtype=int)
    if np.unique(actual).size != 2:
        raise ValueError("Dataset must contain both outcome classes")

    records_to_review = int(np.ceil(review_share * len(actual)))
    ranked_positions = np.argsort(-probabilities, kind="stable")
    selected = np.zeros(len(actual), dtype=bool)
    selected[ranked_positions[:records_to_review]] = True
    found = int(((actual == 1) & selected).sum())
    false_alarms = int(((actual == 0) & selected).sum())
    total_positives = int((actual == 1).sum())
    review = {
        "review_share_pct": 100 * review_share,
        "records": len(actual),
        "selected_records": records_to_review,
        "found_positives": found,
        "false_alarms": false_alarms,
        "missed_positives": total_positives - found,
        "precision_pct": 100 * found / records_to_review,
        "recall_pct": 100 * found / total_positives,
    }
    if entity_id_column is not None:
        ids = dataset[entity_id_column].to_numpy()
        review["unique_entities_to_review"] = len(np.unique(ids[selected]))

    group_rows = []
    for column in group_columns:
        values = dataset[column].astype("object").fillna("Unknown").astype(str).to_numpy()
        for group_value in np.unique(values):
            in_group = values == group_value
            positives = in_group & (actual == 1)
            negatives = in_group & (actual == 0)
            group_size = int(in_group.sum())
            total_group_positives = int(positives.sum())
            total_group_negatives = int(negatives.sum())
            found_group = int((positives & selected).sum())
            false_group = int((negatives & selected).sum())
            group_rows.append({
                "group_column": column,
                "group_value": group_value,
                "records": group_size,
                "total_positives": total_group_positives,
                "total_negatives": total_group_negatives,
                "positive_rate_pct": 100 * total_group_positives / group_size,
                "selected_records": int((in_group & selected).sum()),
                "selection_rate_pct": 100 * (in_group & selected).sum() / group_size,
                "found_positives": found_group,
                "false_alarms": false_group,
                "recall_pct": (
                    100 * found_group / total_group_positives
                    if total_group_positives else np.nan
                ),
                "false_positive_rate_pct": (
                    100 * false_group / total_group_negatives
                    if total_group_negatives else np.nan
                ),
            })
    group_audit = pd.DataFrame(group_rows)
    minimum_group_records = max(100, int(np.ceil(0.05 * len(actual))))
    minimum_outcomes_per_class = 30
    gap_rows = []
    small_groups = pd.DataFrame(columns=[
        "group_column", "group_value", "records", "total_positives", "total_negatives"
    ])
    if not group_audit.empty:
        size_check = group_audit.copy()
        size_check["passes_size_check"] = (
            (size_check["records"] >= minimum_group_records)
            & (size_check["total_positives"] >= minimum_outcomes_per_class)
            & (size_check["total_negatives"] >= minimum_outcomes_per_class)
        )
        small_groups = size_check.loc[
            ~size_check["passes_size_check"], small_groups.columns
        ]
        for column, groups in size_check.groupby("group_column"):
            comparable = groups[groups["passes_size_check"]]
            if len(comparable) < 2:
                continue
            for metric in ["recall_pct", "false_positive_rate_pct"]:
                lowest = comparable.loc[comparable[metric].idxmin()]
                highest = comparable.loc[comparable[metric].idxmax()]
                gap_rows.append({
                    "group_column": column,
                    "metric": metric,
                    "lowest_group": lowest["group_value"],
                    "lowest_pct": lowest[metric],
                    "highest_group": highest["group_value"],
                    "highest_pct": highest[metric],
                    "gap_pp": highest[metric] - lowest[metric],
                })
    group_gaps = pd.DataFrame(gap_rows)

    calibration_data = pd.DataFrame({"actual": actual, "probability": probabilities})
    if np.unique(probabilities).size == 1:
        calibration_data["risk_band"] = "All probabilities equal"
    else:
        calibration_data["risk_band"] = pd.qcut(
            calibration_data["probability"], q=5, duplicates="drop"
        )
    calibration = (
        calibration_data.groupby("risk_band", observed=True)
        .agg(
            records=("actual", "size"),
            mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
            actual_positive_pct=("actual", lambda values: 100 * values.mean()),
        )
        .reset_index()
    )
    calibration["gap_pp"] = (
        calibration["actual_positive_pct"] - calibration["mean_predicted_pct"]
    )
    return {
        "configuration": {
            "target_column": target_column,
            "positive_value": positive_value,
            "score_column": score_column,
            "entity_id_column": entity_id_column,
            "group_columns": group_columns,
            "review_share": review_share,
            "selection_unit": "record",
        },
        "data": {
            "records": len(actual),
            "positive_rate_pct": 100 * actual.mean(),
        },
        "scores": {
            "roc_auc": roc_auc_score(actual, probabilities),
            "mean_predicted_pct": 100 * probabilities.mean(),
            "observed_positive_pct": 100 * actual.mean(),
        },
        "review": review,
        "group_audit": group_audit,
        "group_gaps": group_gaps,
        "small_groups": small_groups,
        "calibration": calibration,
        "screening_rule": {
            "minimum_group_records": minimum_group_records,
            "minimum_outcomes_per_class": minimum_outcomes_per_class,
            "is_statistical_test": False,
        },
    }


In [ ]:
# Prepare the existing UCI model's validation predictions as a separate input table.
from sklearn.model_selection import GroupShuffleSplit

uci_row_groups = np.arange(len(uci_data))
uci_development_positions, _ = next(
    GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(
        uci_data, groups=uci_row_groups
    )
)
_, uci_validation_local_positions = next(
    GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43).split(
        uci_data.iloc[uci_development_positions],
        groups=uci_row_groups[uci_development_positions],
    )
)
uci_validation_positions = uci_development_positions[uci_validation_local_positions]
uci_scored = uci_data.iloc[uci_validation_positions][
    ['Target', 'Daytime/evening attendance']
].copy()
uci_scored['Predicted Dropout Risk'] = uci_model.predict_proba(
    uci_data[uci_features].iloc[uci_validation_positions]
)[:, 1]

# The audit uses the completed score table; it does not train or split a model.
uci_prediction_audit = audit_predictions(
    uci_scored,
    target_column='Target',
    positive_value='Dropout',
    score_column='Predicted Dropout Risk',
    group_columns=['Daytime/evening attendance'],
    review_share=0.20,
)
assert uci_prediction_audit['review'] == uci_report['review']
pd.testing.assert_frame_equal(
    uci_prediction_audit['group_audit'], uci_report['group_audit']
)
pd.testing.assert_frame_equal(
    uci_prediction_audit['calibration'], uci_report['calibration']
)

print('Audit of supplied predictions: no model training or data split inside audit_predictions.')
print('Overall, group, and calibration results match the earlier UCI run.')
display(pd.DataFrame([{
    'Dataset': 'UCI Student Dropout',
    'Validation Records': uci_prediction_audit['review']['records'],
    'Dropouts': int(uci_scored['Target'].eq('Dropout').sum()),
    'Selected Records': uci_prediction_audit['review']['selected_records'],
    'Found Dropouts': uci_prediction_audit['review']['found_positives'],
    'False Alarms': uci_prediction_audit['review']['false_alarms'],
    'ROC AUC': uci_prediction_audit['scores']['roc_auc'],
    'Precision (%)': uci_prediction_audit['review']['precision_pct'],
    'Recall (%)': uci_prediction_audit['review']['recall_pct'],
}]))

Audit of supplied predictions: no model training or data split inside audit_predictions.
Overall, group, and calibration results match the earlier UCI run.


Dataset,Validation Records,Dropouts,Selected Records,Found Dropouts,False Alarms,ROC AUC,Precision (%),Recall (%)
UCI Student Dropout,885,274,177,126,51,0.803,71.2%,46.0%


## UPV 2018: testing the audit on an earlier academic year

Source: [StudentDropoutDataset](https://zenodo.org/records/17239943). The record unit is one student after checking and collapsing course rows. This run tests whether the same input contract and group audit work on a different year. It is a within-year validation, not evidence that a model trained in 2022 transfers to 2018.

In [ ]:
# UPV 2018: reuse the same prediction-time features and audit contract.
upv_2018_url = 'https://zenodo.org/records/17239943/files/dataset_2018_hash.zip?download=1'
upv_2018_response = requests.get(upv_2018_url, timeout=120)
upv_2018_response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(upv_2018_response.content)) as archive:
    upv_2018_raw = pd.read_csv(archive.open('dataset_2018_hash.csv'), sep=';', low_memory=False, usecols=['dni_hash', 'abandono_hash', *upv_features])
upv_2018_columns = ['abandono_hash', *upv_features]
upv_2018_conflicts = upv_2018_raw.groupby('dni_hash')[upv_2018_columns].nunique(dropna=False).gt(1).any(axis=1).sum()
if upv_2018_raw['dni_hash'].isna().any() or upv_2018_conflicts:
    raise ValueError(f'UPV 2018 has missing IDs or {upv_2018_conflicts} conflicting student records')
upv_2018_students = upv_2018_raw.drop_duplicates('dni_hash').copy()
if not upv_2018_students['abandono_hash'].isin(['A', 'B']).all():
    raise ValueError('Unexpected UPV 2018 outcome code')
upv_2018_report, upv_2018_model = run_audit(
    upv_2018_students, target_column='abandono_hash', positive_value='A',
    entity_id_column='dni_hash', feature_columns=upv_features,
    group_columns=['dedicacion'], review_share=0.20,
)
upv_2018_review = upv_2018_report['review']
display(pd.DataFrame([{
    'Dataset': 'UPV 2018', 'Record Unit': 'Student',
    'Course Records': len(upv_2018_raw), 'Students': len(upv_2018_students),
    'Dropouts': int(upv_2018_students['abandono_hash'].eq('A').sum()),
    'Validation Records': upv_2018_report['data']['validation_records'],
    'Selected Records': upv_2018_review['selected_records'],
    'Found Dropouts': upv_2018_review['found_positives'],
    'False Alarms': upv_2018_review['false_alarms'],
    'ROC AUC': upv_2018_report['model']['validation_roc_auc'],
    'Precision (%)': upv_2018_review['precision_pct'],
    'Recall (%)': upv_2018_review['recall_pct'],
}]))
display(upv_2018_report['group_audit'][['group_value', 'records', 'total_positives', 'selected_records', 'found_positives', 'false_alarms', 'recall_pct']].rename(columns={
    'group_value':'Group', 'records':'Students', 'total_positives':'Dropouts',
    'selected_records':'Selected Students', 'found_positives':'Found Dropouts',
    'false_alarms':'False Alarms', 'recall_pct':'Recall (%)',
}))

Dataset,Record Unit,Course Records,Students,Dropouts,Validation Records,Selected Records,Found Dropouts,False Alarms,ROC AUC,Precision (%),Recall (%)
UPV 2018,Student,152 446,20 233,1 436,4 047,810,132,678,0.735,16.3%,49.4%


Group,Students,Dropouts,Selected Students,Found Dropouts,False Alarms,Recall (%)
TC,3 936,247,726,114,612,46.2%
TP,111,20,84,18,66,90.0%


## UPV 2021: checking consistency across another academic year

Source: [StudentDropoutDataset](https://zenodo.org/records/17239943). The same student-level checks, enrollment-time features, and 20% review capacity are applied to the 2021 file. This checks the audit's portability across UPV files; it does not establish performance at a new university.

In [ ]:
# UPV 2021: same preparation and audit, with a separate model fitted only to 2021.
upv_2021_url = 'https://zenodo.org/records/17239943/files/dataset_2021_hash.zip?download=1'
upv_2021_response = requests.get(upv_2021_url, timeout=120)
upv_2021_response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(upv_2021_response.content)) as archive:
    upv_2021_raw = pd.read_csv(archive.open('dataset_2021_hash.csv'), sep=';', low_memory=False, usecols=['dni_hash', 'abandono_hash', *upv_features])
upv_2021_conflicts = upv_2021_raw.groupby('dni_hash')[['abandono_hash', *upv_features]].nunique(dropna=False).gt(1).any(axis=1).sum()
if upv_2021_raw['dni_hash'].isna().any() or upv_2021_conflicts:
    raise ValueError(f'UPV 2021 has missing IDs or {upv_2021_conflicts} conflicting student records')
upv_2021_students = upv_2021_raw.drop_duplicates('dni_hash').copy()
if not upv_2021_students['abandono_hash'].isin(['A', 'B']).all():
    raise ValueError('Unexpected UPV 2021 outcome code')
upv_2021_report, upv_2021_model = run_audit(
    upv_2021_students, target_column='abandono_hash', positive_value='A',
    entity_id_column='dni_hash', feature_columns=upv_features,
    group_columns=['dedicacion'], review_share=0.20,
)
upv_2021_review = upv_2021_report['review']
display(pd.DataFrame([{
    'Dataset': 'UPV 2021', 'Record Unit': 'Student',
    'Course Records': len(upv_2021_raw), 'Students': len(upv_2021_students),
    'Dropouts': int(upv_2021_students['abandono_hash'].eq('A').sum()),
    'Validation Records': upv_2021_report['data']['validation_records'],
    'Selected Records': upv_2021_review['selected_records'],
    'Found Dropouts': upv_2021_review['found_positives'],
    'False Alarms': upv_2021_review['false_alarms'],
    'ROC AUC': upv_2021_report['model']['validation_roc_auc'],
    'Precision (%)': upv_2021_review['precision_pct'],
    'Recall (%)': upv_2021_review['recall_pct'],
}]))
display(upv_2021_report['group_audit'][['group_value', 'records', 'total_positives', 'selected_records', 'found_positives', 'false_alarms', 'recall_pct']].rename(columns={
    'group_value':'Group', 'records':'Students', 'total_positives':'Dropouts',
    'selected_records':'Selected Students', 'found_positives':'Found Dropouts',
    'false_alarms':'False Alarms', 'recall_pct':'Recall (%)',
}))

Dataset,Record Unit,Course Records,Students,Dropouts,Validation Records,Selected Records,Found Dropouts,False Alarms,ROC AUC,Precision (%),Recall (%)
UPV 2021,Student,153 120,20 264,1 364,4 053,811,138,673,0.740,17.0%,49.5%


Group,Students,Dropouts,Selected Students,Found Dropouts,False Alarms,Recall (%)
TC,3 927,257,732,121,611,47.1%
TP,126,22,79,17,62,77.3%


## Cross-year check: UPV 2018 to UPV 2021 and 2022

A model can be evaluated on a later year only after excluding students present in its training year. This check fits the same baseline on 2018 students and audits predictions for students absent from the 2018 file in each later year. It tests change over time within UPV; it is not a test on another university. The 20% review capacity is applied separately to each later-year cohort.

In [ ]:
# A forward-year audit with no student in both training and evaluation.
from sklearn.base import clone
upv_2018_ids = set(upv_2018_students['dni_hash'])
upv_forward_model = clone(upv_2018_model)
upv_forward_model.fit(
    upv_2018_students[upv_features],
    upv_2018_students['abandono_hash'].eq('A').astype(int),
)
forward_rows = []
forward_group_rows = []
for year, later in [(2021, upv_2021_students), (2022, upv_students)]:
    new_students = later.loc[~later['dni_hash'].isin(upv_2018_ids)].copy()
    if new_students['dni_hash'].duplicated().any() or new_students.empty:
        raise ValueError('Forward-year evaluation requires unique students absent from 2018')
    new_students['Predicted Dropout Risk'] = upv_forward_model.predict_proba(new_students[upv_features])[:, 1]
    result = audit_predictions(
        new_students, target_column='abandono_hash', positive_value='A',
        score_column='Predicted Dropout Risk', group_columns=['dedicacion'],
        entity_id_column='dni_hash', review_share=0.20,
    )
    review = result['review']
    forward_rows.append({
        'Dataset': f'UPV {year}', 'Students': len(new_students),
        'Excluded 2018 Students': len(later) - len(new_students),
        'Dropouts': int(new_students['abandono_hash'].eq('A').sum()),
        'Selected Students': review['selected_records'],
        'Found Dropouts': review['found_positives'],
        'False Alarms': review['false_alarms'],
        'ROC AUC': result['scores']['roc_auc'],
        'Precision (%)': review['precision_pct'],
        'Recall (%)': review['recall_pct'],
    })
    groups = result['group_audit']
    for _, group in groups.iterrows():
        forward_group_rows.append({
            'Dataset': f'UPV {year}', 'Group': group['group_value'],
            'Students': group['records'], 'Dropouts': group['total_positives'],
            'Selected Students': group['selected_records'],
            'Found Dropouts': group['found_positives'],
            'False Alarms': group['false_alarms'],
            'Recall (%)': group['recall_pct'],
        })
display(pd.DataFrame(forward_rows))
display(pd.DataFrame(forward_group_rows))

Dataset,Students,Excluded 2018 Students,Dropouts,Selected Students,Found Dropouts,False Alarms,ROC AUC,Precision (%),Recall (%)
UPV 2021,12 974,7 290,904,2 595,307,2 288,0.617,11.8%,34.0%
UPV 2022,15 950,4 477,1 096,3 190,397,2 793,0.635,12.4%,36.2%


Dataset,Group,Students,Dropouts,Selected Students,Found Dropouts,False Alarms,Recall (%)
UPV 2021,TC,12 647,832,2 334,247,2 087,29.7%
UPV 2021,TP,327,72,261,60,201,83.3%
UPV 2022,TC,15 563,1 010,2 861,319,2 542,31.6%
UPV 2022,TP,387,86,329,78,251,90.7%


In [ ]:
group_rates = pd.DataFrame(forward_group_rows).copy()
group_rates['Selection Rate (%)'] = 100 * group_rates['Selected Students'] / group_rates['Students']
group_rates['False Positive Rate (%)'] = 100 * group_rates['False Alarms'] / (group_rates['Students'] - group_rates['Dropouts'])
group_rates['Precision (%)'] = 100 * group_rates['Found Dropouts'] / group_rates['Selected Students']
for column in ['Selection Rate (%)', 'False Positive Rate (%)', 'Precision (%)']:
    group_rates[column] = group_rates[column].map(lambda value: f'{value:.1f}%')
display(group_rates[['Dataset', 'Group', 'Selection Rate (%)', 'False Positive Rate (%)', 'Precision (%)', 'Recall (%)']])

Dataset,Group,Selection Rate (%),False Positive Rate (%),Precision (%),Recall (%)
UPV 2021,TC,18.5%,17.7%,10.6%,29.7%
UPV 2021,TP,79.8%,78.8%,23.0%,83.3%
UPV 2022,TC,18.4%,17.5%,11.1%,31.6%
UPV 2022,TP,85.0%,83.4%,23.7%,90.7%


## What the UPV year checks add to the prototype

- **Repeatability within UPV:** The same student-level adapter runs on 2018, 2021, and 2022. This checks consistency across files from one institution; it does not establish transfer to another university.
- **Performance in later years:** A model trained on 2018 students and evaluated on students absent from the 2018 file has ROC AUC 0.617 in 2021 and 0.635 in 2022. At 20% review capacity, recall is 34.0% and 36.2%. The audit should therefore include a forward-year check when dated data are available.
- **Group behavior over time:** In those forward-year checks, 79.8% of TP students in 2021 and 85.0% in 2022 are selected, compared with 18.5% and 18.4% of TC students. TP false positive rates are 78.8% and 83.4%. Report selection rate, false positive rate, precision, recall, and group size for each period.

**Prototype decision:** Add forward-year evaluation and group-level reporting to the audit interface. A later test at a different university should use dated predictions and verified outcomes to assess cross-institution generality.

## Cross-dataset audit contract check

The same audit is applied to held-out predictions for OULAD, UPV 2022, and UCI. Each dataset keeps its documented decision unit and outcome mapping. This checks the audit interface; each prediction model is trained separately on its own dataset.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Fit one baseline model per dataset, then audit the same held-out scores directly.
oulad_report, oulad_model = run_audit(
    data,
    target_column='final_result',
    positive_value='Withdrawn',
    entity_id_column='id_student',
    group_columns=['imd_band'],
    review_share=0.20,
    feature_columns=[c for c in data.columns if c not in {'final_result', 'id_student'}],
)

cases = [
    ('OULAD', 'Course enrollment', data, oulad_report, oulad_model),
    ('UPV 2022', 'Student', upv_students, upv_report, upv_model),
    ('UCI', 'Student', uci_data, uci_report, uci_model),
]
contract_rows = []
for name, unit, frame, trained_report, trained_model in cases:
    config = trained_report['configuration']
    target = config['target_column']
    entity = config['entity_id_column']
    groups = config['group_columns']
    features = config['feature_columns']
    split_groups = frame[entity].to_numpy() if entity else np.arange(len(frame))
    development, _ = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(frame, groups=split_groups))
    _, validation_local = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43).split(frame.iloc[development], groups=split_groups[development]))
    validation = development[validation_local]
    columns = list(dict.fromkeys([target, *groups, *([entity] if entity else [])]))
    scored = frame.iloc[validation][columns].copy()
    scored['Predicted Dropout Risk'] = trained_model.predict_proba(frame[features].iloc[validation])[:, 1]
    direct_report = audit_predictions(
        scored,
        target_column=target,
        positive_value=config['positive_value'],
        score_column='Predicted Dropout Risk',
        entity_id_column=entity,
        group_columns=groups,
        review_share=0.20,
    )
    assert direct_report['review'] == trained_report['review'], name
    pd.testing.assert_frame_equal(direct_report['group_audit'], trained_report['group_audit'], check_dtype=False)
    contract_rows.append({
        'Dataset': name,
        'Record Unit': unit,
        'Source Records': len(frame),
        'Validation Records': len(scored),
        'Selected Records': direct_report['review']['selected_records'],
        'Unique Students Selected': direct_report['review']['unique_entities_to_review'] if entity else direct_report['review']['selected_records'],
        'Outcome Values': ', '.join(map(str, frame[target].value_counts().index.tolist())),
        'Audit Match': 'Yes',
    })

print('Direct prediction audit matches the trained-model audit for all three datasets.')
display(pd.DataFrame(contract_rows))

Direct prediction audit matches the trained-model audit for all three datasets.


Dataset,Record Unit,Source Records,Validation Records,Selected Records,Unique Students Selected,Outcome Values,Audit Match
OULAD,Course enrollment,32 593,6 538,1 308,1 165,"Pass, Withdrawn, Fail, Distinction",Yes
UPV 2022,Student,20 427,4 086,818,818,"B, A",Yes
UCI,Student,4 424,885,177,177,"Graduate, Dropout, Enrolled",Yes


**Result.** The direct prediction audit reproduces the trained-model review and group metrics on all three held-out sets. This confirms that the audit can accept scores produced outside its own training step.

- OULAD: one decision unit is a course enrollment. The 20% review selects 1,308 enrollments belonging to 1,165 distinct students. A student-level review requires a separate aggregation rule.
- UPV 2022: one decision unit is a student after collapsing course rows. The 20% review selects 818 students.
- UCI: one decision unit is a student. The 20% review selects 177 students. The current binary mapping counts `Enrolled` together with `Graduate` as non-dropout; unresolved outcomes need an explicit policy before using this audit operationally.

**Prototype implication.** Keep outcome mapping and decision unit in the input contract. The next useful change is validation that rejects ambiguous outcome values and accidental duplicate decision units.

## Input validation for observed-outcome audits

New submissions use `audit_validated_predictions`: declare the positive, negative, and excluded outcomes, the decision unit, and its unique key. Unknown outcomes, missing IDs, duplicate decisions, and invalid probabilities stop the audit. Exclusions are reported before metrics are calculated; review capacity applies to the remaining audited rows.

OULAD keeps one course enrollment per decision. UPV keeps one student after the earlier adapter. UCI excludes `Enrolled` from this resolved-outcome check. Its source row number is a technical key because the public file has no student ID; it cannot reveal actual duplicate students.

These checks validate the submitted structure. Prediction timing, the meaning of the outcome, and its observation horizon still require verification from the data provider. Earlier exploratory sections remain reproducible; the validated entry below is used for new submissions.

In [ ]:
# Validated entry for new submissions. Uses audit_predictions from the earlier cell.
from numbers import Real
import numpy as np
import pandas as pd


def _column_list(value, name, allow_empty=False):
    if isinstance(value, str):
        value = [value]
    if value is None:
        value = []
    if not isinstance(value, (list, tuple)):
        raise ValueError(f"{name} must be a column name or a list of column names")
    columns = list(value)
    if (not columns and not allow_empty) or any(
        not isinstance(c, str) or not c.strip() for c in columns
    ):
        raise ValueError(f"{name} must contain non-empty column names")
    if len(columns) != len(set(columns)):
        raise ValueError(f"{name} contains duplicate column names")
    return columns


def _outcome_values(value, name, allow_empty=False):
    if not isinstance(value, (list, tuple)):
        raise ValueError(f"{name} must be an explicit list of outcome values")
    values = list(value)
    if not values and not allow_empty:
        raise ValueError(f"{name} must not be empty")
    for label in values:
        if not pd.api.types.is_scalar(label) or pd.isna(label):
            raise ValueError(f"{name} must contain non-missing scalar values")
    if len(pd.Index(values).unique()) != len(values):
        raise ValueError(f"{name} contains duplicate values")
    return values


def validate_audit_input(
    dataset, *, target_column, positive_value, negative_values,
    score_column, decision_unit, decision_id_columns,
    group_columns=None, entity_id_column=None, excluded_values=(),
    review_share=0.20,
):
    """Validate all submitted rows, then explicitly exclude declared outcomes.

    The key must identify one decision within the submitted batch. For student
    decisions with an entity ID, that ID must itself be unique. This function
    cannot verify that supplied IDs or outcome meanings are factually correct.
    Missing outcomes are rejected; this is the observed-outcome audit mode.
    """
    if not isinstance(dataset, pd.DataFrame) or dataset.empty:
        raise ValueError("dataset must be a non-empty pandas DataFrame")
    if not dataset.columns.is_unique:
        raise ValueError("Dataset contains duplicate column names")
    if isinstance(review_share, (bool, np.bool_)) or not isinstance(review_share, Real) or not np.isfinite(review_share) or not 0 < review_share <= 1:
        raise ValueError("review_share must be a finite number greater than 0 and at most 1")
    if decision_unit not in ("student", "course_enrollment", "student_period", "record"):
        raise ValueError("decision_unit must be student, course_enrollment, student_period, or record")
    if not pd.api.types.is_scalar(positive_value) or pd.isna(positive_value):
        raise ValueError("positive_value must be a non-missing scalar")
    negative_values = _outcome_values(negative_values, "negative_values")
    excluded_values = _outcome_values(excluded_values, "excluded_values", allow_empty=True)
    all_labels = [positive_value, *negative_values, *excluded_values]
    if len(pd.Index(all_labels).unique()) != len(all_labels):
        raise ValueError("Positive, negative, and excluded outcomes must not overlap")
    keys = _column_list(decision_id_columns, "decision_id_columns")
    groups = _column_list(group_columns, "group_columns", allow_empty=True)
    _column_list([target_column, score_column], "target_column and score_column")
    if entity_id_column is not None:
        _column_list(entity_id_column, "entity_id_column")
        if not isinstance(entity_id_column, str):
            raise ValueError("entity_id_column must be one column name")
    if target_column in groups or score_column in groups:
        raise ValueError("Outcome and score columns must be separate from group columns")
    if {target_column, score_column}.intersection(keys):
        raise ValueError("Decision keys must be separate from outcome and score columns")
    if entity_id_column in {target_column, score_column}:
        raise ValueError("Entity ID must be separate from outcome and score columns")
    required = list(dict.fromkeys([
        target_column, score_column, *keys, *groups,
        *([entity_id_column] if entity_id_column is not None else []),
    ]))
    missing = [column for column in required if column not in dataset.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if dataset[target_column].isna().any():
        raise ValueError("Target column contains missing outcomes; resolve them before auditing")
    unknown = ~dataset[target_column].isin(all_labels)
    if unknown.any():
        raise ValueError(f"{int(unknown.sum())} rows contain unmapped outcomes; declare them as positive, negative, or excluded")
    id_columns = list(dict.fromkeys(keys + ([entity_id_column] if entity_id_column else [])))
    for column in id_columns:
        values = dataset[column]
        blank = values.map(lambda v: isinstance(v, str) and not v.strip())
        if values.isna().any() or blank.any():
            raise ValueError(f"ID column contains missing or blank values: {column}")
    if dataset.duplicated(keys).any():
        raise ValueError("Duplicate decision keys; supply one row per decision unit")
    if decision_unit == "student" and entity_id_column is not None and dataset[entity_id_column].duplicated().any():
        raise ValueError("Student audit contains repeated entity IDs; aggregate or correct the records explicitly")
    if dataset[score_column].map(lambda v: isinstance(v, (bool, np.bool_))).any():
        raise ValueError("Scores must be probabilities, not boolean predictions")
    try:
        scores = pd.to_numeric(dataset[score_column], errors="raise").to_numpy(dtype=float)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("Score column must contain numeric probabilities") from exc
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Score column must contain finite probabilities between 0 and 1")
    excluded = dataset[target_column].isin(excluded_values)
    prepared = dataset.loc[~excluded].copy()
    prepared[score_column] = scores[~excluded.to_numpy()]
    if prepared[target_column].eq(positive_value).nunique() != 2:
        raise ValueError("Audited rows must contain both outcome classes after exclusions")
    validation = {
        "input_records": len(dataset),
        "audited_records": len(prepared),
        "excluded_records": int(excluded.sum()),
        "excluded_outcome_counts": dataset.loc[excluded, target_column].value_counts().to_dict(),
        "missing_group_records": {c: int(prepared[c].isna().sum()) for c in groups},
        "selection_population": "Audited rows after explicit outcome exclusions",
        "decision_unit": decision_unit,
        "decision_id_columns": keys,
        "identity_check": "Supplied decision keys are unique; real-world identity is not verified",
    }
    return prepared, validation


def audit_validated_predictions(
    dataset, *, target_column, positive_value, negative_values,
    score_column, decision_unit, decision_id_columns,
    group_columns=None, entity_id_column=None, excluded_values=(),
    review_share=0.20,
):
    """Audit existing probabilities after checking explicit input rules.

    Excluded outcomes do not enter ranking, review capacity, or error metrics.
    Scores must estimate the positive outcome, using information available at
    prediction time. Validation cannot establish their provenance or horizon.
    """
    prepared, validation = validate_audit_input(
        dataset, target_column=target_column, positive_value=positive_value,
        negative_values=negative_values, score_column=score_column,
        decision_unit=decision_unit, decision_id_columns=decision_id_columns,
        group_columns=group_columns, entity_id_column=entity_id_column,
        excluded_values=excluded_values, review_share=review_share,
    )
    report = audit_predictions(
        prepared, target_column=target_column, positive_value=positive_value,
        score_column=score_column, group_columns=group_columns,
        entity_id_column=entity_id_column, review_share=review_share,
    )
    report["configuration"].update({
        "negative_values": list(negative_values),
        "excluded_values": list(excluded_values),
        "selection_unit": decision_unit,
        "decision_id_columns": list(validation["decision_id_columns"]),
    })
    report["input_validation"] = validation
    return report


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Reuse existing held-out scores; no model is fitted in this cell.
input_rules = {
    'OULAD': dict(negative_values=['Pass', 'Fail', 'Distinction'], excluded_values=[],
                  decision_unit='course_enrollment',
                  decision_id_columns=['id_student', 'code_module', 'code_presentation']),
    'UPV 2022': dict(negative_values=['B'], excluded_values=[],
                    decision_unit='student', decision_id_columns=['dni_hash']),
    'UCI': dict(negative_values=['Graduate'], excluded_values=['Enrolled'],
                decision_unit='student', decision_id_columns=['Source Row ID']),
}
validated_predictions = {}
validated_reports = {}
input_check_rows = []
for name, unit, frame, trained_report, trained_model in cases:
    config = trained_report['configuration']
    rules = input_rules[name]
    entity = config['entity_id_column']
    split_groups = frame[entity].to_numpy() if entity else np.arange(len(frame))
    development, _ = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(frame, groups=split_groups))
    _, validation_local = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=43).split(frame.iloc[development], groups=split_groups[development]))
    validation = development[validation_local]
    columns = list(dict.fromkeys([
        config['target_column'], *config['group_columns'],
        *([entity] if entity else []),
        *[c for c in rules['decision_id_columns'] if c in frame.columns],
    ]))
    scored = frame.iloc[validation][columns].copy()
    if name == 'UCI':
        # This technical key checks repeated file rows, not real student identity.
        scored['Source Row ID'] = validation + 1
    scored['Predicted Dropout Risk'] = trained_model.predict_proba(frame[config['feature_columns']].iloc[validation])[:, 1]
    checked = audit_validated_predictions(
        scored, target_column=config['target_column'],
        positive_value=config['positive_value'], score_column='Predicted Dropout Risk',
        group_columns=config['group_columns'], entity_id_column=entity,
        review_share=0.20, **rules,
    )
    if name != 'UCI':
        assert checked['review'] == trained_report['review'], name
        pd.testing.assert_frame_equal(checked['group_audit'], trained_report['group_audit'])
    validated_predictions[name] = scored
    validated_reports[name] = checked
    checks = checked['input_validation']
    review = checked['review']
    input_check_rows.append({
        'Dataset': name,
        'Decision Unit': rules['decision_unit'].replace('_', ' ').title(),
        'Input Records': checks['input_records'],
        'Excluded Records': checks['excluded_records'],
        'Audited Records': checks['audited_records'],
        'Selected Records': review['selected_records'],
        'Found Dropouts': review['found_positives'],
        'False Alarms': review['false_alarms'],
        'Precision (%)': review['precision_pct'],
        'Recall (%)': review['recall_pct'],
        'Input Check': 'Passed',
    })

input_check_table = pd.DataFrame(input_check_rows)
count_columns = ['Input Records', 'Excluded Records', 'Audited Records', 'Selected Records', 'Found Dropouts', 'False Alarms']
formats = {c: lambda v: f'{v:,.0f}'.replace(',', ' ') for c in count_columns}
formats.update({'Precision (%)': '{:.1f}%', 'Recall (%)': '{:.1f}%'})
display(input_check_table.style.hide(axis='index').format(formats))
print('OULAD and UPV review and group metrics match the earlier runs.')
print('UCI uses the same existing scores, with Enrolled excluded only from this audit. Metrics apply to the resolved subset; the model was not retrained.')
print('UCI source-row keys cannot verify actual student identity or duplicate students.')


Dataset,Decision Unit,Input Records,Excluded Records,Audited Records,Selected Records,Found Dropouts,False Alarms,Precision (%),Recall (%),Input Check
OULAD,Course Enrollment,6 538,0,6 538,1 308,642,666,49.1%,31.9%,Passed
UPV 2022,Student,4 086,0,4 086,818,133,685,16.3%,45.5%,Passed
UCI,Student,885,168,717,144,117,27,81.2%,42.7%,Passed


OULAD and UPV review and group metrics match the earlier runs.
UCI uses the same existing scores, with Enrolled excluded only from this audit. Metrics apply to the resolved subset; the model was not retrained.
UCI source-row keys cannot verify actual student identity or duplicate students.


In [ ]:
# Controlled input failures: no edits are made to the original datasets.
case_configs = {name: report['configuration'] for name, _, _, report, _ in cases}
rejection_rows = []

def expect_rejection(name, scenario, frame, expected_message, **changes):
    config = case_configs[name]
    arguments = dict(
        target_column=config['target_column'], positive_value=config['positive_value'],
        score_column='Predicted Dropout Risk', entity_id_column=config['entity_id_column'],
        group_columns=config['group_columns'], review_share=0.20, **input_rules[name],
    )
    arguments.update(changes)
    try:
        audit_validated_predictions(frame, **arguments)
    except ValueError as error:
        assert expected_message in str(error), str(error)
        rejection_rows.append({'Dataset': name, 'Scenario': scenario, 'Input Check': 'Rejected', 'Reason': str(error)})
    else:
        raise AssertionError(f'{name}: {scenario} was incorrectly accepted')

expect_rejection('UCI', 'Undeclared Enrolled Outcome', validated_predictions['UCI'], 'unmapped outcomes', excluded_values=[])
expect_rejection('UPV 2022', 'Duplicate Student Record',
                 pd.concat([validated_predictions['UPV 2022'], validated_predictions['UPV 2022'].iloc[[0]]]),
                 'Duplicate decision keys')
expect_rejection('OULAD', 'Enrollments Declared As Students', validated_predictions['OULAD'],
                 'repeated entity IDs', decision_unit='student')
bad_score = validated_predictions['UPV 2022'].copy()
bad_score.iloc[0, bad_score.columns.get_loc('Predicted Dropout Risk')] = 1.5
expect_rejection('UPV 2022', 'Probability Above One', bad_score, 'finite probabilities')
bad_outcome = validated_predictions['UCI'].copy()
bad_outcome.iloc[0, bad_outcome.columns.get_loc('Target')] = None
expect_rejection('UCI', 'Missing Outcome', bad_outcome, 'missing outcomes')
expect_rejection('UCI', 'Overlapping Outcome Rules', validated_predictions['UCI'],
                 'must not overlap', negative_values=['Graduate', 'Dropout'])
display(pd.DataFrame(rejection_rows).style.hide(axis='index'))

# A hand-calculated example also checks exclusion and ranking together.
toy = pd.DataFrame({
    'ID': [1, 2, 3, 4, 5],
    'Outcome': ['Dropout', 'Graduate', 'Dropout', 'Graduate', 'Enrolled'],
    'Risk': [0.9, 0.8, 0.3, 0.1, 0.99],
})
toy_report = audit_validated_predictions(
    toy, target_column='Outcome', positive_value='Dropout', negative_values=['Graduate'],
    excluded_values=['Enrolled'], score_column='Risk', decision_unit='student',
    decision_id_columns=['ID'], entity_id_column='ID', review_share=0.50,
)
assert toy_report['review']['records'] == 4
assert toy_report['review']['selected_records'] == 2
assert toy_report['review']['found_positives'] == 1
assert toy_report['review']['false_alarms'] == 1
assert toy_report['review']['recall_pct'] == 50
assert toy_report['review']['precision_pct'] == 50
print('Hand-calculated example passed: one excluded record, two selected records, one found dropout, one false alarm.')

missing_group_rows = [
    {'Dataset': name, 'Group Feature': {'imd_band': 'IMD Band', 'estudios_m_hash': "Mother's Education"}.get(column, column), 'Missing Records': count}
    for name, report in validated_reports.items()
    for column, count in report['input_validation']['missing_group_records'].items()
    if count > 0
]
if missing_group_rows:
    print('Missing group values remain visible as Unknown in the group audit:')
    display(pd.DataFrame(missing_group_rows).style.hide(axis='index').format({'Missing Records': lambda v: f'{v:,.0f}'.replace(',', ' ')}))


Dataset,Scenario,Input Check,Reason
UCI,Undeclared Enrolled Outcome,Rejected,"168 rows contain unmapped outcomes; declare them as positive, negative, or excluded"
UPV 2022,Duplicate Student Record,Rejected,Duplicate decision keys; supply one row per decision unit
OULAD,Enrollments Declared As Students,Rejected,Student audit contains repeated entity IDs; aggregate or correct the records explicitly
UPV 2022,Probability Above One,Rejected,Score column must contain finite probabilities between 0 and 1
UCI,Missing Outcome,Rejected,Target column contains missing outcomes; resolve them before auditing
UCI,Overlapping Outcome Rules,Rejected,"Positive, negative, and excluded outcomes must not overlap"


Hand-calculated example passed: one excluded record, two selected records, one found dropout, one false alarm.
Missing group values remain visible as Unknown in the group audit:


Dataset,Group Feature,Missing Records
OULAD,IMD Band,209
UPV 2022,Mother's Education,40


### Input validation result

OULAD and UPV pass with unchanged metrics. UCI excludes 168 Enrolled records: 717 students remain and 144 are selected at 20%. These metrics apply to the resolved subset using existing scores.

All six controlled failures are rejected; the hand-calculated check passes. Missing group values stay visible as Unknown.

Use audit_validated_predictions for new submissions. Passing this check confirms the declared structure; prediction timing, outcome meaning, and real student identity still require source verification.

## Uncertainty in audit metrics

This section adds individual 95% confidence intervals for recall and false-positive rate, showing the relevant outcome counts and contributing students. The current scores and selection flags stay fixed. The intervals describe evaluation-sample uncertainty under independent students; model refitting, re-ranking, and future-year changes are outside this calculation.

One record per student uses Wilson intervals. OULAD uses 2,000 bootstrap samples of whole students, keeping their course records together. Group differences use Newcombe intervals for independent groups or shared student bootstrap draws for repeated records. References are declared explicitly: TC for UPV and Daytime for UCI. UCI continues to exclude Enrolled and assumes one distinct student per source row.

Limited outcome support is flagged. Missing denominators and unsuitable bootstrap cases produce unavailable intervals. These are individual exploratory intervals without adjustment for multiple comparisons; they do not establish causation or discrimination. An interval containing zero does not establish equal performance.

Methods: [NIST Wilson intervals](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm), [Newcombe intervals](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.confint_proportions_2indep.html), [student cluster resampling](https://cameron.econ.ucdavis.edu/research/Cameron_Miller_JHR_2015.pdf).

In [ ]:
"""Conditional intervals for existing, fixed audit selections.

Wilson intervals: https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm
Newcombe differences: statsmodels.stats.proportion.confint_proportions_2indep
Cluster resampling: Cameron and Miller (2015), A Practitioner's Guide to
Cluster-Robust Inference, https://cameron.econ.ucdavis.edu/research/Cameron_Miller_JHR_2015.pdf
"""

from numbers import Integral, Real
from statistics import NormalDist

import numpy as np
import pandas as pd


def _wilson_interval(successes, trials, confidence_level):
    if trials == 0:
        return np.nan, np.nan
    z = NormalDist().inv_cdf((1 + confidence_level) / 2)
    p = successes / trials
    denominator = 1 + z * z / trials
    center = (p + z * z / (2 * trials)) / denominator
    radius = z * np.sqrt(p * (1 - p) / trials + z * z / (4 * trials * trials)) / denominator
    return max(0.0, center - radius), min(1.0, center + radius)


def _newcombe_interval(row, reference, confidence_level):
    p = row["numerator"] / row["denominator_records"]
    q = reference["numerator"] / reference["denominator_records"]
    lp, up = _wilson_interval(row["numerator"], row["denominator_records"], confidence_level)
    lq, uq = _wilson_interval(reference["numerator"], reference["denominator_records"], confidence_level)
    return p - q - np.hypot(p - lp, uq - q), p - q + np.hypot(q - lq, up - p)


def _percentile_interval(draws, confidence_level):
    finite = draws[np.isfinite(draws)]
    if len(finite) < 0.95 * len(draws):
        return np.nan, np.nan, "Too Many Undefined Resamples"
    if np.ptp(finite) < 1e-12:
        return np.nan, np.nan, "Degenerate Bootstrap"
    tail = (1 - confidence_level) / 2
    low, high = np.quantile(finite, [tail, 1 - tail])
    return float(low), float(high), "Available"


def calculate_audit_uncertainty(
    prepared, *, target_column, positive_value, score_column, decision_unit,
    entity_id_column=None, group_columns=None, review_share=0.20,
    confidence_level=0.95, n_resamples=2000, random_state=42,
    group_references=None,
):
    """Consume already validated, outcome-filtered rows.

    Scores and selected/not-selected flags stay fixed. Intervals quantify
    evaluation-sample uncertainty under independent students, allowing dependence
    within each student. They do not include refitting, re-ranking, temporal
    drift, confounding, or simultaneous coverage across multiple comparisons.
    """
    if isinstance(confidence_level, (bool, np.bool_)) or not isinstance(confidence_level, Real) or not np.isfinite(confidence_level) or not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be a finite number between 0 and 1")
    if isinstance(n_resamples, (bool, np.bool_)) or not isinstance(n_resamples, Integral) or n_resamples < 200:
        raise ValueError("n_resamples must be an integer of at least 200")
    if isinstance(random_state, (bool, np.bool_)) or not isinstance(random_state, Integral) or random_state < 0:
        raise ValueError("random_state must be a non-negative integer")
    groups = [group_columns] if isinstance(group_columns, str) else list(group_columns or [])
    references = {} if group_references is None else group_references
    if not isinstance(references, dict) or any(column not in groups for column in references):
        raise ValueError("group_references must map audited group columns to reference values")
    actual = prepared[target_column].eq(positive_value).to_numpy()
    scores = prepared[score_column].to_numpy(dtype=float)
    selected = np.zeros(len(prepared), dtype=bool)
    selected[np.argsort(-scores, kind="stable")[:int(np.ceil(review_share * len(prepared)))]] = True
    codes = pd.factorize(prepared[entity_id_column], sort=False)[0] if entity_id_column else np.arange(len(prepared))
    unit_count = int(codes.max()) + 1
    repeated = unit_count < len(prepared)
    missing_cluster_id = entity_id_column is None and decision_unit != "student"
    method = "Unavailable" if missing_cluster_id else "Student Cluster Bootstrap" if repeated else "Wilson"
    masks = [(None, "All", np.ones(len(prepared), dtype=bool))]
    for column in groups:
        values = prepared[column].astype("object").fillna("Unknown").astype(str).to_numpy()
        if column in references and str(references[column]) not in values:
            raise ValueError(f"Reference group not found in audited rows: {column}")
        masks.extend((column, value, values == value) for value in np.unique(values))
    rows, denominators, numerators = [], [], []
    for column, value, mask in masks:
        for metric, outcomes in [("recall_pct", actual), ("false_positive_rate_pct", ~actual)]:
            denominator = mask & outcomes
            numerator = denominator & selected
            n, k = int(denominator.sum()), int(numerator.sum())
            n_units = len(np.unique(codes[denominator]))
            selected_units = len(np.unique(codes[numerator]))
            unselected_units = len(np.unique(codes[denominator & ~selected]))
            support = "Limited Outcome Support" if n_units < 30 else (
                "Sparse Selected or Unselected Outcomes"
                if min(selected_units, unselected_units) < 10 else ""
            )
            rows.append({
                "group_column": column, "group_value": value, "metric": metric,
                "records": int(mask.sum()), "independent_units": len(np.unique(codes[mask])),
                "numerator": k, "denominator_records": n, "denominator_units": n_units,
                "selected_outcome_units": selected_units, "unselected_outcome_units": unselected_units,
                "estimate_pct": 100 * k / n if n else np.nan,
                "ci_lower_pct": np.nan, "ci_upper_pct": np.nan,
                "ci_width_pp": np.nan, "method": method,
                "ci_status": "No Relevant Outcomes" if n == 0 else "Available",
                "support_note": support,
            })
            denominators.append(denominator)
            numerators.append(numerator)
    bootstrap = None
    if repeated and not missing_cluster_id:
        # Each draw gives one multiplicity to a whole student, across all groups.
        counts = np.zeros((unit_count, 2 * len(rows)), dtype=float)
        for j, (numerator, denominator) in enumerate(zip(numerators, denominators)):
            counts[:, 2 * j] = np.bincount(codes, weights=numerator, minlength=unit_count)
            counts[:, 2 * j + 1] = np.bincount(codes, weights=denominator, minlength=unit_count)
        rng = np.random.default_rng(random_state)
        bootstrap = np.full((n_resamples, len(rows)), np.nan)
        probabilities = np.full(unit_count, 1 / unit_count)
        for start in range(0, n_resamples, 100):
            stop = min(start + 100, n_resamples)
            weights = rng.multinomial(unit_count, probabilities, size=stop - start)
            totals = weights.astype(float) @ counts
            np.divide(totals[:, ::2], totals[:, 1::2],
                      out=bootstrap[start:stop], where=totals[:, 1::2] > 0)
    for j, row in enumerate(rows):
        n, k = row["denominator_records"], row["numerator"]
        if n == 0:
            continue
        if missing_cluster_id:
            row["ci_status"] = "Missing Entity ID"
            continue
        if not repeated:
            low, high = _wilson_interval(k, n, confidence_level)
        elif row["denominator_units"] < 10:
            row["ci_status"] = "Too Few Independent Units"
            continue
        elif k in (0, n):
            row["ci_status"] = "Boundary Estimate: Bootstrap Unavailable"
            continue
        elif min(row["selected_outcome_units"], row["unselected_outcome_units"]) < 5:
            row["ci_status"] = "Too Few Outcome Event Units"
            continue
        else:
            low, high, row["ci_status"] = _percentile_interval(bootstrap[:, j], confidence_level)
        row["ci_lower_pct"], row["ci_upper_pct"] = 100 * low, 100 * high
        row["ci_width_pp"] = 100 * (high - low)
    comparisons = []
    lookup = {(r["group_column"], r["group_value"], r["metric"]): j for j, r in enumerate(rows)}
    for column, reference_value in references.items():
        for j, row in enumerate(rows):
            if row["group_column"] != column or row["group_value"] == str(reference_value):
                continue
            ref_index = lookup[column, str(reference_value), row["metric"]]
            reference = rows[ref_index]
            low = high = np.nan
            status = "Available"
            if row["ci_status"] != "Available" or reference["ci_status"] != "Available":
                status = "Component Interval Unavailable"
            elif repeated:
                low, high, status = _percentile_interval(bootstrap[:, j] - bootstrap[:, ref_index], confidence_level)
            else:
                low, high = _newcombe_interval(row, reference, confidence_level)
            comparisons.append({
                "group_column": column, "group_value": row["group_value"],
                "reference_group": str(reference_value), "metric": row["metric"],
                "difference_pp": row["estimate_pct"] - reference["estimate_pct"],
                "ci_lower_pp": 100 * low, "ci_upper_pp": 100 * high,
                "method": method if repeated or missing_cluster_id else "Newcombe",
                "ci_status": status,
                "contains_zero": bool(low <= 0 <= high) if np.isfinite(low) else None,
                "support_note": "; ".join(dict.fromkeys(note for note in [row["support_note"], reference["support_note"]] if note)),
            })
    table = pd.DataFrame(rows)
    comparison_columns = ["group_column", "group_value", "reference_group", "metric", "difference_pp", "ci_lower_pp", "ci_upper_pp", "method", "ci_status", "contains_zero", "support_note"]
    return {
        "review_uncertainty": table.loc[table["group_column"].isna()].reset_index(drop=True),
        "group_uncertainty": table.loc[table["group_column"].notna()].reset_index(drop=True),
        "group_comparisons": pd.DataFrame(comparisons, columns=comparison_columns),
        "uncertainty_configuration": {
            "confidence_level": confidence_level, "method": method,
            "n_resamples": n_resamples if repeated and not missing_cluster_id else 0,
            "random_state": random_state, "entity_id_column": entity_id_column,
            "independent_units": unit_count if not missing_cluster_id else None,
            "group_references": dict(references),
            "selection_is_fixed": True, "model_is_fixed": True,
            "simultaneous_intervals": False,
            "support_warning_below_units": 30,
            "minimum_bootstrap_denominator_units": 10,
            "minimum_bootstrap_event_units_per_side": 5,
            "sparse_events_warning_below_units_per_side": 10,
            "minimum_valid_resample_share": 0.95,
            "identity_assumption": "Independent students; dependence allowed within supplied entity IDs",
        },
    }


def audit_validated_predictions(
    dataset, *, target_column, positive_value, negative_values,
    score_column, decision_unit, decision_id_columns,
    group_columns=None, entity_id_column=None, excluded_values=(),
    review_share=0.20,
    confidence_level=0.95, n_resamples=2000, random_state=42,
    group_references=None,
):
    """Audit existing probabilities after checking explicit input rules.

    Excluded outcomes do not enter ranking, review capacity, or error metrics.
    Scores must estimate the positive outcome, using information available at
    prediction time. Validation cannot establish their provenance or horizon.
    """
    prepared, validation = validate_audit_input(
        dataset, target_column=target_column, positive_value=positive_value,
        negative_values=negative_values, score_column=score_column,
        decision_unit=decision_unit, decision_id_columns=decision_id_columns,
        group_columns=group_columns, entity_id_column=entity_id_column,
        excluded_values=excluded_values, review_share=review_share,
    )
    report = audit_predictions(
        prepared, target_column=target_column, positive_value=positive_value,
        score_column=score_column, group_columns=group_columns,
        entity_id_column=entity_id_column, review_share=review_share,
    )
    report["configuration"].update({
        "negative_values": list(negative_values),
        "excluded_values": list(excluded_values),
        "selection_unit": decision_unit,
        "decision_id_columns": list(validation["decision_id_columns"]),
    })
    report["input_validation"] = validation
    report.update(calculate_audit_uncertainty(
        prepared, target_column=target_column, positive_value=positive_value,
        score_column=score_column, decision_unit=decision_unit,
        group_columns=group_columns, entity_id_column=entity_id_column,
        review_share=review_share, confidence_level=confidence_level,
        n_resamples=n_resamples, random_state=random_state,
        group_references=group_references,
    ))
    return report


In [ ]:
# The same existing predictions, outcome exclusions, and 20% selection policy.
uncertainty_reports = {}
for name, scored in validated_predictions.items():
    config = case_configs[name]
    references = {'dedicacion': 'TC'} if name == 'UPV 2022' else ({config['group_columns'][0]: 'Daytime'} if name == 'UCI' else {})
    report = audit_validated_predictions(
        scored, target_column=config['target_column'], positive_value=config['positive_value'],
        score_column='Predicted Dropout Risk', group_columns=config['group_columns'],
        entity_id_column=config['entity_id_column'], review_share=0.20,
        confidence_level=0.95, n_resamples=2000, random_state=42,
        group_references=references, **input_rules[name],
    )
    assert report['review'] == validated_reports[name]['review'], name
    pd.testing.assert_frame_equal(report['group_audit'], validated_reports[name]['group_audit'])
    for _, interval_row in report['group_uncertainty'].iterrows():
        point = report['group_audit'].loc[
            (report['group_audit'].group_column == interval_row.group_column)
            & (report['group_audit'].group_value == interval_row.group_value)
        ].iloc[0]
        positive = interval_row.metric == 'recall_pct'
        assert interval_row.numerator == point['found_positives' if positive else 'false_alarms']
        assert interval_row.denominator_records == point['total_positives' if positive else 'total_negatives']
    uncertainty_reports[name] = report

metric_labels = {'recall_pct': 'Recall', 'false_positive_rate_pct': 'False-Positive Rate'}
group_feature_labels = {'imd_band': 'IMD Band', 'dedicacion': 'Study Mode'}
def interval_text(row, difference=False):
    if row['ci_status'] != 'Available':
        return 'Unavailable'
    low = row['ci_lower_pp' if difference else 'ci_lower_pct']
    high = row['ci_upper_pp' if difference else 'ci_upper_pct']
    return f'{low:+.1f} to {high:+.1f}' if difference else f'{low:.1f}% to {high:.1f}%'

summary_rows, focus_rows, comparison_rows, oulad_rows = [], [], [], []
for name, report in uncertainty_reports.items():
    for _, row in report['review_uncertainty'].iterrows():
        summary_rows.append({
            'Dataset': name, 'Metric': metric_labels[row.metric],
            'Estimate': row.estimate_pct, '95% CI': interval_text(row),
            'Relevant Outcomes': row.denominator_records,
            'Contributing Students': row.denominator_units, 'Method': row.method,
        })
    for _, row in report['group_uncertainty'].iterrows():
        item = {
            'Dataset': name, 'Group': ('10-20%' if name == 'OULAD' and row.group_value == '10-20' else row.group_value), 'Metric': metric_labels[row.metric],
            'Estimate': row.estimate_pct, '95% CI': interval_text(row),
            'Relevant Outcomes': row.denominator_records,
            'Contributing Students': row.denominator_units,
            'Interval Status': row.ci_status, 'Support Note': row.support_note or '—',
        }
        if (name == 'UPV 2022' and row.group_column == 'dedicacion') or name == 'UCI':
            focus_rows.append(item)
        if name == 'OULAD':
            oulad_rows.append(item)
    for _, row in report['group_comparisons'].iterrows():
        comparison_rows.append({
            'Dataset': name, 'Comparison': f'{row.group_value} minus {row.reference_group}',
            'Metric': metric_labels[row.metric], 'Difference (pp)': row.difference_pp,
            '95% CI (pp)': interval_text(row, difference=True),
            'Includes Zero': ('Yes' if row.contains_zero else 'No') if row.ci_status == 'Available' else 'Unavailable',
            'Method': row.method, 'Support Note': row.support_note or '—',
        })

def show_intervals(rows):
    formats = {'Estimate': '{:.1f}%', 'Relevant Outcomes': lambda v: f'{v:,.0f}'.replace(',', ' '),
               'Contributing Students': lambda v: f'{v:,.0f}'.replace(',', ' ')}
    display(pd.DataFrame(rows).style.hide(axis='index').format(formats, na_rep='Unavailable'))

print('Overall metrics with individual 95% confidence intervals:')
show_intervals(summary_rows)
print('UPV study mode and UCI attendance groups:')
show_intervals(focus_rows)
print('Signed differences against declared reference groups:')
display(pd.DataFrame(comparison_rows).style.hide(axis='index').format({'Difference (pp)': '{:+.1f}'}, na_rep='Unavailable'))
print('OULAD IMD groups: intervals resample whole students, retaining all of their course records.')
show_intervals(oulad_rows)
print('Existing review and group metrics are unchanged for all three datasets.')
print('Individual intervals are exploratory. Includes Zero does not prove equality; no simultaneous coverage or causal claim is made.')


Overall metrics with individual 95% confidence intervals:


Dataset,Metric,Estimate,95% CI,Relevant Outcomes,Contributing Students,Method
OULAD,Recall,31.9%,29.9% to 34.1%,2 012,1 831,Student Cluster Bootstrap
OULAD,False-Positive Rate,14.7%,13.7% to 15.8%,4 526,4 249,Student Cluster Bootstrap
UPV 2022,Recall,45.5%,39.9% to 51.3%,292,292,Wilson
UPV 2022,False-Positive Rate,18.1%,16.9% to 19.3%,3 794,3 794,Wilson
UCI,Recall,42.7%,37.0% to 48.6%,274,274,Wilson
UCI,False-Positive Rate,6.1%,4.2% to 8.7%,443,443,Wilson


UPV study mode and UCI attendance groups:


Dataset,Group,Metric,Estimate,95% CI,Relevant Outcomes,Contributing Students,Interval Status,Support Note
UPV 2022,TC,Recall,42.8%,37.1% to 48.8%,271,271,Available,—
UPV 2022,TC,False-Positive Rate,16.7%,15.6% to 18.0%,3 706,3 706,Available,—
UPV 2022,TP,Recall,81.0%,60.0% to 92.3%,21,21,Available,Limited Outcome Support
UPV 2022,TP,False-Positive Rate,73.9%,63.8% to 81.9%,88,88,Available,—
UCI,Daytime,Recall,42.7%,36.5% to 49.1%,232,232,Available,—
UCI,Daytime,False-Positive Rate,5.6%,3.8% to 8.3%,411,411,Available,—
UCI,Evening,Recall,42.9%,29.1% to 57.8%,42,42,Available,—
UCI,Evening,False-Positive Rate,12.5%,5.0% to 28.1%,32,32,Available,Sparse Selected or Unselected Outcomes


Signed differences against declared reference groups:


Dataset,Comparison,Metric,Difference (pp),95% CI (pp),Includes Zero,Method,Support Note
UPV 2022,TP minus TC,Recall,+38.1,+16.4 to +50.9,No,Newcombe,Limited Outcome Support
UPV 2022,TP minus TC,False-Positive Rate,+57.1,+47.0 to +65.3,No,Newcombe,—
UCI,Evening minus Daytime,Recall,+0.2,-15.0 to +16.4,Yes,Newcombe,—
UCI,Evening minus Daytime,False-Positive Rate,+6.9,-1.1 to +22.6,Yes,Newcombe,Sparse Selected or Unselected Outcomes


OULAD IMD groups: intervals resample whole students, retaining all of their course records.


Dataset,Group,Metric,Estimate,95% CI,Relevant Outcomes,Contributing Students,Interval Status,Support Note
OULAD,0-10%,Recall,46.2%,39.7% to 52.7%,236,219,Available,—
OULAD,0-10%,False-Positive Rate,26.1%,21.8% to 30.4%,417,395,Available,—
OULAD,10-20%,Recall,45.2%,38.5% to 51.9%,221,201,Available,—
OULAD,10-20%,False-Positive Rate,27.6%,23.5% to 31.9%,463,437,Available,—
OULAD,20-30%,Recall,41.8%,35.6% to 48.5%,244,220,Available,—
OULAD,20-30%,False-Positive Rate,25.5%,21.5% to 29.5%,458,433,Available,—
OULAD,30-40%,Recall,28.2%,22.0% to 34.8%,220,201,Available,—
OULAD,30-40%,False-Positive Rate,14.3%,11.2% to 17.6%,495,467,Available,—
OULAD,40-50%,Recall,34.5%,28.1% to 40.9%,229,206,Available,—
OULAD,40-50%,False-Positive Rate,10.6%,7.7% to 13.4%,445,423,Available,—


Existing review and group metrics are unchanged for all three datasets.
Individual intervals are exploratory. Includes Zero does not prove equality; no simultaneous coverage or causal claim is made.


### What the intervals add

- UPV TP recall is 81.0%, with a 95% interval of 60.0% to 92.3%, based on 21 dropout cases. TP minus TC recall is +38.1 pp (16.4 to 50.9 pp); the false-positive-rate difference is +57.1 pp (47.0 to 65.3 pp).
- UCI Evening minus Daytime recall is +0.2 pp (-15.0 to 16.4 pp). The false-positive-rate difference is +6.9 pp (-1.1 to 22.6 pp). Both intervals include zero; neither a clear direction nor equality is established.
- OULAD keeps each student's course records together. The Unknown group's recall interval is unavailable because too few students contribute found-dropout events.

Existing audit metrics are unchanged. Intervals are conditional on the current model and selection, with individual exploratory coverage under the stated sampling assumptions.

## Common audit across review capacities

The same audit now compares 15%, 20%, and 30% review shares on one set of existing predictions. Input rules, outcome exclusions, groups, and scores remain fixed. Each share receives its own overall metrics, group metrics, and conditional 95% intervals. No model is fitted by this comparison.

The marginal table shows the extra workload, newly found dropout cases, and additional false alarms when capacity increases. These are observed count changes on the same cohort. Intervals are individual and conditional on each selection; they do not test differences between capacities or select an optimal policy.

Capacity is rounded upward after outcome exclusions. OULAD selects course enrollments and separately counts unique students. UPV selects students; UCI selects students with resolved outcomes after excluding Enrolled. Equal scores follow the original input row order. Full results for every share are retained for inspection.

The 20% result is checked against the previous audit, including its intervals. The function accepts other lists of shares without dataset-specific logic.

In [ ]:
# Shared selection rounding, existing audit calculations, and capacity comparison.
# Requires the validated input function defined in the earlier section.

"""Audit observed outcomes and existing dropout probabilities without fitting a model."""

from decimal import Decimal, ROUND_CEILING

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def _review_count(review_share, records):
    """Round a declared decimal share upward without binary-float artifacts."""
    return int((Decimal(str(review_share)) * records).to_integral_value(rounding=ROUND_CEILING))


def audit_predictions(
    dataset,
    target_column,
    positive_value,
    score_column,
    group_columns=None,
    entity_id_column=None,
    review_share=0.20,
):
    """Audit one scored record per decision unit; scores must be probabilities.

    This function does not split data or fit a model. The caller is responsible
    for supplying predictions made without training on these outcomes.
    New audit submissions should use audit_validated_predictions from
    eduai_validated_audit, which checks outcome mapping and decision keys.
    """
    if not isinstance(dataset, pd.DataFrame) or dataset.empty:
        raise ValueError("dataset must be a non-empty pandas DataFrame")
    if not 0 < review_share <= 1:
        raise ValueError("review_share must be greater than 0 and at most 1")
    if isinstance(group_columns, str):
        group_columns = [group_columns]
    group_columns = list(group_columns or [])
    if len(group_columns) != len(set(group_columns)):
        raise ValueError("group_columns contains duplicates")
    if target_column == score_column or score_column in group_columns:
        raise ValueError("The score column must be separate from outcome and groups")

    required = [target_column, score_column, *group_columns]
    if entity_id_column is not None:
        required.append(entity_id_column)
    missing = [column for column in required if column not in dataset.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if dataset[target_column].isna().any():
        raise ValueError("Target column contains missing values")
    if positive_value not in dataset[target_column].unique():
        raise ValueError(f"Positive value not found: {positive_value}")
    if entity_id_column is not None and dataset[entity_id_column].isna().any():
        raise ValueError("Entity ID column contains missing values")

    try:
        probabilities = pd.to_numeric(dataset[score_column], errors="raise").to_numpy(dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError("Score column must contain numeric probabilities") from exc
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Score column must contain finite probabilities between 0 and 1")
    actual = dataset[target_column].eq(positive_value).to_numpy(dtype=int)
    if np.unique(actual).size != 2:
        raise ValueError("Dataset must contain both outcome classes")

    records_to_review = _review_count(review_share, len(actual))
    ranked_positions = np.argsort(-probabilities, kind="stable")
    selected = np.zeros(len(actual), dtype=bool)
    selected[ranked_positions[:records_to_review]] = True
    found = int(((actual == 1) & selected).sum())
    false_alarms = int(((actual == 0) & selected).sum())
    total_positives = int((actual == 1).sum())
    review = {
        "review_share_pct": 100 * review_share,
        "records": len(actual),
        "selected_records": records_to_review,
        "found_positives": found,
        "false_alarms": false_alarms,
        "missed_positives": total_positives - found,
        "precision_pct": 100 * found / records_to_review,
        "recall_pct": 100 * found / total_positives,
    }
    if entity_id_column is not None:
        ids = dataset[entity_id_column].to_numpy()
        review["unique_entities_to_review"] = len(np.unique(ids[selected]))

    group_rows = []
    for column in group_columns:
        values = dataset[column].astype("object").fillna("Unknown").astype(str).to_numpy()
        for group_value in np.unique(values):
            in_group = values == group_value
            positives = in_group & (actual == 1)
            negatives = in_group & (actual == 0)
            group_size = int(in_group.sum())
            total_group_positives = int(positives.sum())
            total_group_negatives = int(negatives.sum())
            found_group = int((positives & selected).sum())
            false_group = int((negatives & selected).sum())
            group_rows.append({
                "group_column": column,
                "group_value": group_value,
                "records": group_size,
                "total_positives": total_group_positives,
                "total_negatives": total_group_negatives,
                "positive_rate_pct": 100 * total_group_positives / group_size,
                "selected_records": int((in_group & selected).sum()),
                "selection_rate_pct": 100 * (in_group & selected).sum() / group_size,
                "found_positives": found_group,
                "false_alarms": false_group,
                "recall_pct": (
                    100 * found_group / total_group_positives
                    if total_group_positives else np.nan
                ),
                "false_positive_rate_pct": (
                    100 * false_group / total_group_negatives
                    if total_group_negatives else np.nan
                ),
            })
    group_audit = pd.DataFrame(group_rows)
    minimum_group_records = max(100, int(np.ceil(0.05 * len(actual))))
    minimum_outcomes_per_class = 30
    gap_rows = []
    small_groups = pd.DataFrame(columns=[
        "group_column", "group_value", "records", "total_positives", "total_negatives"
    ])
    if not group_audit.empty:
        size_check = group_audit.copy()
        size_check["passes_size_check"] = (
            (size_check["records"] >= minimum_group_records)
            & (size_check["total_positives"] >= minimum_outcomes_per_class)
            & (size_check["total_negatives"] >= minimum_outcomes_per_class)
        )
        small_groups = size_check.loc[
            ~size_check["passes_size_check"], small_groups.columns
        ]
        for column, groups in size_check.groupby("group_column"):
            comparable = groups[groups["passes_size_check"]]
            if len(comparable) < 2:
                continue
            for metric in ["recall_pct", "false_positive_rate_pct"]:
                lowest = comparable.loc[comparable[metric].idxmin()]
                highest = comparable.loc[comparable[metric].idxmax()]
                gap_rows.append({
                    "group_column": column,
                    "metric": metric,
                    "lowest_group": lowest["group_value"],
                    "lowest_pct": lowest[metric],
                    "highest_group": highest["group_value"],
                    "highest_pct": highest[metric],
                    "gap_pp": highest[metric] - lowest[metric],
                })
    group_gaps = pd.DataFrame(gap_rows)

    calibration_data = pd.DataFrame({"actual": actual, "probability": probabilities})
    if np.unique(probabilities).size == 1:
        calibration_data["risk_band"] = "All probabilities equal"
    else:
        calibration_data["risk_band"] = pd.qcut(
            calibration_data["probability"], q=5, duplicates="drop"
        )
    calibration = (
        calibration_data.groupby("risk_band", observed=True)
        .agg(
            records=("actual", "size"),
            mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
            actual_positive_pct=("actual", lambda values: 100 * values.mean()),
        )
        .reset_index()
    )
    calibration["gap_pp"] = (
        calibration["actual_positive_pct"] - calibration["mean_predicted_pct"]
    )
    return {
        "configuration": {
            "target_column": target_column,
            "positive_value": positive_value,
            "score_column": score_column,
            "entity_id_column": entity_id_column,
            "group_columns": group_columns,
            "review_share": review_share,
            "selection_unit": "record",
        },
        "data": {
            "records": len(actual),
            "positive_rate_pct": 100 * actual.mean(),
        },
        "scores": {
            "roc_auc": roc_auc_score(actual, probabilities),
            "mean_predicted_pct": 100 * probabilities.mean(),
            "observed_positive_pct": 100 * actual.mean(),
        },
        "review": review,
        "group_audit": group_audit,
        "group_gaps": group_gaps,
        "small_groups": small_groups,
        "calibration": calibration,
        "screening_rule": {
            "minimum_group_records": minimum_group_records,
            "minimum_outcomes_per_class": minimum_outcomes_per_class,
            "is_statistical_test": False,
        },
    }


"""Conditional intervals for existing, fixed audit selections.

Wilson intervals: https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm
Newcombe differences: statsmodels.stats.proportion.confint_proportions_2indep
Cluster resampling: Cameron and Miller (2015), A Practitioner's Guide to
Cluster-Robust Inference, https://cameron.econ.ucdavis.edu/research/Cameron_Miller_JHR_2015.pdf
"""

from numbers import Integral, Real
from statistics import NormalDist

import numpy as np
import pandas as pd




def _wilson_interval(successes, trials, confidence_level):
    if trials == 0:
        return np.nan, np.nan
    z = NormalDist().inv_cdf((1 + confidence_level) / 2)
    p = successes / trials
    denominator = 1 + z * z / trials
    center = (p + z * z / (2 * trials)) / denominator
    radius = z * np.sqrt(p * (1 - p) / trials + z * z / (4 * trials * trials)) / denominator
    return max(0.0, center - radius), min(1.0, center + radius)


def _newcombe_interval(row, reference, confidence_level):
    p = row["numerator"] / row["denominator_records"]
    q = reference["numerator"] / reference["denominator_records"]
    lp, up = _wilson_interval(row["numerator"], row["denominator_records"], confidence_level)
    lq, uq = _wilson_interval(reference["numerator"], reference["denominator_records"], confidence_level)
    return p - q - np.hypot(p - lp, uq - q), p - q + np.hypot(q - lq, up - p)


def _percentile_interval(draws, confidence_level):
    finite = draws[np.isfinite(draws)]
    if len(finite) < 0.95 * len(draws):
        return np.nan, np.nan, "Too Many Undefined Resamples"
    if np.ptp(finite) < 1e-12:
        return np.nan, np.nan, "Degenerate Bootstrap"
    tail = (1 - confidence_level) / 2
    low, high = np.quantile(finite, [tail, 1 - tail])
    return float(low), float(high), "Available"


def calculate_audit_uncertainty(
    prepared, *, target_column, positive_value, score_column, decision_unit,
    entity_id_column=None, group_columns=None, review_share=0.20,
    confidence_level=0.95, n_resamples=2000, random_state=42,
    group_references=None,
):
    """Consume already validated, outcome-filtered rows.

    Scores and selected/not-selected flags stay fixed. Intervals quantify
    evaluation-sample uncertainty under independent students, allowing dependence
    within each student. They do not include refitting, re-ranking, temporal
    drift, confounding, or simultaneous coverage across multiple comparisons.
    """
    if isinstance(confidence_level, (bool, np.bool_)) or not isinstance(confidence_level, Real) or not np.isfinite(confidence_level) or not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be a finite number between 0 and 1")
    if isinstance(n_resamples, (bool, np.bool_)) or not isinstance(n_resamples, Integral) or n_resamples < 200:
        raise ValueError("n_resamples must be an integer of at least 200")
    if isinstance(random_state, (bool, np.bool_)) or not isinstance(random_state, Integral) or random_state < 0:
        raise ValueError("random_state must be a non-negative integer")
    groups = [group_columns] if isinstance(group_columns, str) else list(group_columns or [])
    references = {} if group_references is None else group_references
    if not isinstance(references, dict) or any(column not in groups for column in references):
        raise ValueError("group_references must map audited group columns to reference values")
    actual = prepared[target_column].eq(positive_value).to_numpy()
    scores = prepared[score_column].to_numpy(dtype=float)
    selected = np.zeros(len(prepared), dtype=bool)
    selected[np.argsort(-scores, kind="stable")[:_review_count(review_share, len(prepared))]] = True
    codes = pd.factorize(prepared[entity_id_column], sort=False)[0] if entity_id_column else np.arange(len(prepared))
    unit_count = int(codes.max()) + 1
    repeated = unit_count < len(prepared)
    missing_cluster_id = entity_id_column is None and decision_unit != "student"
    method = "Unavailable" if missing_cluster_id else "Student Cluster Bootstrap" if repeated else "Wilson"
    masks = [(None, "All", np.ones(len(prepared), dtype=bool))]
    for column in groups:
        values = prepared[column].astype("object").fillna("Unknown").astype(str).to_numpy()
        if column in references and str(references[column]) not in values:
            raise ValueError(f"Reference group not found in audited rows: {column}")
        masks.extend((column, value, values == value) for value in np.unique(values))
    rows, denominators, numerators = [], [], []
    for column, value, mask in masks:
        for metric, outcomes in [("recall_pct", actual), ("false_positive_rate_pct", ~actual)]:
            denominator = mask & outcomes
            numerator = denominator & selected
            n, k = int(denominator.sum()), int(numerator.sum())
            n_units = len(np.unique(codes[denominator]))
            selected_units = len(np.unique(codes[numerator]))
            unselected_units = len(np.unique(codes[denominator & ~selected]))
            support = "Limited Outcome Support" if n_units < 30 else (
                "Sparse Selected or Unselected Outcomes"
                if min(selected_units, unselected_units) < 10 else ""
            )
            rows.append({
                "group_column": column, "group_value": value, "metric": metric,
                "records": int(mask.sum()), "independent_units": len(np.unique(codes[mask])),
                "numerator": k, "denominator_records": n, "denominator_units": n_units,
                "selected_outcome_units": selected_units, "unselected_outcome_units": unselected_units,
                "estimate_pct": 100 * k / n if n else np.nan,
                "ci_lower_pct": np.nan, "ci_upper_pct": np.nan,
                "ci_width_pp": np.nan, "method": method,
                "ci_status": "No Relevant Outcomes" if n == 0 else "Available",
                "support_note": support,
            })
            denominators.append(denominator)
            numerators.append(numerator)
    bootstrap = None
    if repeated and not missing_cluster_id:
        # Each draw gives one multiplicity to a whole student, across all groups.
        counts = np.zeros((unit_count, 2 * len(rows)), dtype=float)
        for j, (numerator, denominator) in enumerate(zip(numerators, denominators)):
            counts[:, 2 * j] = np.bincount(codes, weights=numerator, minlength=unit_count)
            counts[:, 2 * j + 1] = np.bincount(codes, weights=denominator, minlength=unit_count)
        rng = np.random.default_rng(random_state)
        bootstrap = np.full((n_resamples, len(rows)), np.nan)
        probabilities = np.full(unit_count, 1 / unit_count)
        for start in range(0, n_resamples, 100):
            stop = min(start + 100, n_resamples)
            weights = rng.multinomial(unit_count, probabilities, size=stop - start)
            totals = weights.astype(float) @ counts
            np.divide(totals[:, ::2], totals[:, 1::2],
                      out=bootstrap[start:stop], where=totals[:, 1::2] > 0)
    for j, row in enumerate(rows):
        n, k = row["denominator_records"], row["numerator"]
        if n == 0:
            continue
        if missing_cluster_id:
            row["ci_status"] = "Missing Entity ID"
            continue
        if not repeated:
            low, high = _wilson_interval(k, n, confidence_level)
        elif row["denominator_units"] < 10:
            row["ci_status"] = "Too Few Independent Units"
            continue
        elif k in (0, n):
            row["ci_status"] = "Boundary Estimate: Bootstrap Unavailable"
            continue
        elif min(row["selected_outcome_units"], row["unselected_outcome_units"]) < 5:
            row["ci_status"] = "Too Few Outcome Event Units"
            continue
        else:
            low, high, row["ci_status"] = _percentile_interval(bootstrap[:, j], confidence_level)
        row["ci_lower_pct"], row["ci_upper_pct"] = 100 * low, 100 * high
        row["ci_width_pp"] = 100 * (high - low)
    comparisons = []
    lookup = {(r["group_column"], r["group_value"], r["metric"]): j for j, r in enumerate(rows)}
    for column, reference_value in references.items():
        for j, row in enumerate(rows):
            if row["group_column"] != column or row["group_value"] == str(reference_value):
                continue
            ref_index = lookup[column, str(reference_value), row["metric"]]
            reference = rows[ref_index]
            low = high = np.nan
            status = "Available"
            if row["ci_status"] != "Available" or reference["ci_status"] != "Available":
                status = "Component Interval Unavailable"
            elif repeated:
                low, high, status = _percentile_interval(bootstrap[:, j] - bootstrap[:, ref_index], confidence_level)
            else:
                low, high = _newcombe_interval(row, reference, confidence_level)
            comparisons.append({
                "group_column": column, "group_value": row["group_value"],
                "reference_group": str(reference_value), "metric": row["metric"],
                "difference_pp": row["estimate_pct"] - reference["estimate_pct"],
                "ci_lower_pp": 100 * low, "ci_upper_pp": 100 * high,
                "method": method if repeated or missing_cluster_id else "Newcombe",
                "ci_status": status,
                "contains_zero": bool(low <= 0 <= high) if np.isfinite(low) else None,
                "support_note": "; ".join(dict.fromkeys(note for note in [row["support_note"], reference["support_note"]] if note)),
            })
    table = pd.DataFrame(rows)
    comparison_columns = ["group_column", "group_value", "reference_group", "metric", "difference_pp", "ci_lower_pp", "ci_upper_pp", "method", "ci_status", "contains_zero", "support_note"]
    return {
        "review_uncertainty": table.loc[table["group_column"].isna()].reset_index(drop=True),
        "group_uncertainty": table.loc[table["group_column"].notna()].reset_index(drop=True),
        "group_comparisons": pd.DataFrame(comparisons, columns=comparison_columns),
        "uncertainty_configuration": {
            "confidence_level": confidence_level, "method": method,
            "n_resamples": n_resamples if repeated and not missing_cluster_id else 0,
            "random_state": random_state, "entity_id_column": entity_id_column,
            "independent_units": unit_count if not missing_cluster_id else None,
            "group_references": dict(references),
            "selection_is_fixed": True, "model_is_fixed": True,
            "simultaneous_intervals": False,
            "support_warning_below_units": 30,
            "minimum_bootstrap_denominator_units": 10,
            "minimum_bootstrap_event_units_per_side": 5,
            "sparse_events_warning_below_units_per_side": 10,
            "minimum_valid_resample_share": 0.95,
            "identity_assumption": "Independent students; dependence allowed within supplied entity IDs",
        },
    }


"""Compare review capacities using one scored cohort and explicit input rules."""

from numbers import Real

import numpy as np
import pandas as pd





def audit_review_shares(
    dataset, *, target_column, positive_value, negative_values,
    score_column, decision_unit, decision_id_columns,
    review_shares=(0.15, 0.20, 0.30), group_columns=None,
    entity_id_column=None, excluded_values=(), confidence_level=0.95,
    n_resamples=2000, random_state=42, group_references=None,
):
    """Run the same validated prediction audit for several review shares.

    Shares are sorted and must be unique. One stable score ranking is used;
    equal scores are resolved by original input row order. Capacity is rounded
    up on the outcome-filtered cohort. Marginal changes are exact observed
    counts, not inference about differences between policies. Each confidence
    interval remains conditional on its own selection, with no simultaneous
    coverage across shares and no automatic choice of an optimal policy.
    """
    if not isinstance(review_shares, (list, tuple)) or not review_shares:
        raise ValueError("review_shares must be a non-empty list or tuple")
    shares = []
    for share in review_shares:
        if isinstance(share, (bool, np.bool_)) or not isinstance(share, Real) or not np.isfinite(share) or not 0 < share <= 1:
            raise ValueError("Each review share must be a finite number greater than 0 and at most 1")
        shares.append(float(share))
    if len(shares) != len(set(shares)):
        raise ValueError("review_shares contains duplicate values")
    shares.sort()
    common = dict(
        target_column=target_column, positive_value=positive_value,
        negative_values=negative_values, score_column=score_column,
        decision_unit=decision_unit, decision_id_columns=decision_id_columns,
        group_columns=group_columns, entity_id_column=entity_id_column,
        excluded_values=excluded_values,
    )
    prepared, validation = validate_audit_input(dataset, review_share=shares[0], **common)
    ranked_scores = np.sort(prepared[score_column].to_numpy(dtype=float))[::-1]
    reports, summary_rows, group_tables, comparison_tables = {}, [], [], []
    previous_review = None
    marginal_rows = []
    for share in shares:
        report = audit_validated_predictions(
            dataset, review_share=share, confidence_level=confidence_level,
            n_resamples=n_resamples, random_state=random_state,
            group_references=group_references, **common,
        )
        reports[share] = report
        review = report["review"]
        total_positives = review["found_positives"] + review["missed_positives"]
        total_negatives = review["records"] - total_positives
        selected_count = _review_count(share, len(prepared))
        cutoff = ranked_scores[selected_count - 1]
        above_cutoff = int((ranked_scores > cutoff).sum())
        overall = report["review_uncertainty"].set_index("metric")
        summary = {
            **review,
            "actual_review_share_pct": 100 * selected_count / len(prepared),
            "total_positives": total_positives, "total_negatives": total_negatives,
            "false_positive_rate_pct": 100 * review["false_alarms"] / total_negatives,
            "cutoff_score": cutoff,
            "cutoff_tie_records": int((ranked_scores == cutoff).sum()),
            "selected_at_cutoff": selected_count - above_cutoff,
        }
        for metric, prefix in [("recall_pct", "recall"), ("false_positive_rate_pct", "false_positive_rate")]:
            interval = overall.loc[metric]
            for source, suffix in [("ci_lower_pct", "ci_lower_pct"), ("ci_upper_pct", "ci_upper_pct"),
                                   ("ci_status", "ci_status"), ("support_note", "support_note")]:
                summary[f"{prefix}_{suffix}"] = interval[source]
        summary_rows.append(summary)
        groups = report["group_audit"].copy()
        if not groups.empty:
            for metric, prefix in [("recall_pct", "recall"), ("false_positive_rate_pct", "false_positive_rate")]:
                intervals = report["group_uncertainty"]
                fields = ["group_column", "group_value", "ci_lower_pct", "ci_upper_pct", "ci_status", "support_note", "denominator_units"]
                part = intervals.loc[intervals.metric == metric, fields].copy()
                part = part.rename(columns={c: f"{prefix}_{c}" for c in fields[2:]})
                groups = groups.merge(part, on=["group_column", "group_value"], validate="one_to_one")
        groups.insert(0, "review_share_pct", 100 * share)
        group_tables.append(groups)
        comparisons = report["group_comparisons"].copy()
        comparisons.insert(0, "review_share_pct", 100 * share)
        comparison_tables.append(comparisons)
        if previous_review is not None:
            previous_share, previous = previous_review
            added = review["selected_records"] - previous["selected_records"]
            found = review["found_positives"] - previous["found_positives"]
            false_alarms = review["false_alarms"] - previous["false_alarms"]
            marginal_rows.append({
                "from_review_share_pct": 100 * previous_share,
                "to_review_share_pct": 100 * share,
                "additional_records": added, "additional_found_positives": found,
                "additional_false_alarms": false_alarms,
                "marginal_precision_pct": 100 * found / added if added else np.nan,
            })
        previous_review = share, review
    marginal_columns = ["from_review_share_pct", "to_review_share_pct", "additional_records", "additional_found_positives", "additional_false_alarms", "marginal_precision_pct"]
    return {
        "configuration": {
            "review_shares": shares, "selection_unit": decision_unit,
            "selection_population": validation["selection_population"],
            "rounding_rule": "Ceiling of declared decimal share times audited records",
            "tie_rule": "Stable original input row order",
            "same_scores_and_population": True, "model_refitted": False,
            "confidence_level": confidence_level, "simultaneous_intervals": False,
            "marginal_changes_are_observed_counts": True,
        },
        "input_validation": validation,
        "summary": pd.DataFrame(summary_rows),
        "group_audit": pd.concat(group_tables, ignore_index=True),
        "group_comparisons": pd.concat(comparison_tables, ignore_index=True),
        "marginal_changes": pd.DataFrame(marginal_rows, columns=marginal_columns),
        "reports": reports,
    }


In [ ]:
review_shares_to_check = [0.15, 0.20, 0.30]
capacity_results = {}
for name, scored in validated_predictions.items():
    config = case_configs[name]
    references = uncertainty_reports[name]['uncertainty_configuration']['group_references']
    comparison = audit_review_shares(
        scored, target_column=config['target_column'], positive_value=config['positive_value'],
        score_column='Predicted Dropout Risk', group_columns=config['group_columns'],
        entity_id_column=config['entity_id_column'], review_shares=review_shares_to_check,
        group_references=references, confidence_level=0.95, n_resamples=2000, random_state=42,
        **input_rules[name],
    )
    # A common run must reproduce the earlier 20% audit, including intervals.
    previous = uncertainty_reports[name]
    at_twenty = comparison['reports'][0.20]
    assert at_twenty['review'] == previous['review'], name
    for table in ['group_audit', 'review_uncertainty', 'group_uncertainty', 'group_comparisons']:
        pd.testing.assert_frame_equal(at_twenty[table], previous[table])
    summary = comparison['summary']
    for column in ['selected_records', 'found_positives', 'false_alarms']:
        assert (summary[column].diff().dropna() >= 0).all(), (name, column)
    assert (summary['found_positives'] + summary['false_alarms'] == summary['selected_records']).all()
    capacity_results[name] = comparison

# Tables for reading; all numeric results remain in capacity_results.
def capacity_ci(row, prefix):
    if row[f'{prefix}_ci_status'] != 'Available':
        return 'Unavailable'
    return f"{row[f'{prefix}_ci_lower_pct']:.1f}% to {row[f'{prefix}_ci_upper_pct']:.1f}%"

count_rows, quality_rows, focus_capacity_rows, marginal_capacity_rows = [], [], [], []
for name, comparison in capacity_results.items():
    for _, row in comparison['summary'].iterrows():
        unique_students = row.get('unique_entities_to_review', row.selected_records if input_rules[name]['decision_unit'] == 'student' else np.nan)
        count_rows.append({
            'Dataset': name, 'Review Share': row.review_share_pct,
            'Selected Records': row.selected_records,
            'Unique Students Selected': unique_students,
            'Found Dropouts': row.found_positives, 'Missed Dropouts': row.missed_positives,
            'False Alarms': row.false_alarms,
        })
        quality_rows.append({
            'Dataset': name, 'Review Share': row.review_share_pct,
            'Precision': row.precision_pct, 'Recall': row.recall_pct,
            'Recall 95% CI': capacity_ci(row, 'recall'),
            'False-Positive Rate': row.false_positive_rate_pct,
            'False-Positive Rate 95% CI': capacity_ci(row, 'false_positive_rate'),
        })
    for _, row in comparison['group_audit'].iterrows():
        if not ((name == 'UPV 2022' and row.group_column == 'dedicacion') or name == 'UCI'):
            continue
        notes = list(dict.fromkeys(note for note in [row.recall_support_note, row.false_positive_rate_support_note] if note))
        focus_capacity_rows.append({
            'Dataset': name, 'Review Share': row.review_share_pct, 'Group': row.group_value,
            'Selected Records': row.selected_records, 'Found Dropouts': row.found_positives,
            'False Alarms': row.false_alarms, 'Recall': row.recall_pct,
            'Recall 95% CI': capacity_ci(row, 'recall'),
            'False-Positive Rate': row.false_positive_rate_pct,
            'False-Positive Rate 95% CI': capacity_ci(row, 'false_positive_rate'),
            'Support Note': '; '.join(notes) or '—',
        })
    for _, row in comparison['marginal_changes'].iterrows():
        marginal_capacity_rows.append({
            'Dataset': name, 'From Review Share': row.from_review_share_pct,
            'To Review Share': row.to_review_share_pct,
            'Additional Records': row.additional_records,
            'Additional Found Dropouts': row.additional_found_positives,
            'Additional False Alarms': row.additional_false_alarms,
            'Marginal Precision': row.marginal_precision_pct,
        })

def show_capacity_table(rows, percent_columns=(), count_columns=()):
    formats = {column: '{:.1f}%' for column in percent_columns}
    formats.update({column: lambda v: f'{v:,.0f}'.replace(',', ' ') for column in count_columns})
    display(pd.DataFrame(rows).style.hide(axis='index').format(formats, na_rep='Unavailable'))

print('Review workload and observed outcomes:')
show_capacity_table(count_rows, ['Review Share'], ['Selected Records', 'Unique Students Selected', 'Found Dropouts', 'Missed Dropouts', 'False Alarms'])
print('Overall metrics and individual conditional 95% intervals:')
show_capacity_table(quality_rows, ['Review Share', 'Precision', 'Recall', 'False-Positive Rate'])
print('UPV study mode and UCI attendance groups; full group results for all datasets are retained in capacity_results:')
show_capacity_table(focus_capacity_rows, ['Review Share', 'Recall', 'False-Positive Rate'], ['Selected Records', 'Found Dropouts', 'False Alarms'])
print('Observed changes between adjacent review capacities:')
show_capacity_table(marginal_capacity_rows, ['From Review Share', 'To Review Share', 'Marginal Precision'], ['Additional Records', 'Additional Found Dropouts', 'Additional False Alarms'])
print('The 20% reports match the previous audits, including their intervals, for all three datasets.')
print('No model was trained. Each comparison uses the same scored cohort and outcome rules.')
print('OULAD capacity is measured in enrollments; UCI continues to exclude Enrolled. Marginal changes are observed counts, with no inference about policy differences.')


Review workload and observed outcomes:


Dataset,Review Share,Selected Records,Unique Students Selected,Found Dropouts,Missed Dropouts,False Alarms
OULAD,15.0%,981,873,510,1 502,471
OULAD,20.0%,1 308,1 165,642,1 370,666
OULAD,30.0%,1 962,1 745,888,1 124,1 074
UPV 2022,15.0%,613,613,112,180,501
UPV 2022,20.0%,818,818,133,159,685
UPV 2022,30.0%,1 226,1 226,175,117,1 051
UCI,15.0%,108,108,93,181,15
UCI,20.0%,144,144,117,157,27
UCI,30.0%,216,216,160,114,56


Overall metrics and individual conditional 95% intervals:


Dataset,Review Share,Precision,Recall,Recall 95% CI,False-Positive Rate,False-Positive Rate 95% CI
OULAD,15.0%,52.0%,25.3%,23.4% to 27.3%,10.4%,9.5% to 11.3%
OULAD,20.0%,49.1%,31.9%,29.9% to 34.1%,14.7%,13.7% to 15.8%
OULAD,30.0%,45.3%,44.1%,41.9% to 46.3%,23.7%,22.6% to 25.0%
UPV 2022,15.0%,18.3%,38.4%,33.0% to 44.1%,13.2%,12.2% to 14.3%
UPV 2022,20.0%,16.3%,45.5%,39.9% to 51.3%,18.1%,16.9% to 19.3%
UPV 2022,30.0%,14.3%,59.9%,54.2% to 65.4%,27.7%,26.3% to 29.1%
UCI,15.0%,86.1%,33.9%,28.6% to 39.7%,3.4%,2.1% to 5.5%
UCI,20.0%,81.2%,42.7%,37.0% to 48.6%,6.1%,4.2% to 8.7%
UCI,30.0%,74.1%,58.4%,52.5% to 64.1%,12.6%,9.9% to 16.1%


UPV study mode and UCI attendance groups; full group results for all datasets are retained in capacity_results:


Dataset,Review Share,Group,Selected Records,Found Dropouts,False Alarms,Recall,Recall 95% CI,False-Positive Rate,False-Positive Rate 95% CI,Support Note
UPV 2022,15.0%,TC,533,95,438,35.1%,29.6% to 40.9%,11.8%,10.8% to 12.9%,—
UPV 2022,15.0%,TP,80,17,63,81.0%,60.0% to 92.3%,71.6%,61.4% to 80.0%,Limited Outcome Support
UPV 2022,20.0%,TC,736,116,620,42.8%,37.1% to 48.8%,16.7%,15.6% to 18.0%,—
UPV 2022,20.0%,TP,82,17,65,81.0%,60.0% to 92.3%,73.9%,63.8% to 81.9%,Limited Outcome Support
UPV 2022,30.0%,TC,1 137,157,980,57.9%,52.0% to 63.7%,26.4%,25.0% to 27.9%,—
UPV 2022,30.0%,TP,89,18,71,85.7%,65.4% to 95.0%,80.7%,71.2% to 87.6%,Limited Outcome Support
UCI,15.0%,Daytime,93,81,12,34.9%,29.1% to 41.2%,2.9%,1.7% to 5.0%,—
UCI,15.0%,Evening,15,12,3,28.6%,17.2% to 43.6%,9.4%,3.2% to 24.2%,Sparse Selected or Unselected Outcomes
UCI,20.0%,Daytime,122,99,23,42.7%,36.5% to 49.1%,5.6%,3.8% to 8.3%,—
UCI,20.0%,Evening,22,18,4,42.9%,29.1% to 57.8%,12.5%,5.0% to 28.1%,Sparse Selected or Unselected Outcomes


Observed changes between adjacent review capacities:


Dataset,From Review Share,To Review Share,Additional Records,Additional Found Dropouts,Additional False Alarms,Marginal Precision
OULAD,15.0%,20.0%,327,132,195,40.4%
OULAD,20.0%,30.0%,654,246,408,37.6%
UPV 2022,15.0%,20.0%,205,21,184,10.2%
UPV 2022,20.0%,30.0%,408,42,366,10.3%
UCI,15.0%,20.0%,36,24,12,66.7%
UCI,20.0%,30.0%,72,43,29,59.7%


The 20% reports match the previous audits, including their intervals, for all three datasets.
No model was trained. Each comparison uses the same scored cohort and outcome rules.
OULAD capacity is measured in enrollments; UCI continues to exclude Enrolled. Marginal changes are observed counts, with no inference about policy differences.


### What the capacity comparison shows

The common run reproduces all previous 20% reports, including intervals. It uses one scored cohort and the same outcome rules for each dataset.

- UPV, 20% to 30%: 408 additional students reviewed, 42 additional dropout cases found, and 366 additional false alarms. TP recall changes from 81.0% to 85.7%; TC recall changes from 42.8% to 57.9%.
- UCI, 20% to 30%: 72 additional students reviewed, 43 additional dropout cases found, and 29 additional false alarms. Enrolled remains excluded throughout.
- OULAD, 20% to 30%: 654 additional enrollments reviewed, 246 additional Withdrawn cases found, and 408 additional false alarms. Unique students selected increase from 1,165 to 1,745.

Marginal Precision describes only the added records: UCI's 43 found cases among 72 additional records give 59.7%. Interpret results within each cohort and outcome definition. Review capacity is a resource choice; the comparison does not automatically choose a best share.

Change review_shares_to_check to inspect other capacities. Full group metrics, group comparisons, and individual reports remain in capacity_results.

## Consolidated audit report

This section builds a readable report from the existing capacity_results. It presents the data scope, outcome mapping, all reviewed capacities, additional workload, and group details for an explicitly selected focus share. Findings are generated from declared rules and retain internal evidence references.

The builder formats existing numbers and intervals. It does not rerun the audit or select a recommended capacity. Missing prediction timing and outcome horizon remain visible as Not supplied. Group labels use caller-supplied mappings; source codes are preserved when their meanings have not been supplied.

Reports are prepared for OULAD, UPV 2022, and UCI. Set report_dataset_to_view to choose the displayed report and report_focus_share to choose the capacity for group details. Detailed group counts and intervals can be expanded in the report.

In [ ]:
"""Build a readable, deterministic report from an existing capacity audit."""

from html import escape
from numbers import Real

import numpy as np
import pandas as pd


def _report_number(value):
    return "Unavailable" if pd.isna(value) else f"{value:,.0f}".replace(",", " ")


def _report_percent(value):
    return "Unavailable" if pd.isna(value) else f"{value:.1f}%"


def _report_quantity(value, singular):
    return f"{_report_number(value)} {singular if value == 1 else singular + 's'}"


def _report_interval(row, prefix=None, difference=False):
    status = row[f"{prefix}_ci_status"] if prefix else row["ci_status"]
    if status != "Available":
        return "Unavailable"
    suffix = "pp" if difference else "pct"
    low = row[f"{prefix}_ci_lower_pct"] if prefix else row[f"ci_lower_{suffix}"]
    high = row[f"{prefix}_ci_upper_pct"] if prefix else row[f"ci_upper_{suffix}"]
    if not np.isfinite([low, high]).all():
        return "Unavailable"
    return f"{low:+.1f} to {high:+.1f} pp" if difference else f"{low:.1f}% to {high:.1f}%"


def build_audit_report(comparison, *, dataset_name, focus_share,
                       column_labels=None, group_labels=None, context=None):
    """Format existing results and describe them using explicit factual rules.

    No model, probability, metric, interval, or optimal policy is calculated here.
    focus_share must be supplied and must already exist in the capacity audit.
    Optional context is provider-supplied descriptive text, not verified evidence.
    Tables retain original source codes unless display labels are supplied.
    """
    if not isinstance(dataset_name, str) or not dataset_name.strip():
        raise ValueError("dataset_name must be a non-empty string")
    if isinstance(focus_share, (bool, np.bool_)) or not isinstance(focus_share, Real) or not np.isfinite(focus_share):
        raise ValueError("focus_share must be a finite numeric share present in the audit")
    required = ["configuration", "input_validation", "summary", "reports", "marginal_changes"]
    if not isinstance(comparison, dict) or any(key not in comparison for key in required):
        raise ValueError("comparison must be an audit_review_shares result")
    if not isinstance(comparison["reports"], dict) or focus_share not in comparison["reports"]:
        raise ValueError("focus_share is not present in the capacity audit")
    if not isinstance(comparison["summary"], pd.DataFrame) or comparison["summary"].empty:
        raise ValueError("The capacity summary must be a non-empty DataFrame")
    for name, mapping in [("column_labels", column_labels), ("group_labels", group_labels), ("context", context)]:
        if mapping is not None and not isinstance(mapping, dict):
            raise ValueError(f"{name} must be a dictionary")
    columns = column_labels or {}
    labels = group_labels or {}
    context = context or {}
    if any(not isinstance(mapping, dict) for mapping in labels.values()):
        raise ValueError("group_labels must map column names to dictionaries of value labels")
    if any(not isinstance(key, str) or not isinstance(value, str) for key, value in context.items()):
        raise ValueError("context must contain text keys and text values")
    focus = comparison["reports"][focus_share]
    config, validation = focus["configuration"], comparison["input_validation"]
    uncertainty = focus["uncertainty_configuration"]
    review = focus["review"]
    summary = comparison["summary"].copy()
    selected_summary = summary.loc[np.isclose(summary.review_share_pct, 100 * focus_share, rtol=0, atol=1e-10)]
    if len(selected_summary) != 1:
        raise ValueError("The focus share must have exactly one summary row")
    focus_row = selected_summary.iloc[0]
    if focus_row.selected_records != review["selected_records"] or focus_row.records != validation["audited_records"]:
        raise ValueError("Focus summary, individual audit, and audited population disagree")
    unit = config["selection_unit"]
    unit_noun = {"student": "student", "course_enrollment": "course enrollment",
                 "student_period": "student-period record", "record": "record"}.get(unit, "record")
    confidence_label = f"{100 * uncertainty['confidence_level']:g}% CI"
    metric_labels = {"recall_pct": "Recall", "false_positive_rate_pct": "False-Positive Rate"}

    def feature_label(column):
        return str(columns.get(column, column))

    def value_label(column, value):
        return str(labels.get(column, {}).get(value, value))

    def joined(values):
        return ", ".join(map(str, values)) or "None"

    scope_rows = [
        ("Dataset", dataset_name), ("Input Records", _report_number(validation["input_records"])),
        ("Excluded Records", _report_number(validation["excluded_records"])),
        ("Audited Records", _report_number(validation["audited_records"])),
        ("Decision Unit", unit.replace("_", " ").title()),
        ("Outcome Column", config["target_column"]), ("Positive Outcome", str(config["positive_value"])),
        ("Negative Outcomes", joined(config["negative_values"])),
        ("Excluded Outcomes", joined(config["excluded_values"])),
        ("Decision Key", joined(config["decision_id_columns"])),
        ("Student ID Column", config["entity_id_column"] or "Not supplied"),
        ("Group Features", ", ".join(feature_label(c) for c in config["group_columns"]) or "None"),
        ("Score Column", config["score_column"]), ("Focus Review Share", _report_percent(100 * focus_share)),
        ("Actual Selected Share", _report_percent(focus_row.actual_review_share_pct)),
        ("Compared Review Shares", ", ".join(_report_percent(100 * s) for s in comparison["configuration"]["review_shares"])),
        ("Confidence Level", _report_percent(100 * uncertainty["confidence_level"])),
        ("Interval Method", uncertainty["method"]),
        ("Bootstrap Resamples", _report_number(uncertainty["n_resamples"]) if uncertainty["n_resamples"] else "Not used"),
        ("Resampling Seed", str(uncertainty["random_state"]) if uncertainty["n_resamples"] else "Not used"),
        ("Reference Groups", ", ".join(f"{feature_label(c)}: {value_label(c, v)}" for c, v in uncertainty["group_references"].items()) or "None"),
        ("Capacity Rounding", comparison["configuration"]["rounding_rule"]),
        ("Equal-Score Rule", comparison["configuration"]["tie_rule"]),
        ("Prediction Timing", context.get("Prediction Timing") or "Not supplied"),
        ("Outcome Horizon", context.get("Outcome Horizon") or "Not supplied"),
    ]
    scope_rows.extend((key, value) for key, value in context.items() if key not in {"Prediction Timing", "Outcome Horizon"})
    capacity_rows = []
    for _, row in summary.iterrows():
        capacity_rows.append({
            "Review Share": _report_percent(row.review_share_pct),
            "Selected Records": _report_number(row.selected_records),
            "Found Cases": _report_number(row.found_positives),
            "Missed Cases": _report_number(row.missed_positives),
            "False Alarms": _report_number(row.false_alarms),
            "Precision": _report_percent(row.precision_pct), "Recall": _report_percent(row.recall_pct),
            f"Recall {confidence_label}": _report_interval(row, "recall"),
            "False-Positive Rate": _report_percent(row.false_positive_rate_pct),
            f"False-Positive Rate {confidence_label}": _report_interval(row, "false_positive_rate"),
        })
    group_count_rows = []
    for _, row in focus["group_audit"].iterrows():
        group_count_rows.append({
            "Group Feature": feature_label(row.group_column), "Group": value_label(row.group_column, row.group_value),
            "Records": _report_number(row.records), "Positive Outcomes": _report_number(row.total_positives),
            "Selected Records": _report_number(row.selected_records),
            "Found Cases": _report_number(row.found_positives),
            "Missed Cases": _report_number(row.total_positives - row.found_positives),
            "False Alarms": _report_number(row.false_alarms),
        })
    group_interval_rows = []
    for _, row in focus["group_uncertainty"].iterrows():
        group_interval_rows.append({
            "Group Feature": feature_label(row.group_column), "Group": value_label(row.group_column, row.group_value),
            "Metric": metric_labels[row.metric], "Estimate": _report_percent(row.estimate_pct),
            confidence_label: _report_interval(row), "Relevant Outcomes": _report_number(row.denominator_records),
            "Contributing Students": _report_number(row.denominator_units),
            "Interval Status": row.ci_status, "Support Note": row.support_note or "—",
        })
    group_comparison_rows = []
    for _, row in focus["group_comparisons"].iterrows():
        group_comparison_rows.append({
            "Group Feature": feature_label(row.group_column),
            "Comparison": f"{value_label(row.group_column, row.group_value)} minus {value_label(row.group_column, row.reference_group)}",
            "Metric": metric_labels[row.metric],
            "Difference (pp)": "Unavailable" if pd.isna(row.difference_pp) else f"{row.difference_pp:+.1f}",
            confidence_label: _report_interval(row, difference=True),
            "Interval Status": row.ci_status, "Support Note": row.support_note or "—",
        })
    marginal_rows = []
    for _, row in comparison["marginal_changes"].iterrows():
        marginal_rows.append({
            "From Review Share": _report_percent(row.from_review_share_pct),
            "To Review Share": _report_percent(row.to_review_share_pct),
            "Additional Records": _report_number(row.additional_records),
            "Additional Found Cases": _report_number(row.additional_found_positives),
            "Additional False Alarms": _report_number(row.additional_false_alarms),
            "Marginal Precision": _report_percent(row.marginal_precision_pct),
        })
    tables = {
        "scope": pd.DataFrame(scope_rows, columns=["Field", "Value"]),
        "capacity_results": pd.DataFrame(capacity_rows),
        "group_counts": pd.DataFrame(group_count_rows),
        "group_uncertainty": pd.DataFrame(group_interval_rows),
        "group_comparisons": pd.DataFrame(group_comparison_rows),
        "marginal_changes": pd.DataFrame(marginal_rows),
    }
    findings = []

    def add(rule, text, evidence):
        findings.append({"rule": rule, "text": text, "evidence": evidence})

    total_cases = review["found_positives"] + review["missed_positives"]
    add("focus_outcomes",
        f"At {_report_percent(100 * focus_share)} review capacity, the selection contains {_report_quantity(review['selected_records'], unit_noun)}. "
        f"It finds {_report_number(review['found_positives'])} of {_report_quantity(total_cases, 'positive case')}, "
        f"misses {_report_number(review['missed_positives'])}, and produces {_report_quantity(review['false_alarms'], 'false alarm')}.",
        f"reports[{focus_share}].review")
    if validation["excluded_records"]:
        excluded_counts = ", ".join(f"{label}: {_report_number(count)}" for label, count in validation["excluded_outcome_counts"].items())
        add("outcome_exclusions", f"The outcome rules exclude {_report_quantity(validation['excluded_records'], 'input record')} ({excluded_counts}). "
            f"Results apply to the remaining {_report_number(validation['audited_records'])} records.", "input_validation")
    if "unique_entities_to_review" in review and unit != "student":
        add("record_and_student_units", f"The selection of {_report_quantity(review['selected_records'], unit_noun)} contains "
            f"{_report_number(review['unique_entities_to_review'])} distinct supplied student IDs.", f"reports[{focus_share}].review")
    if focus_row.selected_at_cutoff < focus_row.cutoff_tie_records:
        add("cutoff_ties", f"The cutoff score is shared by {_report_number(focus_row.cutoff_tie_records)} records; "
            f"{_report_number(focus_row.selected_at_cutoff)} of those records are selected using original input row order.", "summary.cutoff_tie_records")
    for _, row in comparison["marginal_changes"].iterrows():
        add("capacity_change", f"Increasing capacity from {_report_percent(row.from_review_share_pct)} to {_report_percent(row.to_review_share_pct)} "
            f"adds {_report_quantity(row.additional_records, unit_noun)} to review, "
            f"finds {_report_quantity(row.additional_found_positives, 'additional positive case')}, "
            f"and adds {_report_quantity(row.additional_false_alarms, 'false alarm')}.", "marginal_changes")
    for _, row in focus["group_comparisons"].iterrows():
        label = f"{feature_label(row.group_column)}: {value_label(row.group_column, row.group_value)} minus {value_label(row.group_column, row.reference_group)}"
        interval = _report_interval(row, difference=True)
        direction = "The interval is unavailable; the direction is not assessed."
        if interval != "Unavailable":
            direction = "The interval includes zero; the direction is unresolved." if row.ci_lower_pp <= 0 <= row.ci_upper_pp else (
                "The interval lies above zero under the stated assumptions." if row.ci_lower_pp > 0 else "The interval lies below zero under the stated assumptions."
            )
        difference = "Unavailable" if pd.isna(row.difference_pp) else f"{row.difference_pp:+.1f} pp"
        note = f" Support note: {row.support_note}." if row.support_note else ""
        add("group_difference", f"At {_report_percent(100 * focus_share)}, {label}, {metric_labels[row.metric]} difference: "
            f"{difference}; individual {confidence_label}: {interval}. {direction}{note}", f"reports[{focus_share}].group_comparisons")
    if focus["group_comparisons"].empty and not focus["group_audit"].empty:
        add("no_group_reference", "Group metrics are available; no reference group was supplied for signed group comparisons.",
            f"reports[{focus_share}].uncertainty_configuration.group_references")
    if focus["group_audit"].empty:
        add("no_groups", "No group columns were supplied; this report contains overall results only.",
            f"reports[{focus_share}].configuration.group_columns")
    intervals = focus["group_uncertainty"]
    if not intervals.empty:
        unavailable = int((intervals.ci_status != "Available").sum())
        limited = int(intervals.support_note.ne("").sum())
        if unavailable or limited:
            add("group_interval_support", f"At the focus capacity: unavailable group-metric intervals, {_report_number(unavailable)}; "
                f"group metrics with support notes, {_report_number(limited)}. Inspect their relevant-outcome and contributing-student counts.",
                f"reports[{focus_share}].group_uncertainty")
    for column, count in validation["missing_group_records"].items():
        if count:
            add("missing_group_values", f"{feature_label(column)} is missing for {_report_number(count)} audited records; "
                "those records remain visible in the Unknown group.", "input_validation.missing_group_records")
    limitations = [
        "Counts describe the submitted, outcome-filtered cohort. Positive cases mean the declared positive outcome; course withdrawal and university dropout are different events.",
        "Intervals describe evaluation-sample uncertainty for fixed scores and selections, assuming independent students and allowing dependence within supplied student IDs. They do not cover model refitting, re-ranking, or future-cohort changes.",
        "Intervals are individual exploratory comparisons without simultaneous coverage across groups or review shares. An interval including zero does not establish equality; excluding zero does not establish causation or discrimination.",
        "Capacity changes are observed count differences. The report does not infer significance of policy changes or select an optimal capacity.",
        "Recall divides found positive cases by all positive cases. False-positive rate divides false alarms by negative cases. Precision is the positive-case share of selected records; marginal precision uses only additional selected records.",
        "Group labels follow the supplied display mapping; unmapped values remain source codes. Support flags and bootstrap guards are diagnostic rules, not guarantees of interval coverage.",
    ]
    if config["entity_id_column"] is None:
        limitations.append("No student ID was supplied. Student identity and duplicate students cannot be verified from technical row keys; student intervals rely on the one-row-per-student assumption.")
    if validation["excluded_records"]:
        limitations.append("Outcome exclusions change the evaluation population. Compare capacities within this filtered cohort; results do not describe the excluded outcomes.")
    if not context.get("Prediction Timing") or not context.get("Outcome Horizon"):
        limitations.append("Prediction timing and/or outcome horizon were not supplied. Verify them with the data provider before interpreting this as an operational evaluation.")
    else:
        limitations.append("Prediction timing and outcome horizon are supplied descriptions; this report does not verify their provenance or rule out outcome leakage.")
    confidence_title = f"Group results at {_report_percent(100 * focus_share)}"

    def table_html(key):
        table = tables[key]
        if table.empty:
            return "<p>No results for this section.</p>"
        return '<div class="audit-table">' + table.to_html(index=False, border=0, escape=True) + "</div>"

    parts = [
        "<!doctype html><html><head><meta charset=\"utf-8\">",
        f"<title>Audit Report: {escape(dataset_name)}</title>",
        "<style>.eduai-report{font-family:Arial,sans-serif;line-height:1.45;max-width:1200px;margin:auto;padding:16px}"
        ".eduai-report .audit-table{overflow-x:auto}.eduai-report table{border-collapse:collapse;font-size:0.9em}"
        ".eduai-report th,.eduai-report td{border:1px solid currentColor;padding:6px 8px;text-align:left;vertical-align:top}"
        ".eduai-report details{margin:12px 0}.eduai-report summary{cursor:pointer;font-weight:bold}"
        ".eduai-report li{margin:8px 0}</style></head><body><article class=\"eduai-report\">",
        f"<h2>Audit Report: {escape(dataset_name)}</h2>",
        f"<p>Focus review share: <strong>{_report_percent(100 * focus_share)}</strong>. "
        "This is the supplied report focus, not a recommended capacity.</p>",
        "<h3>Key Findings</h3><ul>" + "".join(f"<li>{escape(finding['text'])}</li>" for finding in findings) + "</ul>",
        "<h3>Data and Scope</h3>" + table_html("scope"),
        "<h3>Review Capacities</h3>" + table_html("capacity_results"),
        "<h3>Additional Workload</h3>" + table_html("marginal_changes"),
        f"<h3>{escape(confidence_title)}</h3>" + table_html("group_comparisons"),
        "<details><summary>Group Counts</summary>" + table_html("group_counts") + "</details>",
        "<details><summary>Group Metrics and Uncertainty</summary>" + table_html("group_uncertainty") + "</details>",
        "<h3>Reading Notes and Limits</h3><ul>" + "".join(f"<li>{escape(note)}</li>" for note in limitations) + "</ul>",
        "</article></body></html>",
    ]
    return {"dataset_name": dataset_name, "focus_share": float(focus_share),
            "tables": tables, "findings": findings, "limitations": limitations,
            "html": "".join(parts)}


In [ ]:
from IPython.display import HTML

# Display controls: group details use this explicitly chosen share.
report_dataset_to_view = 'UPV 2022'
report_focus_share = 0.20

report_column_labels = {
    'imd_band': 'IMD Band', 'dedicacion': 'Study Mode',
    'desplazado_hash': 'Displacement Status', 'estudios_m_hash': "Mother's Education",
}
report_outcome_meanings = {
    'OULAD': 'Withdrawal from a course presentation, using the declared Withdrawn outcome.',
    'UPV 2022': 'Student dropout according to the declared UPV outcome mapping.',
    'UCI': 'Student dropout in the resolved-outcome evaluation subset; Enrolled is excluded.',
}
report_identity_notes = {
    'OULAD': 'Supplied student IDs; each student can have several course-enrollment records.',
    'UPV 2022': 'One aggregated record per supplied student ID, prepared by the earlier UPV adapter.',
    'UCI': 'One student per source row according to the dataset description; technical row keys only.',
}

if report_dataset_to_view not in capacity_results:
    raise ValueError('Choose a dataset available in capacity_results')
audit_reports = {}
for name, comparison in capacity_results.items():
    labels_for_report = report_column_labels.copy()
    if name == 'UCI':
        labels_for_report.update({column: 'Attendance' for column in case_configs[name]['group_columns']})
    before_summary = comparison['summary'].copy(deep=True)
    report = build_audit_report(
        comparison, dataset_name=name, focus_share=report_focus_share,
        column_labels=labels_for_report,
        group_labels={'imd_band': {'10-20': '10-20%'}},
        context={
            'Evaluation Design': 'Retrospective held-out predictions from the earlier notebook experiments.',
            'Positive Case Meaning': report_outcome_meanings[name],
            'Identity Basis': report_identity_notes[name],
        },
    )
    # Report assembly must preserve the numbers and source results.
    pd.testing.assert_frame_equal(before_summary, comparison['summary'])
    for position, (_, source) in enumerate(comparison['summary'].iterrows()):
        presented = report['tables']['capacity_results'].iloc[position]
        assert presented['Selected Records'] == _report_number(source.selected_records), name
        assert presented['Found Cases'] == _report_number(source.found_positives), name
        assert presented['Missed Cases'] == _report_number(source.missed_positives), name
        assert presented['False Alarms'] == _report_number(source.false_alarms), name
    scope = report['tables']['scope'].set_index('Field')['Value']
    assert scope['Audited Records'] == _report_number(comparison['input_validation']['audited_records'])
    assert scope['Excluded Records'] == _report_number(comparison['input_validation']['excluded_records'])
    audit_reports[name] = report

print('Reports ready: ' + ', '.join(audit_reports))
print('All report counts match the existing audits. Report assembly does not rerun their calculations.')
display(HTML(audit_reports[report_dataset_to_view]['html']))


Reports ready: OULAD, UPV 2022, UCI
All report counts match the existing audits. Report assembly does not rerun their calculations.


Field,Value
Dataset,UPV 2022
Input Records,4 086
Excluded Records,0
Audited Records,4 086
Decision Unit,Student
Outcome Column,abandono_hash
Positive Outcome,A
Negative Outcomes,B
Excluded Outcomes,None
Decision Key,dni_hash
